<a href="https://colab.research.google.com/github/skhosanam/Health-Misinformation-Detection-and-Propagation-in-Online-Social-Networks/blob/main/Health_Misinformation_Modelling_Detection_%26_Propgation_M_SIKOSANA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# Health misinformation classification reproducibility pipeline
# ============================================================

import argparse
import json
import os
import random
import re
import shutil
import subprocess
import sys
import warnings
from collections import Counter
from pathlib import Path

# ============================================================
# 0. Package installation for Colab
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "numpy",
        "scikit-learn",
        "nltk",
        "matplotlib",
        "scipy<1.13",
        "gensim==4.3.2",
        "torch",
        "transformers",
        "accelerate",
    ],
    check=True,
)

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")


# ============================================================
# 1. General configuration
# ============================================================

SEED = 42
TEST_SIZE = 0.20
POSITIVE_LABEL = 0

MAX_FEATURES_ML = 30000
MIN_DF = 2

MAX_LEN = 128
MAX_VOCAB = 30000
WORD2VEC_DIM = 300
BATCH_SIZE = 32

DEFAULT_FAKE_FILE = "fakeNews.csv"
DEFAULT_TRUE_FILE = "trueNews.csv"
DEFAULT_OUTPUT_DIR = "results"


# ============================================================
# 2. Colab upload support
# ============================================================

def upload_data_if_needed(fake_path=DEFAULT_FAKE_FILE, true_path=DEFAULT_TRUE_FILE):
    fake_path = Path(fake_path)
    true_path = Path(true_path)

    if fake_path.exists() and true_path.exists():
        return str(fake_path), str(true_path)

    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)

    data_fake = data_dir / fake_path.name
    data_true = data_dir / true_path.name

    if data_fake.exists() and data_true.exists():
        return str(data_fake), str(data_true)

    try:
        from google.colab import files

        print("Upload fakeNews.csv and trueNews.csv")
        uploaded = files.upload()

        for filename in uploaded.keys():
            source = Path(filename)
            destination = data_dir / filename

            if destination.exists():
                destination.unlink()

            shutil.move(str(source), str(destination))

        if data_fake.exists() and data_true.exists():
            return str(data_fake), str(data_true)

        if Path("fakeNews.csv").exists() and Path("trueNews.csv").exists():
            return "fakeNews.csv", "trueNews.csv"

        raise FileNotFoundError(
            "Expected fakeNews.csv and trueNews.csv after upload."
        )

    except Exception as exc:
        raise FileNotFoundError(
            "Data files not found. Place fakeNews.csv and trueNews.csv in the current "
            "folder or in a data/ folder, or run this in Colab and upload both files."
        ) from exc


# ============================================================
# 3. Reproducibility and preprocessing
# ============================================================

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^A-Za-z0-9!?.,;:'%\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    return re.findall(r"[a-zA-Z][a-zA-Z']+|\d+", str(text).lower())


def download_nltk_resources():
    try:
        import nltk

        for package in ["wordnet", "omw-1.4"]:
            try:
                nltk.data.find(f"corpora/{package}")
            except LookupError:
                nltk.download(package, quiet=True)
    except Exception:
        pass


def lemmatize_tokens(text):
    download_nltk_resources()

    try:
        from nltk.stem import WordNetLemmatizer

        lemmatizer = WordNetLemmatizer()
        return [lemmatizer.lemmatize(tok) for tok in tokenize(text)]
    except Exception:
        return tokenize(text)


def stem_tokens(text):
    try:
        from nltk.stem import PorterStemmer

        stemmer = PorterStemmer()
        return [stemmer.stem(tok) for tok in tokenize(text)]
    except Exception:
        return tokenize(text)


# ============================================================
# 4. Data loading and splitting
# ============================================================

def load_data(fake_path, true_path):
    fake = pd.read_csv(fake_path)
    true = pd.read_csv(true_path)

    if "Text" not in fake.columns:
        raise ValueError("fakeNews.csv must contain a column called Text")

    if "Text" not in true.columns:
        raise ValueError("trueNews.csv must contain a column called Text")

    fake_df = pd.DataFrame(
        {
            "text": fake["Text"].astype(str),
            "label": 0,
            "source_file": Path(fake_path).name,
        }
    )

    true_df = pd.DataFrame(
        {
            "text": true["Text"].astype(str),
            "label": 1,
            "source_file": Path(true_path).name,
        }
    )

    df = pd.concat([fake_df, true_df], ignore_index=True)
    df["text_clean"] = df["text"].map(clean_text)
    df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)

    return df


def save_dataset_summary(df, out_dir):
    summary = {
        "n_total": int(len(df)),
        "n_fake_label_0": int((df["label"] == 0).sum()),
        "n_true_label_1": int((df["label"] == 1).sum()),
        "fake_percent": round(float((df["label"] == 0).mean() * 100), 2),
        "true_percent": round(float((df["label"] == 1).mean() * 100), 2),
        "n_exact_duplicate_clean_text": int(df["text_clean"].duplicated().sum()),
    }

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(out_dir / "dataset_summary.csv", index=False)

    with open(out_dir / "dataset_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    return summary_df


def fixed_split(df):
    return train_test_split(
        df["text_clean"].values,
        df["label"].values,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df["label"].values,
    )


def save_split_summary(y_train, y_test, out_dir):
    split_summary = {
        "train_size": int(len(y_train)),
        "test_size": int(len(y_test)),
        "train_fake_label_0": int((y_train == 0).sum()),
        "train_true_label_1": int((y_train == 1).sum()),
        "test_fake_label_0": int((y_test == 0).sum()),
        "test_true_label_1": int((y_test == 1).sum()),
        "split": "fixed stratified 80/20",
        "random_state": SEED,
        "primary_reporting_label": POSITIVE_LABEL,
    }

    split_df = pd.DataFrame([split_summary])
    split_df.to_csv(out_dir / "split_summary.csv", index=False)

    return split_df


# ============================================================
# 5. Evaluation
# ============================================================

def compute_metrics(y_true, y_pred, y_score=None, positive_label=POSITIVE_LABEL):
    output = {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_primary": f1_score(
            y_true,
            y_pred,
            pos_label=positive_label,
            zero_division=0,
        ),
        "precision_primary": precision_score(
            y_true,
            y_pred,
            pos_label=positive_label,
            zero_division=0,
        ),
        "recall_primary": recall_score(
            y_true,
            y_pred,
            pos_label=positive_label,
            zero_division=0,
        ),
        "f1_macro": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0,
        ),
        "f1_weighted": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0,
        ),
    }

    if y_score is not None:
        try:
            y_binary = (np.asarray(y_true) == positive_label).astype(int)
            output["roc_auc_primary"] = roc_auc_score(y_binary, y_score)
        except Exception:
            output["roc_auc_primary"] = np.nan
    else:
        output["roc_auc_primary"] = np.nan

    return output


def percent_table(df):
    metric_cols = [
        "accuracy",
        "f1_primary",
        "precision_primary",
        "recall_primary",
        "f1_macro",
        "f1_weighted",
        "roc_auc_primary",
    ]

    output = df.copy()

    for col in metric_cols:
        if col in output.columns:
            output[col] = (output[col] * 100).round(2)

    return output


def get_score_for_positive_class(model, X, positive_label=POSITIVE_LABEL):
    try:
        if hasattr(model, "predict_proba"):
            probabilities = model.predict_proba(X)
            classes = list(model.classes_)
            index = classes.index(positive_label)
            return probabilities[:, index]

        if hasattr(model, "decision_function"):
            scores = model.decision_function(X)
            classes = list(model.classes_)

            if len(classes) == 2:
                if classes[1] == positive_label:
                    return scores
                return -scores

    except Exception:
        return None

    return None


# ============================================================
# 6. Conventional machine learning
# ============================================================

def get_vectorizers():
    fewer_stopwords = sorted(
        set(ENGLISH_STOP_WORDS)
        - {"no", "not", "nor", "never", "none", "cannot", "against"}
    )

    return {
        "baseline": TfidfVectorizer(
            lowercase=True,
            tokenizer=tokenize,
            token_pattern=None,
            ngram_range=(1, 1),
            max_features=MAX_FEATURES_ML,
            min_df=MIN_DF,
            sublinear_tf=True,
        ),
        "ngram": TfidfVectorizer(
            lowercase=True,
            tokenizer=tokenize,
            token_pattern=None,
            ngram_range=(1, 2),
            max_features=MAX_FEATURES_ML,
            min_df=MIN_DF,
            sublinear_tf=True,
        ),
        "fewer_stop_words": TfidfVectorizer(
            lowercase=True,
            tokenizer=tokenize,
            token_pattern=None,
            stop_words=fewer_stopwords,
            ngram_range=(1, 1),
            max_features=MAX_FEATURES_ML,
            min_df=MIN_DF,
            sublinear_tf=True,
        ),
        "lemmatisation": TfidfVectorizer(
            lowercase=True,
            tokenizer=lemmatize_tokens,
            token_pattern=None,
            ngram_range=(1, 1),
            max_features=MAX_FEATURES_ML,
            min_df=MIN_DF,
            sublinear_tf=True,
        ),
        "stemming": TfidfVectorizer(
            lowercase=True,
            tokenizer=stem_tokens,
            token_pattern=None,
            ngram_range=(1, 1),
            max_features=MAX_FEATURES_ML,
            min_df=MIN_DF,
            sublinear_tf=True,
        ),
    }


def get_classifiers():
    return {
        "NB": MultinomialNB(),
        "GB": GradientBoostingClassifier(random_state=SEED),
        "SVM": LinearSVC(random_state=SEED),
        "DT": DecisionTreeClassifier(random_state=SEED),
        "RF": RandomForestClassifier(
            n_estimators=300,
            random_state=SEED,
            n_jobs=-1,
        ),
        "Bagg": BaggingClassifier(
            random_state=SEED,
            n_jobs=-1,
        ),
        "AdaB": AdaBoostClassifier(random_state=SEED),
        "SGD": SGDClassifier(
            loss="log_loss",
            random_state=SEED,
            max_iter=2000,
            tol=1e-4,
        ),
        "LR": LogisticRegression(
            max_iter=3000,
            random_state=SEED,
            n_jobs=-1,
        ),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(100,),
            activation="relu",
            max_iter=300,
            random_state=SEED,
            early_stopping=True,
        ),
    }


def run_conventional_ml(X_train, X_test, y_train, y_test, out_dir):
    rows = []

    vectorizers = get_vectorizers()
    classifiers = get_classifiers()

    for preprocessing_name, vectorizer in vectorizers.items():
        print(f"\nRunning preprocessing: {preprocessing_name}")

        X_train_vec = vectorizer.fit_transform(X_train)
        X_test_vec = vectorizer.transform(X_test)

        for classifier_name, classifier in classifiers.items():
            print(f"  Training {classifier_name}")

            model = clone(classifier)
            model.fit(X_train_vec, y_train)

            predictions = model.predict(X_test_vec)
            score = get_score_for_positive_class(model, X_test_vec)

            metrics = compute_metrics(y_test, predictions, score)

            rows.append(
                {
                    "model_family": "conventional_ml",
                    "preprocessing": preprocessing_name,
                    "model": classifier_name,
                    **metrics,
                }
            )

    results = pd.DataFrame(rows)
    results.to_csv(out_dir / "conventional_ml_results_raw.csv", index=False)

    display_results = percent_table(results)
    display_results.to_csv(out_dir / "conventional_ml_results_percent.csv", index=False)

    summary_rows = []

    for preprocessing_name in vectorizers.keys():
        subset = display_results[
            display_results["preprocessing"] == preprocessing_name
        ]

        accuracy_row = {
            "metric_preprocessing": f"accuracy_{preprocessing_name}"
        }
        f1_row = {
            "metric_preprocessing": f"f1_{preprocessing_name}"
        }

        for classifier_name in classifiers.keys():
            selected = subset[subset["model"] == classifier_name].iloc[0]
            accuracy_row[classifier_name] = selected["accuracy"]
            f1_row[classifier_name] = selected["f1_primary"]

        summary_rows.append(accuracy_row)
        summary_rows.append(f1_row)

    summary = pd.DataFrame(summary_rows)
    summary.to_csv(out_dir / "conventional_ml_summary.csv", index=False)

    return results


# ============================================================
# 7. PyTorch utilities
# ============================================================

def torch_device():
    import torch

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def build_vocab(token_lists, max_vocab=MAX_VOCAB, min_freq=2):
    counter = Counter(token for tokens in token_lists for token in tokens)

    vocab = {
        "<PAD>": 0,
        "<UNK>": 1,
    }

    for word, count in counter.most_common(max_vocab - 2):
        if count >= min_freq:
            vocab[word] = len(vocab)

    return vocab


def encode_sequence(tokens, vocab, max_len=MAX_LEN):
    ids = [vocab.get(token, vocab["<UNK>"]) for token in tokens[:max_len]]

    if len(ids) < max_len:
        ids += [vocab["<PAD>"]] * (max_len - len(ids))

    return ids


def train_word2vec_embedding(token_lists, vocab):
    from gensim.models import Word2Vec

    word2vec = Word2Vec(
        sentences=token_lists,
        vector_size=WORD2VEC_DIM,
        window=5,
        min_count=1,
        workers=4,
        sg=1,
        seed=SEED,
        epochs=20,
    )

    embedding_matrix = np.random.normal(
        scale=0.05,
        size=(len(vocab), WORD2VEC_DIM),
    ).astype(np.float32)

    embedding_matrix[vocab["<PAD>"]] = np.zeros(WORD2VEC_DIM)

    for word, index in vocab.items():
        if word in word2vec.wv:
            embedding_matrix[index] = word2vec.wv[word]

    return embedding_matrix


def make_word_loaders(X_train, X_test, y_train, y_test):
    import torch
    from torch.utils.data import DataLoader, Dataset

    train_tokens = [tokenize(text) for text in X_train]
    test_tokens = [tokenize(text) for text in X_test]

    vocab = build_vocab(train_tokens)
    embedding_matrix = train_word2vec_embedding(train_tokens, vocab)

    X_train_encoded = np.array(
        [encode_sequence(tokens, vocab) for tokens in train_tokens],
        dtype=np.int64,
    )

    X_test_encoded = np.array(
        [encode_sequence(tokens, vocab) for tokens in test_tokens],
        dtype=np.int64,
    )

    class WordDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.tensor(X, dtype=torch.long)
            self.y = torch.tensor(y, dtype=torch.long)

        def __len__(self):
            return len(self.y)

        def __getitem__(self, index):
            return {
                "input_ids": self.X[index],
                "labels": self.y[index],
            }

    train_loader = DataLoader(
        WordDataset(X_train_encoded, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    test_loader = DataLoader(
        WordDataset(X_test_encoded, y_test),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    return train_loader, test_loader, embedding_matrix


def train_torch_model(model, train_loader, test_loader, epochs=3, lr=1e-3):
    import torch
    import torch.nn as nn

    device = torch_device()
    print(f"Using device: {device}")

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=lr,
    )

    for epoch in range(epochs):
        model.train()
        losses = []

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)

            if "attention_mask" in batch:
                attention_mask = batch["attention_mask"].to(device)
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
            else:
                logits = model(input_ids=input_ids)

            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())

        print(f"    epoch {epoch + 1}/{epochs}, loss={np.mean(losses):.4f}")

    model.eval()

    y_true = []
    y_pred = []
    y_score = []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].cpu().numpy()

            if "attention_mask" in batch:
                attention_mask = batch["attention_mask"].to(device)
                logits = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )
            else:
                logits = model(input_ids=input_ids)

            probabilities = torch.softmax(logits, dim=1).cpu().numpy()
            predictions = probabilities.argmax(axis=1)

            y_true.extend(labels.tolist())
            y_pred.extend(predictions.tolist())
            y_score.extend(probabilities[:, POSITIVE_LABEL].tolist())

    return compute_metrics(
        np.array(y_true),
        np.array(y_pred),
        np.array(y_score),
    )


# ============================================================
# 8. Word2Vec neural models
# ============================================================

def run_word2vec_neural(X_train, X_test, y_train, y_test, out_dir, epochs=8):
    import torch
    import torch.nn as nn

    train_loader, test_loader, embedding_matrix = make_word_loaders(
        X_train,
        X_test,
        y_train,
        y_test,
    )

    def make_embedding_layer():
        weights = torch.tensor(embedding_matrix, dtype=torch.float32)

        return nn.Embedding.from_pretrained(
            weights,
            freeze=False,
            padding_idx=0,
        )

    class CNNWord2Vec(nn.Module):
        def __init__(self):
            super().__init__()
            self.embedding = make_embedding_layer()
            self.conv = nn.Conv1d(
                WORD2VEC_DIM,
                128,
                kernel_size=5,
                padding=2,
            )
            self.relu = nn.ReLU()
            self.pool = nn.AdaptiveMaxPool1d(1)
            self.dropout = nn.Dropout(0.4)
            self.fc = nn.Linear(128, 2)

        def forward(self, input_ids):
            x = self.embedding(input_ids)
            x = x.transpose(1, 2)
            x = self.relu(self.conv(x))
            x = self.pool(x).squeeze(-1)
            x = self.dropout(x)
            return self.fc(x)

    class LSTMWord2Vec(nn.Module):
        def __init__(self):
            super().__init__()
            self.embedding = make_embedding_layer()
            self.lstm = nn.LSTM(
                input_size=WORD2VEC_DIM,
                hidden_size=300,
                batch_first=True,
            )
            self.dropout = nn.Dropout(0.6)
            self.fc = nn.Linear(300, 2)

        def forward(self, input_ids):
            x = self.embedding(input_ids)
            _, (h, _) = self.lstm(x)
            h = self.dropout(h[-1])
            return self.fc(h)

    class CNNLSTMWord2Vec(nn.Module):
        def __init__(self):
            super().__init__()
            self.embedding = make_embedding_layer()
            self.conv = nn.Conv1d(
                WORD2VEC_DIM,
                128,
                kernel_size=5,
                padding=2,
            )
            self.relu = nn.ReLU()
            self.lstm = nn.LSTM(
                input_size=128,
                hidden_size=300,
                batch_first=True,
            )
            self.dropout = nn.Dropout(0.6)
            self.fc = nn.Linear(300, 2)

        def forward(self, input_ids):
            x = self.embedding(input_ids)
            x = x.transpose(1, 2)
            x = self.relu(self.conv(x))
            x = x.transpose(1, 2)
            _, (h, _) = self.lstm(x)
            h = self.dropout(h[-1])
            return self.fc(h)

    model_map = {
        "CNN+Word2Vec": CNNWord2Vec,
        "LSTM+Word2Vec": LSTMWord2Vec,
        "CNN-LSTM+Word2Vec": CNNLSTMWord2Vec,
    }

    rows = []

    for model_name, model_class in model_map.items():
        print(f"\nRunning {model_name}")
        set_seed(SEED)

        model = model_class()
        metrics = train_torch_model(
            model,
            train_loader,
            test_loader,
            epochs=epochs,
            lr=1e-3,
        )

        rows.append(
            {
                "model_family": "word2vec_neural",
                "model": model_name,
                **metrics,
            }
        )

    results = pd.DataFrame(rows)
    results.to_csv(out_dir / "word2vec_neural_results_raw.csv", index=False)
    percent_table(results).to_csv(
        out_dir / "word2vec_neural_results_percent.csv",
        index=False,
    )

    return results


# ============================================================
# 9. Transformer data loaders
# ============================================================

def make_transformer_loaders(X_train, X_test, y_train, y_test, model_name):
    import torch
    from torch.utils.data import DataLoader, Dataset
    from transformers import AutoTokenizer

    tokenizer_hf = AutoTokenizer.from_pretrained(model_name)

    train_encoded = tokenizer_hf(
        list(X_train),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    test_encoded = tokenizer_hf(
        list(X_test),
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN,
        return_tensors="pt",
    )

    class TransformerDataset(Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = torch.tensor(labels, dtype=torch.long)

        def __len__(self):
            return len(self.labels)

        def __getitem__(self, index):
            return {
                "input_ids": self.encodings["input_ids"][index],
                "attention_mask": self.encodings["attention_mask"][index],
                "labels": self.labels[index],
            }

    train_loader = DataLoader(
        TransformerDataset(train_encoded, y_train),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    test_loader = DataLoader(
        TransformerDataset(test_encoded, y_test),
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    return train_loader, test_loader


# ============================================================
# 10. Frozen DistilBERT embedding models
# ============================================================

def run_frozen_distilbert_models(
    X_train,
    X_test,
    y_train,
    y_test,
    out_dir,
    epochs=3,
):
    import torch
    import torch.nn as nn
    from transformers import AutoModel

    model_name = "distilbert-base-uncased"

    train_loader, test_loader = make_transformer_loaders(
        X_train,
        X_test,
        y_train,
        y_test,
        model_name,
    )

    class FrozenBase(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder = AutoModel.from_pretrained(model_name)

            for parameter in self.encoder.parameters():
                parameter.requires_grad = False

        def encode(self, input_ids, attention_mask):
            with torch.no_grad():
                output = self.encoder(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )

            return output.last_hidden_state

    class CNNFrozenDistilBERT(FrozenBase):
        def __init__(self):
            super().__init__()
            self.conv = nn.Conv1d(768, 128, kernel_size=5, padding=2)
            self.relu = nn.ReLU()
            self.pool = nn.AdaptiveMaxPool1d(1)
            self.dropout = nn.Dropout(0.4)
            self.fc = nn.Linear(128, 2)

        def forward(self, input_ids, attention_mask):
            x = self.encode(input_ids, attention_mask)
            x = x.transpose(1, 2)
            x = self.relu(self.conv(x))
            x = self.pool(x).squeeze(-1)
            x = self.dropout(x)
            return self.fc(x)

    class LSTMFrozenDistilBERT(FrozenBase):
        def __init__(self):
            super().__init__()
            self.lstm = nn.LSTM(
                input_size=768,
                hidden_size=300,
                batch_first=True,
            )
            self.dropout = nn.Dropout(0.6)
            self.fc = nn.Linear(300, 2)

        def forward(self, input_ids, attention_mask):
            x = self.encode(input_ids, attention_mask)
            _, (h, _) = self.lstm(x)
            h = self.dropout(h[-1])
            return self.fc(h)

    class CNNLSTMFrozenDistilBERT(FrozenBase):
        def __init__(self):
            super().__init__()
            self.conv = nn.Conv1d(768, 128, kernel_size=5, padding=2)
            self.relu = nn.ReLU()
            self.lstm = nn.LSTM(
                input_size=128,
                hidden_size=300,
                batch_first=True,
            )
            self.dropout = nn.Dropout(0.6)
            self.fc = nn.Linear(300, 2)

        def forward(self, input_ids, attention_mask):
            x = self.encode(input_ids, attention_mask)
            x = x.transpose(1, 2)
            x = self.relu(self.conv(x))
            x = x.transpose(1, 2)
            _, (h, _) = self.lstm(x)
            h = self.dropout(h[-1])
            return self.fc(h)

    model_map = {
        "CNN+FrozenDistilBERT": CNNFrozenDistilBERT,
        "LSTM+FrozenDistilBERT": LSTMFrozenDistilBERT,
        "CNN-LSTM+FrozenDistilBERT": CNNLSTMFrozenDistilBERT,
    }

    rows = []

    for model_name, model_class in model_map.items():
        print(f"\nRunning {model_name}")
        set_seed(SEED)

        model = model_class()
        metrics = train_torch_model(
            model,
            train_loader,
            test_loader,
            epochs=epochs,
            lr=1e-3,
        )

        rows.append(
            {
                "model_family": "frozen_distilbert",
                "model": model_name,
                **metrics,
            }
        )

    results = pd.DataFrame(rows)
    results.to_csv(
        out_dir / "frozen_distilbert_results_raw.csv",
        index=False,
    )

    percent_table(results).to_csv(
        out_dir / "frozen_distilbert_results_percent.csv",
        index=False,
    )

    return results


# ============================================================
# 11. Fine-tuned transformer models
# ============================================================

def run_finetuned_transformers(
    X_train,
    X_test,
    y_train,
    y_test,
    out_dir,
    epochs=3,
):
    import torch.nn as nn
    from transformers import AutoModelForSequenceClassification

    model_map = {
        "FineTunedDistilBERT": "distilbert-base-uncased",
        "FineTunedRoBERTa": "roberta-base",
    }

    rows = []

    for display_name, model_name in model_map.items():
        print(f"\nFine-tuning {display_name}")
        set_seed(SEED)

        train_loader, test_loader = make_transformer_loaders(
            X_train,
            X_test,
            y_train,
            y_test,
            model_name,
        )

        class HFClassifier(nn.Module):
            def __init__(self):
                super().__init__()
                self.model = AutoModelForSequenceClassification.from_pretrained(
                    model_name,
                    num_labels=2,
                )

            def forward(self, input_ids, attention_mask):
                output = self.model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                )

                return output.logits

        model = HFClassifier()

        metrics = train_torch_model(
            model,
            train_loader,
            test_loader,
            epochs=epochs,
            lr=2e-5,
        )

        rows.append(
            {
                "model_family": "finetuned_transformer",
                "model": display_name,
                **metrics,
            }
        )

    results = pd.DataFrame(rows)
    results.to_csv(
        out_dir / "finetuned_transformer_results_raw.csv",
        index=False,
    )

    percent_table(results).to_csv(
        out_dir / "finetuned_transformer_results_percent.csv",
        index=False,
    )

    return results


# ============================================================
# 12. Generic summary tables
# ============================================================

def create_summary_tables(out_dir):
    raw_files = [
        out_dir / "word2vec_neural_results_raw.csv",
        out_dir / "frozen_distilbert_results_raw.csv",
        out_dir / "finetuned_transformer_results_raw.csv",
    ]

    frames = []

    for file in raw_files:
        if file.exists():
            frames.append(pd.read_csv(file))

    if not frames:
        return

    all_results = pd.concat(frames, ignore_index=True)
    all_results.to_csv(
        out_dir / "all_neural_transformer_results_raw.csv",
        index=False,
    )

    display_results = percent_table(all_results)
    display_results.to_csv(
        out_dir / "all_neural_transformer_results_percent.csv",
        index=False,
    )

    selected_columns = [
        "model_family",
        "model",
        "accuracy",
        "f1_primary",
        "precision_primary",
        "recall_primary",
        "roc_auc_primary",
        "f1_macro",
        "f1_weighted",
    ]

    selected_columns = [
        column for column in selected_columns if column in display_results.columns
    ]

    deep_learning_models = [
        "CNN+Word2Vec",
        "LSTM+Word2Vec",
        "CNN+FrozenDistilBERT",
        "LSTM+FrozenDistilBERT",
    ]

    hybrid_models = [
        "CNN-LSTM+Word2Vec",
        "CNN-LSTM+FrozenDistilBERT",
    ]

    transformer_models = [
        "FineTunedDistilBERT",
        "FineTunedRoBERTa",
    ]

    deep_learning_summary = display_results[
        display_results["model"].isin(deep_learning_models)
    ][selected_columns].copy()

    hybrid_summary = display_results[
        display_results["model"].isin(hybrid_models)
    ][selected_columns].copy()

    transformer_summary = display_results[
        display_results["model"].isin(transformer_models)
    ][selected_columns].copy()

    if len(deep_learning_summary) > 0:
        deep_learning_summary.to_csv(
            out_dir / "deep_learning_summary.csv",
            index=False,
        )

    if len(hybrid_summary) > 0:
        hybrid_summary.to_csv(
            out_dir / "hybrid_model_summary.csv",
            index=False,
        )

    if len(transformer_summary) > 0:
        transformer_summary.to_csv(
            out_dir / "transformer_model_summary.csv",
            index=False,
        )


# ============================================================
# 13. Main
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--fake", type=str, default=DEFAULT_FAKE_FILE)
    parser.add_argument("--true", type=str, default=DEFAULT_TRUE_FILE)
    parser.add_argument("--out", type=str, default=DEFAULT_OUTPUT_DIR)

    parser.add_argument("--run_ml", action="store_true")
    parser.add_argument("--run_word2vec_dl", action="store_true")
    parser.add_argument("--run_frozen_distilbert", action="store_true")
    parser.add_argument("--run_transformers", action="store_true")
    parser.add_argument("--run_all", action="store_true")

    parser.add_argument("--epochs_word", type=int, default=8)
    parser.add_argument("--epochs_frozen", type=int, default=3)
    parser.add_argument("--epochs_transformers", type=int, default=3)

    args, _ = parser.parse_known_args()

    set_seed(SEED)
    download_nltk_resources()

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    fake_path, true_path = upload_data_if_needed(args.fake, args.true)

    df = load_data(fake_path, true_path)

    dataset_summary = save_dataset_summary(df, out_dir)

    X_train, X_test, y_train, y_test = fixed_split(df)

    split_summary = save_split_summary(y_train, y_test, out_dir)

    print("\nDataset summary")
    print(dataset_summary.to_string(index=False))

    print("\nDataset split summary")
    print(split_summary.to_string(index=False))

    if args.run_all:
        args.run_ml = True
        args.run_word2vec_dl = True
        args.run_frozen_distilbert = True
        args.run_transformers = True

    if not any(
        [
            args.run_ml,
            args.run_word2vec_dl,
            args.run_frozen_distilbert,
            args.run_transformers,
        ]
    ):
        print("\nNo run flag supplied. Running conventional ML only.")
        args.run_ml = True

    if args.run_ml:
        run_conventional_ml(
            X_train,
            X_test,
            y_train,
            y_test,
            out_dir,
        )

    if args.run_word2vec_dl:
        run_word2vec_neural(
            X_train,
            X_test,
            y_train,
            y_test,
            out_dir,
            epochs=args.epochs_word,
        )

    if args.run_frozen_distilbert:
        run_frozen_distilbert_models(
            X_train,
            X_test,
            y_train,
            y_test,
            out_dir,
            epochs=args.epochs_frozen,
        )

    if args.run_transformers:
        run_finetuned_transformers(
            X_train,
            X_test,
            y_train,
            y_test,
            out_dir,
            epochs=args.epochs_transformers,
        )

    create_summary_tables(out_dir)

    print("\nFinished. Output files:")

    for file in sorted(out_dir.glob("*")):
        print("  ", file)

    try:
        from IPython.display import display

        selected_outputs = [
            out_dir / "dataset_summary.csv",
            out_dir / "split_summary.csv",
            out_dir / "conventional_ml_summary.csv",
            out_dir / "deep_learning_summary.csv",
            out_dir / "hybrid_model_summary.csv",
            out_dir / "transformer_model_summary.csv",
        ]

        for output_file in selected_outputs:
            if output_file.exists():
                print("\n" + "=" * 90)
                print(output_file)
                print("=" * 90)
                display(pd.read_csv(output_file))

    except Exception:
        pass


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# Multi-branch fusion model for health misinformation detection
# Single uninterrupted Colab-ready script
# Monkeypox dataset is balanced before splitting
# ============================================================

import argparse
import copy
import html
import json
import os
import random
import re
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

# ============================================================
# 0. Package installation for Colab
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "numpy",
        "scikit-learn",
        "nltk",
        "torch",
        "transformers",
        "accelerate",
    ],
    check=True,
)

import numpy as np
import pandas as pd

from scipy.special import rel_entr
from scipy.stats import spearmanr

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore")


# ============================================================
# 1. General configuration
# ============================================================

SEED = 42
POSITIVE_LABEL = 1
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10
PATIENCE = 3
FREEZE_EPOCHS = 5
LEARNING_RATE = 2e-5
DROPOUT = 0.3
HIDDEN_DIM = 256

LAMBDA_CLS = 1.0
LAMBDA_REG = 0.25
LAMBDA_CPS = 0.25

DEFAULT_DATA_FILE = "monkeypox_dataset.csv"
DEFAULT_OUTPUT_DIR = "results_fusion_balanced_monkeypox"
TRANSFORMER_MODEL_NAME = "distilbert-base-uncased"


# ============================================================
# 2. Data upload support
# ============================================================

def upload_data_if_needed(default_file=DEFAULT_DATA_FILE):
    default_path = Path(default_file)

    if default_path.exists():
        return str(default_path)

    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)

    data_path = data_dir / default_path.name

    if data_path.exists():
        return str(data_path)

    try:
        from google.colab import files

        print("Upload monkeypox_dataset.csv")
        uploaded = files.upload()

        for filename in uploaded.keys():
            source = Path(filename)
            destination = data_dir / filename

            if destination.exists():
                destination.unlink()

            shutil.move(str(source), str(destination))

        if data_path.exists():
            return str(data_path)

        csv_files = sorted(data_dir.glob("*.csv"))

        if len(csv_files) == 1:
            return str(csv_files[0])

        for file in csv_files:
            if "monkeypox" in file.name.lower():
                return str(file)

        raise FileNotFoundError("No suitable Monkeypox CSV file detected.")

    except Exception as exc:
        raise FileNotFoundError(
            "Dataset not found. Place monkeypox_dataset.csv in the current folder "
            "or in a data/ folder, or upload it when prompted in Colab."
        ) from exc


# ============================================================
# 3. Reproducibility and text utilities
# ============================================================

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass


def download_nltk_resources():
    try:
        import nltk

        for package in ["vader_lexicon"]:
            try:
                nltk.data.find(f"sentiment/{package}")
            except LookupError:
                nltk.download(package, quiet=True)
    except Exception:
        pass


set_seed(SEED)
download_nltk_resources()


def fix_text_encoding(text):
    if pd.isna(text):
        return ""

    text = str(text)

    try:
        text = text.encode("latin1").decode("utf-8")
    except Exception:
        pass

    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_text(text):
    text = fix_text_encoding(text)
    text = re.sub(r"http\S+|www\.\S+|_URL_", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"[^A-Za-z0-9#@!?.,;:'%_\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    return re.findall(r"[a-zA-Z][a-zA-Z']+|\d+", str(text).lower())


def sentence_count(text):
    sentences = re.split(r"[.!?]+", str(text))
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    return max(1, len(sentences))


def syllable_count_word(word):
    word = str(word).lower()
    word = re.sub(r"[^a-z]", "", word)

    if not word:
        return 0

    groups = re.findall(r"[aeiouy]+", word)
    count = len(groups)

    if word.endswith("e") and count > 1:
        count -= 1

    return max(1, count)


def syllable_count_text(tokens):
    return sum(syllable_count_word(token) for token in tokens)


def safe_divide(numerator, denominator):
    if denominator == 0:
        return 0.0

    return float(numerator) / float(denominator)


def get_vader():
    try:
        from nltk.sentiment import SentimentIntensityAnalyzer

        return SentimentIntensityAnalyzer()
    except Exception:
        return None


VADER = get_vader()


def sentiment_compound(text):
    if VADER is None:
        return 0.0

    try:
        return float(VADER.polarity_scores(str(text))["compound"])
    except Exception:
        return 0.0


def vader_valence_score(tokens):
    if VADER is None:
        return 0.0

    total = 0.0

    for token in tokens:
        total += abs(float(VADER.lexicon.get(token.lower(), 0.0)))

    return safe_divide(total, len(tokens))


# ============================================================
# 4. Data loading, label mapping and balancing
# ============================================================

def detect_text_column(df):
    candidates = [
        "text",
        "Text",
        "tweet",
        "Tweet",
        "body",
        "Body",
        "content",
        "Content",
        "statement",
        "Statement",
    ]

    for column in candidates:
        if column in df.columns:
            return column

    raise ValueError(
        "No text column found. Expected one of: text, Text, tweet, body, content, statement."
    )


def detect_label_column(df):
    candidates = [
        "binary_class",
        "Binary Label",
        "binary_label",
        "label",
        "Label",
        "class",
        "Class",
        "target",
        "Target",
    ]

    for column in candidates:
        if column in df.columns:
            return column

    raise ValueError(
        "No label column found. Expected one of: binary_class, Binary Label, label, class, target."
    )


def map_labels(series):
    values = series.copy()

    if pd.api.types.is_numeric_dtype(values):
        values = pd.to_numeric(values, errors="coerce").astype("Int64")

        unique_values = sorted([int(value) for value in values.dropna().unique()])

        if set(unique_values).issubset({0, 1}):
            return values.astype(int)

        if len(unique_values) == 2:
            mapping = {
                unique_values[0]: 0,
                unique_values[1]: 1,
            }
            return values.map(mapping).astype(int)

        raise ValueError("Numeric label column must contain exactly two classes.")

    lower = values.astype(str).str.strip().str.lower()

    mapping = {
        "fake": 1,
        "false": 1,
        "misinformation": 1,
        "misleading": 1,
        "rumour": 1,
        "rumor": 1,
        "unreliable": 1,
        "1": 1,
        "true": 0,
        "real": 0,
        "factual": 0,
        "credible": 0,
        "reliable": 0,
        "0": 0,
    }

    mapped = lower.map(mapping)

    if mapped.isna().any():
        unknown = sorted(lower[mapped.isna()].unique().tolist())
        raise ValueError(f"Unmapped label values found: {unknown}")

    return mapped.astype(int)


def detect_engagement_columns(df):
    candidates = {
        "retweet_count": ["retweet_count", "retweets", "retweet", "retweeted"],
        "reply_count": ["reply_count", "replies", "reply"],
        "like_count": ["like_count", "likes", "favorite_count", "favourites", "favorites"],
        "quote_count": ["quote_count", "quotes", "quote"],
        "share_count": ["share_count", "shares", "share"],
    }

    found = {}

    lower_map = {
        str(column).lower().strip(): column
        for column in df.columns
    }

    for standard_name, options in candidates.items():
        for option in options:
            if option.lower() in lower_map:
                found[standard_name] = lower_map[option.lower()]
                break

    return found


def load_dataset(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    df = pd.read_csv(path, low_memory=False)

    text_column = detect_text_column(df)
    label_column = detect_label_column(df)
    engagement_columns = detect_engagement_columns(df)

    output = pd.DataFrame()
    output["text"] = df[text_column].astype(str)
    output["label"] = map_labels(df[label_column])
    output["text_clean"] = output["text"].map(clean_text)

    for standard_name, original_column in engagement_columns.items():
        output[standard_name] = pd.to_numeric(df[original_column], errors="coerce").fillna(0.0)

    for standard_name in ["retweet_count", "reply_count", "like_count", "quote_count", "share_count"]:
        if standard_name not in output.columns:
            output[standard_name] = 0.0

    output["engagement_raw"] = (
        output["retweet_count"]
        + output["reply_count"]
        + output["like_count"]
        + output["quote_count"]
        + output["share_count"]
    )

    output = output[output["text_clean"].str.len() > 0].reset_index(drop=True)

    metadata = {
        "input_file": str(path),
        "text_column": text_column,
        "label_column": label_column,
        "engagement_columns_detected": engagement_columns,
        "positive_label": POSITIVE_LABEL,
    }

    return output, metadata


def balance_binary_dataset(df):
    class_counts_before = df["label"].value_counts().sort_index().to_dict()
    minimum_class_size = int(df["label"].value_counts().min())

    balanced_parts = []

    for label_value, group in df.groupby("label"):
        balanced_parts.append(
            group.sample(
                n=minimum_class_size,
                random_state=SEED,
                replace=False,
            )
        )

    balanced = (
        pd.concat(balanced_parts, ignore_index=True)
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )

    class_counts_after = balanced["label"].value_counts().sort_index().to_dict()

    summary = {
        "balancing_method": "random_undersampling_to_minority_class",
        "seed": SEED,
        "n_before": int(len(df)),
        "n_after": int(len(balanced)),
        "class_counts_before": class_counts_before,
        "class_counts_after": class_counts_after,
        "minority_class_size": minimum_class_size,
    }

    return balanced, summary


def create_splits(df):
    train_val_df, test_df = train_test_split(
        df,
        test_size=TEST_SIZE,
        random_state=SEED,
        stratify=df["label"],
    )

    validation_fraction_of_train_val = VALIDATION_SIZE / (1.0 - TEST_SIZE)

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=validation_fraction_of_train_val,
        random_state=SEED,
        stratify=train_val_df["label"],
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    return train_df, val_df, test_df


def save_data_summaries(original_df, balanced_df, balance_summary, split_dfs, metadata, out_dir):
    train_df, val_df, test_df = split_dfs

    dataset_summary = pd.DataFrame(
        [
            {
                "input_file": metadata["input_file"],
                "text_column": metadata["text_column"],
                "label_column": metadata["label_column"],
                "positive_label": metadata["positive_label"],
                "n_original": int(len(original_df)),
                "n_balanced": int(len(balanced_df)),
                "original_label_0": int((original_df["label"] == 0).sum()),
                "original_label_1": int((original_df["label"] == 1).sum()),
                "balanced_label_0": int((balanced_df["label"] == 0).sum()),
                "balanced_label_1": int((balanced_df["label"] == 1).sum()),
                "engagement_available": bool(original_df["engagement_raw"].sum() > 0),
                "engagement_raw_sum_balanced": float(balanced_df["engagement_raw"].sum()),
                "exact_duplicate_clean_text_balanced": int(balanced_df["text_clean"].duplicated().sum()),
            }
        ]
    )

    split_summary = pd.DataFrame(
        [
            {
                "split": "train",
                "n": int(len(train_df)),
                "label_0": int((train_df["label"] == 0).sum()),
                "label_1": int((train_df["label"] == 1).sum()),
            },
            {
                "split": "validation",
                "n": int(len(val_df)),
                "label_0": int((val_df["label"] == 0).sum()),
                "label_1": int((val_df["label"] == 1).sum()),
            },
            {
                "split": "test",
                "n": int(len(test_df)),
                "label_0": int((test_df["label"] == 0).sum()),
                "label_1": int((test_df["label"] == 1).sum()),
            },
        ]
    )

    dataset_summary.to_csv(out_dir / "dataset_summary.csv", index=False)
    split_summary.to_csv(out_dir / "split_summary.csv", index=False)

    with open(out_dir / "balance_summary.json", "w", encoding="utf-8") as file:
        json.dump(balance_summary, file, indent=2)

    with open(out_dir / "dataset_metadata.json", "w", encoding="utf-8") as file:
        json.dump(metadata, file, indent=2)

    return dataset_summary, split_summary


# ============================================================
# 5. Feature engineering
# ============================================================

DISCOURSE_CUES = {
    "however", "therefore", "because", "although", "whereas", "while", "moreover",
    "furthermore", "nevertheless", "but", "so", "since", "hence", "thus", "thereby",
}

ATTRIBUTION_CUES = {
    "according", "source", "reported", "report", "study", "studies", "evidence",
    "research", "scientists", "experts", "cdc", "who", "nhs", "journal",
}

ELABORATION_CUES = {
    "because", "therefore", "evidence", "study", "studies", "data", "research",
    "analysis", "however", "although", "compared", "therefore", "suggests",
}

SENSATIONAL_CUES = {
    "shocking", "secret", "truth", "exposed", "wake", "urgent", "breaking",
    "coverup", "cover-up", "hoax", "scandal", "danger", "must", "now",
}

URGENCY_CUES = {
    "now", "urgent", "immediately", "breaking", "alert", "act", "today", "before",
}

SUPPORT_CUES = {
    "true", "correct", "accurate", "confirmed", "evidence", "agree", "yes",
    "proven", "valid", "fact",
}

DENIAL_CUES = {
    "fake", "false", "hoax", "lie", "lies", "wrong", "debunk", "misleading",
    "not", "never", "no",
}

QUERY_CUES = {
    "what", "why", "how", "when", "where", "who", "really", "is", "are", "can",
}

MODAL_VERBS = {
    "must", "should", "could", "would", "might", "may", "will", "shall", "ought",
}

GROUP_PRONOUNS = {
    "we", "us", "our", "ours", "ourselves", "everyone", "people", "public",
}

ALL_PRONOUNS = {
    "i", "me", "my", "mine", "we", "us", "our", "ours", "you", "your", "yours",
    "he", "him", "his", "she", "her", "hers", "they", "them", "their", "theirs",
}

SOCIAL_COMPARISON_CUES = {
    "everyone", "others", "people", "nobody", "noone", "many", "most", "public",
    "community", "they", "them",
}

CERTAINTY_CUES = {
    "definitely", "certainly", "surely", "clearly", "obviously", "undeniable",
    "proven", "guaranteed", "always", "never", "fact",
}

INSTRUCTIONAL_CUES = {
    "share", "click", "read", "watch", "listen", "check", "follow", "join",
    "wake", "stop", "avoid", "take", "use",
}

DIRECTIVE_HASHTAGS = {
    "wakeup", "joinus", "sharethis", "stopthelies", "truth", "actnow", "readthis",
}


def count_tokens_in_set(tokens, vocabulary):
    return sum(1 for token in tokens if token in vocabulary)


def extract_single_feature_set(text):
    raw_text = str(text)
    clean = clean_text(raw_text)
    tokens = tokenize(clean)
    token_count = max(1, len(tokens))
    unique_count = len(set(tokens))
    sentences = sentence_count(clean)
    syllables = syllable_count_text(tokens)
    letters = re.findall(r"[A-Za-z]", raw_text)
    uppercase_letters = re.findall(r"[A-Z]", raw_text)

    fkgl = (
        0.39 * safe_divide(token_count, sentences)
        + 11.8 * safe_divide(syllables, token_count)
        - 15.59
    )

    ttr = safe_divide(unique_count, token_count)
    polarity = sentiment_compound(clean)
    average_sentence_length = safe_divide(token_count, sentences)

    exclamation_ratio = safe_divide(raw_text.count("!"), token_count)
    question_ratio = safe_divide(raw_text.count("?"), token_count)
    uppercase_ratio = safe_divide(len(uppercase_letters), len(letters))
    all_caps_count = len(re.findall(r"\b[A-Z]{2,}\b", raw_text))
    urgency_ratio = safe_divide(count_tokens_in_set(tokens, URGENCY_CUES), token_count)

    rhetoric = [
        safe_divide(count_tokens_in_set(tokens, DISCOURSE_CUES), token_count),
        safe_divide(count_tokens_in_set(tokens, ATTRIBUTION_CUES) + clean.count("URL"), token_count),
        safe_divide(count_tokens_in_set(tokens, ELABORATION_CUES), token_count),
        safe_divide(count_tokens_in_set(tokens, SENSATIONAL_CUES), token_count),
    ]

    support_score = count_tokens_in_set(tokens, SUPPORT_CUES)
    denial_score = count_tokens_in_set(tokens, DENIAL_CUES)
    query_score = count_tokens_in_set(tokens, QUERY_CUES) + raw_text.count("?")
    comment_score = max(1.0, token_count / 20.0)

    stance_raw = np.array(
        [
            support_score,
            denial_score,
            query_score,
            comment_score,
        ],
        dtype=float,
    )

    stance = (stance_raw / stance_raw.sum()).tolist()

    elm = [
        fkgl,
        ttr,
        polarity,
        float(token_count),
        average_sentence_length,
        exclamation_ratio,
        question_ratio,
        uppercase_ratio,
        float(all_caps_count),
        urgency_ratio,
    ]

    modal_ratio = safe_divide(count_tokens_in_set(tokens, MODAL_VERBS), token_count)
    group_pronoun_count = count_tokens_in_set(tokens, GROUP_PRONOUNS)
    all_pronoun_count = count_tokens_in_set(tokens, ALL_PRONOUNS)
    mention_count = len(re.findall(r"@\w+|USER|\bRT\b", raw_text))
    social_comparison_ratio = safe_divide(count_tokens_in_set(tokens, SOCIAL_COMPARISON_CUES), token_count)
    certainty_ratio = safe_divide(count_tokens_in_set(tokens, CERTAINTY_CUES), token_count)
    instructional_ratio = safe_divide(count_tokens_in_set(tokens, INSTRUCTIONAL_CUES), token_count)

    hashtags = re.findall(r"#([A-Za-z0-9_]+)", raw_text)
    directive_hashtag_count = sum(1 for tag in hashtags if tag.lower() in DIRECTIVE_HASHTAGS)
    hashtag_directive_ratio = safe_divide(directive_hashtag_count, max(1, len(hashtags)))

    tpb = [
        polarity,
        vader_valence_score(tokens),
        modal_ratio,
        safe_divide(group_pronoun_count, all_pronoun_count),
        safe_divide(mention_count, token_count),
        social_comparison_ratio,
        certainty_ratio,
        instructional_ratio,
        hashtag_directive_ratio,
    ]

    return rhetoric, stance, elm, tpb


def extract_feature_matrices(df):
    rhetoric_rows = []
    stance_rows = []
    elm_rows = []
    tpb_rows = []

    for text in df["text"].tolist():
        rhetoric, stance, elm, tpb = extract_single_feature_set(text)
        rhetoric_rows.append(rhetoric)
        stance_rows.append(stance)
        elm_rows.append(elm)
        tpb_rows.append(tpb)

    rhetoric_features = np.asarray(rhetoric_rows, dtype=np.float32)
    stance_features = np.asarray(stance_rows, dtype=np.float32)
    elm_features = np.asarray(elm_rows, dtype=np.float32)
    tpb_features = np.asarray(tpb_rows, dtype=np.float32)

    return rhetoric_features, stance_features, elm_features, tpb_features


def fit_transform_feature_branches(train_df, val_df, test_df):
    train_rhet, train_stance, train_elm, train_tpb = extract_feature_matrices(train_df)
    val_rhet, val_stance, val_elm, val_tpb = extract_feature_matrices(val_df)
    test_rhet, test_stance, test_elm, test_tpb = extract_feature_matrices(test_df)

    scalers = {
        "rhetoric": MinMaxScaler(),
        "stance": MinMaxScaler(),
        "elm": MinMaxScaler(),
        "tpb": MinMaxScaler(),
    }

    train_rhet = scalers["rhetoric"].fit_transform(train_rhet)
    val_rhet = scalers["rhetoric"].transform(val_rhet)
    test_rhet = scalers["rhetoric"].transform(test_rhet)

    train_stance = scalers["stance"].fit_transform(train_stance)
    val_stance = scalers["stance"].transform(val_stance)
    test_stance = scalers["stance"].transform(test_stance)

    train_elm = scalers["elm"].fit_transform(train_elm)
    val_elm = scalers["elm"].transform(val_elm)
    test_elm = scalers["elm"].transform(test_elm)

    train_tpb = scalers["tpb"].fit_transform(train_tpb)
    val_tpb = scalers["tpb"].transform(val_tpb)
    test_tpb = scalers["tpb"].transform(test_tpb)

    feature_sets = {
        "train": {
            "rhetoric": train_rhet.astype(np.float32),
            "stance": train_stance.astype(np.float32),
            "elm": train_elm.astype(np.float32),
            "tpb": train_tpb.astype(np.float32),
        },
        "validation": {
            "rhetoric": val_rhet.astype(np.float32),
            "stance": val_stance.astype(np.float32),
            "elm": val_elm.astype(np.float32),
            "tpb": val_tpb.astype(np.float32),
        },
        "test": {
            "rhetoric": test_rhet.astype(np.float32),
            "stance": test_stance.astype(np.float32),
            "elm": test_elm.astype(np.float32),
            "tpb": test_tpb.astype(np.float32),
        },
    }

    return feature_sets, scalers


# ============================================================
# 6. Targets for regression and CPS
# ============================================================

def make_scaled_target(train_values, val_values, test_values):
    scaler = MinMaxScaler()

    train_scaled = scaler.fit_transform(np.asarray(train_values).reshape(-1, 1)).ravel()
    val_scaled = scaler.transform(np.asarray(val_values).reshape(-1, 1)).ravel()
    test_scaled = scaler.transform(np.asarray(test_values).reshape(-1, 1)).ravel()

    train_scaled = np.clip(train_scaled, 0, 1)
    val_scaled = np.clip(val_scaled, 0, 1)
    test_scaled = np.clip(test_scaled, 0, 1)

    return train_scaled.astype(np.float32), val_scaled.astype(np.float32), test_scaled.astype(np.float32), scaler


def create_targets(train_df, val_df, test_df, feature_sets):
    train_engagement = np.log1p(train_df["engagement_raw"].values)
    val_engagement = np.log1p(val_df["engagement_raw"].values)
    test_engagement = np.log1p(test_df["engagement_raw"].values)

    train_reg, val_reg, test_reg, engagement_scaler = make_scaled_target(
        train_engagement,
        val_engagement,
        test_engagement,
    )

    train_psych = np.concatenate(
        [feature_sets["train"]["elm"], feature_sets["train"]["tpb"]],
        axis=1,
    )
    val_psych = np.concatenate(
        [feature_sets["validation"]["elm"], feature_sets["validation"]["tpb"]],
        axis=1,
    )
    test_psych = np.concatenate(
        [feature_sets["test"]["elm"], feature_sets["test"]["tpb"]],
        axis=1,
    )

    train_cps_raw = train_psych.mean(axis=1)
    val_cps_raw = val_psych.mean(axis=1)
    test_cps_raw = test_psych.mean(axis=1)

    train_cps, val_cps, test_cps, cps_scaler = make_scaled_target(
        train_cps_raw,
        val_cps_raw,
        test_cps_raw,
    )

    targets = {
        "train": {
            "classification": train_df["label"].values.astype(np.float32),
            "regression": train_reg,
            "cps": train_cps,
            "engagement_raw": train_df["engagement_raw"].values.astype(np.float32),
        },
        "validation": {
            "classification": val_df["label"].values.astype(np.float32),
            "regression": val_reg,
            "cps": val_cps,
            "engagement_raw": val_df["engagement_raw"].values.astype(np.float32),
        },
        "test": {
            "classification": test_df["label"].values.astype(np.float32),
            "regression": test_reg,
            "cps": test_cps,
            "engagement_raw": test_df["engagement_raw"].values.astype(np.float32),
        },
    }

    scalers = {
        "engagement": engagement_scaler,
        "cps": cps_scaler,
    }

    return targets, scalers


# ============================================================
# 7. Baseline classifiers
# ============================================================

def run_baseline_classifiers(train_df, test_df, out_dir):
    vectorizer = TfidfVectorizer(
        lowercase=True,
        tokenizer=tokenize,
        token_pattern=None,
        ngram_range=(1, 2),
        max_features=30000,
        min_df=2,
        sublinear_tf=True,
    )

    X_train = vectorizer.fit_transform(train_df["text_clean"])
    X_test = vectorizer.transform(test_df["text_clean"])

    y_train = train_df["label"].values
    y_test = test_df["label"].values

    baselines = {
        "TFIDF_LogisticRegression": LogisticRegression(
            max_iter=3000,
            random_state=SEED,
            class_weight="balanced",
        ),
        "TFIDF_LinearSVM": LinearSVC(
            random_state=SEED,
            class_weight="balanced",
        ),
    }

    rows = []

    for name, model in baselines.items():
        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        score = None

        if hasattr(model, "predict_proba"):
            score = model.predict_proba(X_test)[:, POSITIVE_LABEL]

        if hasattr(model, "decision_function"):
            score = model.decision_function(X_test)

        rows.append(
            {
                "model": name,
                **classification_metrics(y_test, predictions, score),
            }
        )

    baseline_df = pd.DataFrame(rows)
    baseline_df.to_csv(out_dir / "baseline_classifier_results.csv", index=False)

    return baseline_df


# ============================================================
# 8. PyTorch dataset and model
# ============================================================

def torch_device():
    import torch

    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


class FusionDataset:
    def __init__(self, texts, features, targets, tokenizer):
        import torch
        from torch.utils.data import Dataset

        class _Dataset(Dataset):
            def __init__(self, texts, features, targets, tokenizer):
                self.texts = list(texts)
                self.features = features
                self.targets = targets
                self.tokenizer = tokenizer

                self.encoded = self.tokenizer(
                    self.texts,
                    truncation=True,
                    padding="max_length",
                    max_length=MAX_LEN,
                    return_tensors="pt",
                )

            def __len__(self):
                return len(self.texts)

            def __getitem__(self, index):
                item = {
                    "input_ids": self.encoded["input_ids"][index],
                    "attention_mask": self.encoded["attention_mask"][index],
                    "rhetoric": torch.tensor(self.features["rhetoric"][index], dtype=torch.float32),
                    "stance": torch.tensor(self.features["stance"][index], dtype=torch.float32),
                    "elm": torch.tensor(self.features["elm"][index], dtype=torch.float32),
                    "tpb": torch.tensor(self.features["tpb"][index], dtype=torch.float32),
                    "label": torch.tensor(self.targets["classification"][index], dtype=torch.float32),
                    "regression": torch.tensor(self.targets["regression"][index], dtype=torch.float32),
                    "cps": torch.tensor(self.targets["cps"][index], dtype=torch.float32),
                    "engagement_raw": torch.tensor(self.targets["engagement_raw"][index], dtype=torch.float32),
                }

                return item

        self.dataset = _Dataset(texts, features, targets, tokenizer)


def create_loaders(train_df, val_df, test_df, feature_sets, targets):
    import torch
    from torch.utils.data import DataLoader
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_MODEL_NAME)

    train_dataset = FusionDataset(
        train_df["text_clean"].tolist(),
        feature_sets["train"],
        targets["train"],
        tokenizer,
    ).dataset

    val_dataset = FusionDataset(
        val_df["text_clean"].tolist(),
        feature_sets["validation"],
        targets["validation"],
        tokenizer,
    ).dataset

    test_dataset = FusionDataset(
        test_df["text_clean"].tolist(),
        feature_sets["test"],
        targets["test"],
        tokenizer,
    ).dataset

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )

    return train_loader, val_loader, test_loader


class MultiBranchFusionModel:
    def __init__(self, enabled_branches):
        import torch
        import torch.nn as nn
        from transformers import AutoModel

        class _Model(nn.Module):
            def __init__(self, enabled_branches):
                super().__init__()

                self.enabled_branches = enabled_branches
                self.text_dim = 768
                self.rhetoric_dim = 4
                self.stance_dim = 4
                self.elm_dim = 10
                self.tpb_dim = 9
                self.fusion_dim = 795

                if self.enabled_branches.get("text", True):
                    self.transformer = AutoModel.from_pretrained(TRANSFORMER_MODEL_NAME)
                else:
                    self.transformer = None

                self.dropout = nn.Dropout(DROPOUT)

                self.fusion = nn.Sequential(
                    nn.Linear(self.fusion_dim, HIDDEN_DIM),
                    nn.ReLU(),
                    nn.Dropout(DROPOUT),
                    nn.Linear(HIDDEN_DIM, HIDDEN_DIM // 2),
                    nn.ReLU(),
                    nn.Dropout(DROPOUT),
                )

                self.classification_head = nn.Linear(HIDDEN_DIM // 2, 1)
                self.regression_head = nn.Linear(HIDDEN_DIM // 2, 1)
                self.cps_head = nn.Linear(HIDDEN_DIM // 2, 1)

            def set_transformer_trainability(self, trainable):
                if self.transformer is None:
                    return

                for parameter in self.transformer.parameters():
                    parameter.requires_grad = trainable

            def forward(self, input_ids, attention_mask, rhetoric, stance, elm, tpb):
                batch_size = input_ids.size(0)
                device = input_ids.device

                if self.transformer is not None and self.enabled_branches.get("text", True):
                    transformer_output = self.transformer(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                    )
                    text_vector = transformer_output.last_hidden_state[:, 0, :]
                else:
                    text_vector = torch.zeros(batch_size, self.text_dim, device=device)

                if not self.enabled_branches.get("rhetoric", True):
                    rhetoric = torch.zeros_like(rhetoric)

                if not self.enabled_branches.get("stance", True):
                    stance = torch.zeros_like(stance)

                if not self.enabled_branches.get("psychological", True):
                    elm = torch.zeros_like(elm)
                    tpb = torch.zeros_like(tpb)

                fused = torch.cat(
                    [
                        text_vector,
                        rhetoric,
                        stance,
                        elm,
                        tpb,
                    ],
                    dim=1,
                )

                hidden = self.fusion(self.dropout(fused))

                classification_logit = self.classification_head(hidden).squeeze(-1)
                regression_output = torch.sigmoid(self.regression_head(hidden).squeeze(-1))
                cps_output = torch.sigmoid(self.cps_head(hidden).squeeze(-1))

                return classification_logit, regression_output, cps_output

        self.model = _Model(enabled_branches)


# ============================================================
# 9. Training and evaluation
# ============================================================

def classification_metrics(y_true, y_pred, y_score=None):
    output = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
    }

    if y_score is not None and len(np.unique(y_true)) == 2:
        try:
            output["roc_auc"] = roc_auc_score(y_true, y_score)
        except Exception:
            output["roc_auc"] = np.nan
    else:
        output["roc_auc"] = np.nan

    return output


def kl_divergence_histogram(y_true, y_pred, bins=20):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    hist_true, bin_edges = np.histogram(y_true, bins=bins, range=(0, 1), density=False)
    hist_pred, _ = np.histogram(y_pred, bins=bin_edges, density=False)

    hist_true = hist_true.astype(float) + 1e-9
    hist_pred = hist_pred.astype(float) + 1e-9

    hist_true = hist_true / hist_true.sum()
    hist_pred = hist_pred / hist_pred.sum()

    return float(np.sum(rel_entr(hist_true, hist_pred)))


def propagation_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mse = mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    if len(np.unique(y_true)) > 1 and len(np.unique(y_pred)) > 1:
        rho = float(spearmanr(y_true, y_pred).correlation)
    else:
        rho = np.nan

    top_k = min(5, len(y_true))

    if top_k > 0:
        true_top = set(np.argsort(y_true)[-top_k:])
        pred_top = set(np.argsort(y_pred)[-top_k:])
        top_5_hit_rate = len(true_top & pred_top) / top_k
    else:
        top_5_hit_rate = np.nan

    threshold = float(np.percentile(y_true, 90))
    true_viral = y_true >= threshold
    pred_viral = y_pred >= threshold

    viral_precision = precision_score(
        true_viral,
        pred_viral,
        zero_division=0,
    )

    viral_recall = recall_score(
        true_viral,
        pred_viral,
        zero_division=0,
    )

    kld = kl_divergence_histogram(y_true, y_pred)

    return {
        "mse_propagation": mse,
        "mae_propagation": mae,
        "spearman_rank": rho,
        "top_5_hit_rate": top_5_hit_rate,
        "viral_threshold": threshold,
        "precision_viral": viral_precision,
        "recall_viral": viral_recall,
        "kl_divergence": kld,
    }


def collect_predictions(model, loader, device):
    import torch

    model.eval()

    y_true = []
    y_score = []
    y_pred = []
    reg_true = []
    reg_pred = []
    cps_true = []
    cps_pred = []
    engagement_raw = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            rhetoric = batch["rhetoric"].to(device)
            stance = batch["stance"].to(device)
            elm = batch["elm"].to(device)
            tpb = batch["tpb"].to(device)

            labels = batch["label"].cpu().numpy()
            regression = batch["regression"].cpu().numpy()
            cps = batch["cps"].cpu().numpy()
            engagement = batch["engagement_raw"].cpu().numpy()

            logits, reg_out, cps_out = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                rhetoric=rhetoric,
                stance=stance,
                elm=elm,
                tpb=tpb,
            )

            probabilities = torch.sigmoid(logits).cpu().numpy()
            predictions = (probabilities >= 0.5).astype(int)

            y_true.extend(labels.astype(int).tolist())
            y_score.extend(probabilities.tolist())
            y_pred.extend(predictions.tolist())
            reg_true.extend(regression.tolist())
            reg_pred.extend(reg_out.cpu().numpy().tolist())
            cps_true.extend(cps.tolist())
            cps_pred.extend(cps_out.cpu().numpy().tolist())
            engagement_raw.extend(engagement.tolist())

    return {
        "y_true": np.asarray(y_true),
        "y_score": np.asarray(y_score),
        "y_pred": np.asarray(y_pred),
        "reg_true": np.asarray(reg_true),
        "reg_pred": np.asarray(reg_pred),
        "cps_true": np.asarray(cps_true),
        "cps_pred": np.asarray(cps_pred),
        "engagement_raw": np.asarray(engagement_raw),
    }


def train_model(train_loader, val_loader, test_loader, enabled_branches, variant_name):
    import torch
    import torch.nn as nn

    set_seed(SEED)

    device = torch_device()

    fusion_model = MultiBranchFusionModel(enabled_branches).model
    fusion_model = fusion_model.to(device)

    train_labels = []

    for batch in train_loader:
        train_labels.extend(batch["label"].numpy().tolist())

    train_labels = np.asarray(train_labels)

    n_positive = max(1, int((train_labels == 1).sum()))
    n_negative = max(1, int((train_labels == 0).sum()))
    pos_weight_value = n_negative / n_positive

    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

    classification_loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    regression_loss_fn = nn.MSELoss()
    cps_loss_fn = nn.MSELoss()

    optimizer = torch.optim.AdamW(
        fusion_model.parameters(),
        lr=LEARNING_RATE,
    )

    best_state = None
    best_val_loss = np.inf
    patience_counter = 0
    history_rows = []

    for epoch in range(EPOCHS):
        if epoch < FREEZE_EPOCHS:
            fusion_model.set_transformer_trainability(False)
        else:
            fusion_model.set_transformer_trainability(True)

        fusion_model.train()
        epoch_losses = []

        for batch in train_loader:
            optimizer.zero_grad()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            rhetoric = batch["rhetoric"].to(device)
            stance = batch["stance"].to(device)
            elm = batch["elm"].to(device)
            tpb = batch["tpb"].to(device)

            labels = batch["label"].to(device)
            regression_target = batch["regression"].to(device)
            cps_target = batch["cps"].to(device)

            logits, regression_output, cps_output = fusion_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                rhetoric=rhetoric,
                stance=stance,
                elm=elm,
                tpb=tpb,
            )

            loss_cls = classification_loss_fn(logits, labels)
            loss_reg = regression_loss_fn(regression_output, regression_target)
            loss_cps = cps_loss_fn(cps_output, cps_target)

            loss = (
                LAMBDA_CLS * loss_cls
                + LAMBDA_REG * loss_reg
                + LAMBDA_CPS * loss_cps
            )

            loss.backward()
            optimizer.step()

            epoch_losses.append(float(loss.item()))

        val_predictions = collect_predictions(fusion_model, val_loader, device)

        val_cls_loss = classification_loss_fn(
            torch.tensor(val_predictions["y_score"], dtype=torch.float32).to(device),
            torch.tensor(val_predictions["y_true"], dtype=torch.float32).to(device),
        )

        val_reg_loss = regression_loss_fn(
            torch.tensor(val_predictions["reg_pred"], dtype=torch.float32).to(device),
            torch.tensor(val_predictions["reg_true"], dtype=torch.float32).to(device),
        )

        val_cps_loss = cps_loss_fn(
            torch.tensor(val_predictions["cps_pred"], dtype=torch.float32).to(device),
            torch.tensor(val_predictions["cps_true"], dtype=torch.float32).to(device),
        )

        val_total_loss = float(
            LAMBDA_CLS * val_cls_loss.item()
            + LAMBDA_REG * val_reg_loss.item()
            + LAMBDA_CPS * val_cps_loss.item()
        )

        history_rows.append(
            {
                "variant": variant_name,
                "epoch": epoch + 1,
                "train_loss": float(np.mean(epoch_losses)),
                "validation_total_loss": val_total_loss,
            }
        )

        print(
            f"{variant_name} | epoch {epoch + 1}/{EPOCHS} | "
            f"train_loss={np.mean(epoch_losses):.4f} | val_loss={val_total_loss:.4f}"
        )

        if val_total_loss < best_val_loss:
            best_val_loss = val_total_loss
            best_state = copy.deepcopy(fusion_model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= PATIENCE:
            print(f"{variant_name} | early stopping at epoch {epoch + 1}")
            break

    if best_state is not None:
        fusion_model.load_state_dict(best_state)

    test_predictions = collect_predictions(fusion_model, test_loader, device)

    cls = classification_metrics(
        test_predictions["y_true"],
        test_predictions["y_pred"],
        test_predictions["y_score"],
    )

    prop = propagation_metrics(
        test_predictions["reg_true"],
        test_predictions["reg_pred"],
    )

    cps_mae = mean_absolute_error(
        test_predictions["cps_true"],
        test_predictions["cps_pred"],
    )

    result = {
        "variant": variant_name,
        "text_branch": enabled_branches.get("text", True),
        "rhetorical_branch": enabled_branches.get("rhetoric", True),
        "stance_branch": enabled_branches.get("stance", True),
        "psychological_branch": enabled_branches.get("psychological", True),
        **cls,
        **prop,
        "cps_mae": cps_mae,
        "best_validation_loss": best_val_loss,
    }

    prediction_df = pd.DataFrame(
        {
            "variant": variant_name,
            "y_true": test_predictions["y_true"],
            "classification_score": test_predictions["y_score"],
            "classification_prediction": test_predictions["y_pred"],
            "propagation_true": test_predictions["reg_true"],
            "propagation_pred": test_predictions["reg_pred"],
            "cps_true": test_predictions["cps_true"],
            "cps_pred": test_predictions["cps_pred"],
            "engagement_raw": test_predictions["engagement_raw"],
        }
    )

    history_df = pd.DataFrame(history_rows)

    return result, prediction_df, history_df


# ============================================================
# 10. CPS statistics
# ============================================================

def create_cps_statistics(prediction_df, out_dir):
    full_predictions = prediction_df[prediction_df["variant"] == "FullFusion"].copy()

    if len(full_predictions) == 0:
        full_predictions = prediction_df.copy()

    cps_values = full_predictions["cps_pred"].values

    cps_min = float(np.min(cps_values))
    cps_max = float(np.max(cps_values))

    if cps_max > 0:
        log10_cps_max = float(np.log10(cps_max))
    else:
        log10_cps_max = np.nan

    stats = pd.DataFrame(
        [
            {
                "cps_mean": float(np.mean(cps_values)),
                "cps_std": float(np.std(cps_values)),
                "cps_min": cps_min,
                "cps_max": cps_max,
                "log10_cps_max": log10_cps_max,
            }
        ]
    )

    stats.to_csv(out_dir / "cps_statistics.csv", index=False)

    return stats


# ============================================================
# 11. Main pipeline
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--data", type=str, default=DEFAULT_DATA_FILE)
    parser.add_argument("--out", type=str, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--skip_ablation", action="store_true")
    parser.add_argument("--skip_baselines", action="store_true")

    args, _ = parser.parse_known_args()

    set_seed(SEED)

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    data_path = upload_data_if_needed(args.data)

    print(f"\nUsing dataset: {data_path}")

    original_df, metadata = load_dataset(data_path)

    balanced_df, balance_summary = balance_binary_dataset(original_df)

    train_df, val_df, test_df = create_splits(balanced_df)

    dataset_summary, split_summary = save_data_summaries(
        original_df=original_df,
        balanced_df=balanced_df,
        balance_summary=balance_summary,
        split_dfs=(train_df, val_df, test_df),
        metadata=metadata,
        out_dir=out_dir,
    )

    print("\nDataset summary")
    print(dataset_summary.to_string(index=False))

    print("\nSplit summary")
    print(split_summary.to_string(index=False))

    print("\nBalance summary")
    print(json.dumps(balance_summary, indent=2))

    if not args.skip_baselines:
        print("\nRunning baseline classifiers")
        baseline_df = run_baseline_classifiers(train_df, test_df, out_dir)
        print(baseline_df.to_string(index=False))

    print("\nExtracting engineered feature branches")
    feature_sets, feature_scalers = fit_transform_feature_branches(train_df, val_df, test_df)

    targets, target_scalers = create_targets(train_df, val_df, test_df, feature_sets)

    print("\nCreating data loaders")
    train_loader, val_loader, test_loader = create_loaders(
        train_df=train_df,
        val_df=val_df,
        test_df=test_df,
        feature_sets=feature_sets,
        targets=targets,
    )

    variants = {
        "FullFusion": {
            "text": True,
            "rhetoric": True,
            "stance": True,
            "psychological": True,
        },
    }

    if not args.skip_ablation:
        variants.update(
            {
                "NoRhetoricalBranch": {
                    "text": True,
                    "rhetoric": False,
                    "stance": True,
                    "psychological": True,
                },
                "NoStanceBranch": {
                    "text": True,
                    "rhetoric": True,
                    "stance": False,
                    "psychological": True,
                },
                "NoPsychologicalBranch": {
                    "text": True,
                    "rhetoric": True,
                    "stance": True,
                    "psychological": False,
                },
                "NoTextualBranch": {
                    "text": False,
                    "rhetoric": True,
                    "stance": True,
                    "psychological": True,
                },
            }
        )

    result_rows = []
    prediction_frames = []
    history_frames = []

    for variant_name, branch_config in variants.items():
        print(f"\nTraining variant: {variant_name}")

        result, prediction_df, history_df = train_model(
            train_loader=train_loader,
            val_loader=val_loader,
            test_loader=test_loader,
            enabled_branches=branch_config,
            variant_name=variant_name,
        )

        result_rows.append(result)
        prediction_frames.append(prediction_df)
        history_frames.append(history_df)

    results_df = pd.DataFrame(result_rows)
    predictions_df = pd.concat(prediction_frames, ignore_index=True)
    history_df = pd.concat(history_frames, ignore_index=True)

    results_df.to_csv(out_dir / "fusion_model_results.csv", index=False)
    predictions_df.to_csv(out_dir / "test_predictions.csv", index=False)
    history_df.to_csv(out_dir / "training_history.csv", index=False)

    full_row = results_df[results_df["variant"] == "FullFusion"].iloc[0]

    ablation_rows = []

    for _, row in results_df.iterrows():
        ablation_rows.append(
            {
                "variant": row["variant"],
                "f1": row["f1"],
                "roc_auc": row["roc_auc"],
                "f1_drop_from_full": float(full_row["f1"] - row["f1"]),
                "roc_auc_drop_from_full": float(full_row["roc_auc"] - row["roc_auc"]),
                "f1_relative_drop_percent": float(
                    100 * (full_row["f1"] - row["f1"]) / max(full_row["f1"], 1e-9)
                ),
                "roc_auc_relative_drop_percent": float(
                    100 * (full_row["roc_auc"] - row["roc_auc"]) / max(full_row["roc_auc"], 1e-9)
                ),
            }
        )

    ablation_df = pd.DataFrame(ablation_rows)
    ablation_df.to_csv(out_dir / "ablation_results.csv", index=False)

    propagation_df = results_df[
        [
            "variant",
            "mse_propagation",
            "mae_propagation",
            "spearman_rank",
            "top_5_hit_rate",
            "viral_threshold",
            "precision_viral",
            "recall_viral",
            "kl_divergence",
        ]
    ].copy()

    propagation_df.to_csv(out_dir / "propagation_results.csv", index=False)

    cps_stats = create_cps_statistics(predictions_df, out_dir)

    print("\nFusion model results")
    print(results_df.to_string(index=False))

    print("\nAblation results")
    print(ablation_df.to_string(index=False))

    print("\nPropagation results")
    print(propagation_df.to_string(index=False))

    print("\nCPS statistics")
    print(cps_stats.to_string(index=False))

    print("\nFinished. Output files:")

    for file in sorted(out_dir.glob("*")):
        print("  ", file)

    try:
        from IPython.display import display

        selected_outputs = [
            out_dir / "dataset_summary.csv",
            out_dir / "split_summary.csv",
            out_dir / "baseline_classifier_results.csv",
            out_dir / "fusion_model_results.csv",
            out_dir / "ablation_results.csv",
            out_dir / "propagation_results.csv",
            out_dir / "cps_statistics.csv",
        ]

        for output_file in selected_outputs:
            if output_file.exists():
                print("\n" + "=" * 90)
                print(output_file)
                print("=" * 90)
                display(pd.read_csv(output_file))

    except Exception:
        pass


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# ELM-informed hybrid CNN-LSTM health misinformation pipeline
# Single uninterrupted Colab-ready script
# ============================================================

import argparse
import copy
import html
import importlib.util
import json
import os
import random
import re
import shutil
import subprocess
import sys
import warnings
from pathlib import Path

# ============================================================
# 0. Package installation for Colab
# ============================================================

PACKAGE_IMPORTS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scikit-learn": "sklearn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "tensorflow": "tensorflow",
    "textblob": "textblob",
    "nltk": "nltk",
}

for package_name, module_name in PACKAGE_IMPORTS.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            check=True,
        )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
import tensorflow as tf

from scipy.stats import ttest_rel, wilcoxon
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from textblob import TextBlob

from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import (
    Concatenate,
    Conv1D,
    Dense,
    Dropout,
    Embedding,
    Input,
    LSTM,
)
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

warnings.filterwarnings("ignore")

# ============================================================
# 1. General configuration
# ============================================================

SEED = 42
POSITIVE_LABEL = 1

N_SPLITS = 10
VALIDATION_SIZE_WITHIN_FOLD = 0.10

MAX_WORDS = 30000
MAX_SEQUENCE_LENGTH = 128
EMBEDDING_DIM = 100

CNN_FILTERS = 64
CNN_KERNEL_SIZE = 3
LSTM_UNITS = 100
DROPOUT_RATE = 0.50

EPOCHS = 10
BATCH_SIZE = 32
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 2

DEFAULT_FAKE_FILE = "fakeNews.csv"
DEFAULT_TRUE_FILE = "trueNews.csv"
DEFAULT_OUTPUT_DIR = "results_elm_cnn_lstm"

os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"


# ============================================================
# 2. Reproducibility utilities
# ============================================================

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)

    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass


set_seed(SEED)


def download_nltk_resources():
    try:
        import nltk

        resources = [
            ("tokenizers/punkt", "punkt"),
            ("tokenizers/punkt_tab", "punkt_tab"),
            ("taggers/averaged_perceptron_tagger", "averaged_perceptron_tagger"),
            ("taggers/averaged_perceptron_tagger_eng", "averaged_perceptron_tagger_eng"),
        ]

        for resource_path, package_name in resources:
            try:
                nltk.data.find(resource_path)
            except LookupError:
                nltk.download(package_name, quiet=True)

    except Exception:
        pass


download_nltk_resources()


# ============================================================
# 3. Data upload support
# ============================================================

def upload_data_if_needed(fake_file=DEFAULT_FAKE_FILE, true_file=DEFAULT_TRUE_FILE):
    fake_path = Path(fake_file)
    true_path = Path(true_file)

    if fake_path.exists() and true_path.exists():
        return str(fake_path), str(true_path)

    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)

    data_fake = data_dir / fake_path.name
    data_true = data_dir / true_path.name

    if data_fake.exists() and data_true.exists():
        return str(data_fake), str(data_true)

    try:
        from google.colab import files

        print("Upload fakeNews.csv and trueNews.csv")
        uploaded = files.upload()

        for filename in uploaded.keys():
            source = Path(filename)
            destination = data_dir / filename

            if destination.exists():
                destination.unlink()

            shutil.move(str(source), str(destination))

        if data_fake.exists() and data_true.exists():
            return str(data_fake), str(data_true)

        if Path("fakeNews.csv").exists() and Path("trueNews.csv").exists():
            return "fakeNews.csv", "trueNews.csv"

        raise FileNotFoundError("Expected fakeNews.csv and trueNews.csv after upload.")

    except Exception as exc:
        raise FileNotFoundError(
            "Data files not found. Place fakeNews.csv and trueNews.csv in the current "
            "folder or in a data/ folder, or run this in Colab and upload both files."
        ) from exc


# ============================================================
# 4. Text cleaning and feature utilities
# ============================================================

def fix_text_encoding(text):
    if pd.isna(text):
        return ""

    text = str(text)

    try:
        text = text.encode("latin1").decode("utf-8")
    except Exception:
        pass

    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_text(text):
    text = fix_text_encoding(text)
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^A-Za-z0-9!?.,;:'%\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    return re.findall(r"[a-zA-Z][a-zA-Z']+|\d+", str(text).lower())


def sentence_split(text):
    sentences = re.split(r"[.!?]+", str(text))
    sentences = [sentence.strip() for sentence in sentences if sentence.strip()]
    return sentences if len(sentences) > 0 else [str(text)]


def syllable_count_word(word):
    word = str(word).lower()
    word = re.sub(r"[^a-z]", "", word)

    if len(word) == 0:
        return 0

    groups = re.findall(r"[aeiouy]+", word)
    count = len(groups)

    if word.endswith("e") and count > 1:
        count -= 1

    return max(1, count)


def safe_divide(numerator, denominator):
    if denominator == 0:
        return 0.0

    return float(numerator) / float(denominator)


def sentiment_polarity(text):
    try:
        return float(TextBlob(str(text)).sentiment.polarity)
    except Exception:
        return 0.0


URGENCY_TERMS = {
    "now", "urgent", "immediately", "breaking", "alert", "act", "today",
    "before", "warning", "share", "must", "stop",
}

COORDINATING_CONJUNCTIONS = {
    "and", "but", "or", "so", "yet", "for", "nor",
}

SUBORDINATING_CUES = {
    "because", "although", "though", "since", "while", "whereas", "if",
    "unless", "when", "after", "before", "until",
}

RISK_TERMS = {
    "death", "dead", "danger", "dangerous", "risk", "harm", "harmful",
    "unsafe", "panic", "fear", "scared", "threat", "crisis",
}

ATTRIBUTION_TERMS = {
    "according", "source", "reported", "report", "study", "studies",
    "evidence", "research", "experts", "scientists", "cdc", "who", "nhs",
}

DISCOURSE_MARKERS = {
    "however", "therefore", "because", "although", "whereas", "moreover",
    "furthermore", "nevertheless", "thus", "hence", "since",
}


def pos_distribution(tokens):
    try:
        import nltk

        tagged = nltk.pos_tag(tokens)

        if len(tagged) == 0:
            return 0.0, 0.0, 0.0

        noun_count = sum(1 for _, tag in tagged if tag.startswith("NN"))
        verb_count = sum(1 for _, tag in tagged if tag.startswith("VB"))
        adj_adv_count = sum(
            1 for _, tag in tagged
            if tag.startswith("JJ") or tag.startswith("RB")
        )

        total = len(tagged)

        return (
            safe_divide(noun_count, total),
            safe_divide(verb_count, total),
            safe_divide(adj_adv_count, total),
        )

    except Exception:
        return 0.0, 0.0, 0.0


def extract_elm_features(text):
    raw_text = str(text)
    cleaned = clean_text(raw_text)
    tokens = tokenize(cleaned)
    sentences = sentence_split(cleaned)

    token_count = max(1, len(tokens))
    sentence_count = max(1, len(sentences))
    unique_tokens = len(set(tokens))
    syllables = sum(syllable_count_word(token) for token in tokens)

    flesch_kincaid_grade = (
        0.39 * safe_divide(token_count, sentence_count)
        + 11.8 * safe_divide(syllables, token_count)
        - 15.59
    )

    vocabulary_richness = safe_divide(unique_tokens, token_count)
    polarity = sentiment_polarity(cleaned)
    text_length = float(token_count)
    average_words_per_sentence = safe_divide(token_count, sentence_count)

    letters = re.findall(r"[A-Za-z]", raw_text)
    uppercase_letters = re.findall(r"[A-Z]", raw_text)

    exclamation_ratio = safe_divide(raw_text.count("!"), token_count)
    question_ratio = safe_divide(raw_text.count("?"), token_count)
    capitalisation_ratio = safe_divide(len(uppercase_letters), len(letters))
    all_caps_count = float(len(re.findall(r"\b[A-Z]{2,}\b", raw_text)))
    urgency_frequency = safe_divide(
        sum(1 for token in tokens if token in URGENCY_TERMS),
        token_count,
    )

    return [
        flesch_kincaid_grade,
        vocabulary_richness,
        polarity,
        text_length,
        average_words_per_sentence,
        exclamation_ratio,
        question_ratio,
        capitalisation_ratio,
        all_caps_count,
        urgency_frequency,
    ]


def extract_additional_linguistic_features(text):
    raw_text = str(text)
    cleaned = clean_text(raw_text)
    tokens = tokenize(cleaned)
    sentences = sentence_split(cleaned)

    token_count = max(1, len(tokens))
    sentence_count = max(1, len(sentences))

    comma_ratio = safe_divide(raw_text.count(","), token_count)
    semicolon_ratio = safe_divide(raw_text.count(";"), token_count)
    colon_ratio = safe_divide(raw_text.count(":"), token_count)
    quote_ratio = safe_divide(raw_text.count('"') + raw_text.count("'"), token_count)

    simple_count = 0
    compound_count = 0
    complex_count = 0

    for sentence in sentences:
        sentence_tokens = tokenize(sentence)

        has_coordinating = any(token in COORDINATING_CONJUNCTIONS for token in sentence_tokens)
        has_subordinating = any(token in SUBORDINATING_CUES for token in sentence_tokens)

        if has_subordinating:
            complex_count += 1
        elif has_coordinating:
            compound_count += 1
        else:
            simple_count += 1

    simple_sentence_ratio = safe_divide(simple_count, sentence_count)
    compound_sentence_ratio = safe_divide(compound_count, sentence_count)
    complex_sentence_ratio = safe_divide(complex_count, sentence_count)

    noun_ratio, verb_ratio, adjective_adverb_ratio = pos_distribution(tokens)

    risk_term_ratio = safe_divide(sum(1 for token in tokens if token in RISK_TERMS), token_count)
    attribution_ratio = safe_divide(sum(1 for token in tokens if token in ATTRIBUTION_TERMS), token_count)
    discourse_marker_ratio = safe_divide(sum(1 for token in tokens if token in DISCOURSE_MARKERS), token_count)

    return [
        comma_ratio,
        semicolon_ratio,
        colon_ratio,
        quote_ratio,
        simple_sentence_ratio,
        compound_sentence_ratio,
        complex_sentence_ratio,
        noun_ratio,
        verb_ratio,
        adjective_adverb_ratio,
        risk_term_ratio,
        attribution_ratio,
        discourse_marker_ratio,
    ]


def extract_feature_matrices(texts):
    elm_rows = []
    extended_rows = []

    for text in texts:
        elm_rows.append(extract_elm_features(text))
        extended_rows.append(extract_additional_linguistic_features(text))

    elm_features = np.asarray(elm_rows, dtype=np.float32)
    additional_features = np.asarray(extended_rows, dtype=np.float32)

    return elm_features, additional_features


# ============================================================
# 5. Data loading
# ============================================================

def load_covid_fnir_data(fake_path, true_path):
    fake_path = Path(fake_path)
    true_path = Path(true_path)

    if not fake_path.exists():
        raise FileNotFoundError(f"Fake news file not found: {fake_path}")

    if not true_path.exists():
        raise FileNotFoundError(f"True news file not found: {true_path}")

    fake = pd.read_csv(fake_path)
    true = pd.read_csv(true_path)

    if "Text" not in fake.columns:
        raise ValueError("fakeNews.csv must contain a column called Text")

    if "Text" not in true.columns:
        raise ValueError("trueNews.csv must contain a column called Text")

    fake_df = pd.DataFrame(
        {
            "text": fake["Text"].astype(str),
            "label": 1,
            "source_file": fake_path.name,
        }
    )

    true_df = pd.DataFrame(
        {
            "text": true["Text"].astype(str),
            "label": 0,
            "source_file": true_path.name,
        }
    )

    df = pd.concat([fake_df, true_df], ignore_index=True)
    df["text_clean"] = df["text"].map(clean_text)
    df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)
    df["row_id"] = np.arange(len(df))

    return df


def save_dataset_summary(df, fake_path, true_path, out_dir):
    summary = {
        "fake_file": str(fake_path),
        "true_file": str(true_path),
        "n_total": int(len(df)),
        "n_true_label_0": int((df["label"] == 0).sum()),
        "n_fake_label_1": int((df["label"] == 1).sum()),
        "true_percent": round(float((df["label"] == 0).mean() * 100), 2),
        "fake_percent": round(float((df["label"] == 1).mean() * 100), 2),
        "n_exact_duplicate_clean_text": int(df["text_clean"].duplicated().sum()),
        "positive_label": POSITIVE_LABEL,
    }

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(out_dir / "dataset_summary.csv", index=False)

    with open(out_dir / "dataset_summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)

    return summary_df


def save_runtime_versions(out_dir):
    versions = {
        "python": sys.version,
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "scipy": scipy.__version__,
        "tensorflow": tf.__version__,
        "random_seed": SEED,
    }

    with open(out_dir / "runtime_versions.json", "w", encoding="utf-8") as file:
        json.dump(versions, file, indent=2)

    return versions


def save_feature_definitions(out_dir):
    elm_definitions = [
        ("central", "flesch_kincaid_grade", "Readability and complexity proxy"),
        ("central", "vocabulary_richness", "Unique tokens divided by total tokens"),
        ("central", "sentiment_polarity", "TextBlob polarity score"),
        ("central", "text_length", "Number of tokens"),
        ("central", "average_words_per_sentence", "Token count divided by sentence count"),
        ("peripheral", "exclamation_ratio", "Exclamation marks divided by token count"),
        ("peripheral", "question_ratio", "Question marks divided by token count"),
        ("peripheral", "capitalisation_ratio", "Uppercase letters divided by all letters"),
        ("peripheral", "all_caps_count", "Count of all-uppercase words with at least two letters"),
        ("peripheral", "urgency_frequency", "Urgency terms divided by token count"),
    ]

    additional_definitions = [
        ("additional", "comma_ratio", "Commas divided by token count"),
        ("additional", "semicolon_ratio", "Semicolons divided by token count"),
        ("additional", "colon_ratio", "Colons divided by token count"),
        ("additional", "quote_ratio", "Quote characters divided by token count"),
        ("additional", "simple_sentence_ratio", "Proportion of simple sentences"),
        ("additional", "compound_sentence_ratio", "Proportion of compound sentences"),
        ("additional", "complex_sentence_ratio", "Proportion of complex sentences"),
        ("additional", "noun_ratio", "Noun POS tags divided by all POS tags"),
        ("additional", "verb_ratio", "Verb POS tags divided by all POS tags"),
        ("additional", "adjective_adverb_ratio", "Adjective and adverb POS tags divided by all POS tags"),
        ("additional", "risk_term_ratio", "Risk-related terms divided by token count"),
        ("additional", "attribution_ratio", "Attribution terms divided by token count"),
        ("additional", "discourse_marker_ratio", "Discourse markers divided by token count"),
    ]

    rows = []

    for group, name, definition in elm_definitions + additional_definitions:
        rows.append(
            {
                "feature_group": group,
                "feature_name": name,
                "definition": definition,
            }
        )

    feature_df = pd.DataFrame(rows)
    feature_df.to_csv(out_dir / "feature_definitions.csv", index=False)

    return feature_df


# ============================================================
# 6. Model builders
# ============================================================

def build_text_branch(sequence_input, vocab_size):
    x = Embedding(
        input_dim=vocab_size,
        output_dim=EMBEDDING_DIM,
        input_length=MAX_SEQUENCE_LENGTH,
        name="embedding",
    )(sequence_input)

    x = Conv1D(
        filters=CNN_FILTERS,
        kernel_size=CNN_KERNEL_SIZE,
        activation="relu",
        padding="same",
        name="conv1d",
    )(x)

    x = Dropout(DROPOUT_RATE, name="cnn_dropout")(x)

    x = LSTM(
        LSTM_UNITS,
        name="lstm",
    )(x)

    return x


def compile_model(model):
    optimizer = tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE)

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name="accuracy"),
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.AUC(name="auc"),
        ],
    )

    return model


def build_base_model(vocab_size):
    sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,), name="sequence_input")
    text_vector = build_text_branch(sequence_input, vocab_size)
    output = Dense(1, activation="sigmoid", name="classification_output")(text_vector)

    model = Model(inputs=sequence_input, outputs=output, name="Base_CNN_LSTM")

    return compile_model(model)


def build_elm_only_model(elm_dim):
    elm_input = Input(shape=(elm_dim,), name="elm_input")

    x = Dense(64, activation="relu", name="elm_dense_1")(elm_input)
    x = Dropout(0.30, name="elm_dropout_1")(x)
    x = Dense(32, activation="relu", name="elm_dense_2")(x)
    output = Dense(1, activation="sigmoid", name="classification_output")(x)

    model = Model(inputs=elm_input, outputs=output, name="ELM_Only_Model")

    return compile_model(model)


def build_enhanced_model(vocab_size, elm_dim):
    sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,), name="sequence_input")
    elm_input = Input(shape=(elm_dim,), name="elm_input")

    text_vector = build_text_branch(sequence_input, vocab_size)

    fused = Concatenate(name="text_elm_concatenation")([text_vector, elm_input])
    x = Dense(64, activation="relu", name="fusion_dense")(fused)
    x = Dropout(0.30, name="fusion_dropout")(x)
    output = Dense(1, activation="sigmoid", name="classification_output")(x)

    model = Model(
        inputs=[sequence_input, elm_input],
        outputs=output,
        name="Enhanced_CNN_LSTM_ELM",
    )

    return compile_model(model)


def build_extended_model(vocab_size, extended_dim):
    sequence_input = Input(shape=(MAX_SEQUENCE_LENGTH,), name="sequence_input")
    extended_input = Input(shape=(extended_dim,), name="extended_feature_input")

    text_vector = build_text_branch(sequence_input, vocab_size)

    fused = Concatenate(name="text_extended_concatenation")([text_vector, extended_input])
    x = Dense(96, activation="relu", name="extended_fusion_dense_1")(fused)
    x = Dropout(0.30, name="extended_fusion_dropout_1")(x)
    x = Dense(48, activation="relu", name="extended_fusion_dense_2")(x)
    output = Dense(1, activation="sigmoid", name="classification_output")(x)

    model = Model(
        inputs=[sequence_input, extended_input],
        outputs=output,
        name="Extended_CNN_LSTM_Features",
    )

    return compile_model(model)


# ============================================================
# 7. Metrics and evaluation
# ============================================================

def compute_classification_metrics(y_true, y_score, threshold=0.50):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "recall": recall_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "f1": f1_score(
            y_true,
            y_pred,
            pos_label=POSITIVE_LABEL,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(y_true, y_score),
    }

    return metrics, y_pred


def prediction_input_for_variant(variant_name, X_seq, X_elm, X_extended):
    if variant_name == "Base":
        return X_seq

    if variant_name == "ELM_Only":
        return X_elm

    if variant_name == "Enhanced_Text_ELM":
        return [X_seq, X_elm]

    if variant_name == "Extended_Text_ELM_Additional":
        return [X_seq, X_extended]

    raise ValueError(f"Unknown model variant: {variant_name}")


def build_model_for_variant(variant_name, vocab_size, elm_dim, extended_dim):
    if variant_name == "Base":
        return build_base_model(vocab_size)

    if variant_name == "ELM_Only":
        return build_elm_only_model(elm_dim)

    if variant_name == "Enhanced_Text_ELM":
        return build_enhanced_model(vocab_size, elm_dim)

    if variant_name == "Extended_Text_ELM_Additional":
        return build_extended_model(vocab_size, extended_dim)

    raise ValueError(f"Unknown model variant: {variant_name}")


# ============================================================
# 8. Cross-validation training
# ============================================================

def run_cross_validation(df, out_dir):
    texts = df["text_clean"].values
    labels = df["label"].values.astype(int)
    row_ids = df["row_id"].values.astype(int)

    raw_elm_features, raw_additional_features = extract_feature_matrices(texts)
    raw_extended_features = np.concatenate(
        [raw_elm_features, raw_additional_features],
        axis=1,
    )

    n_class_0 = int((labels == 0).sum())
    n_class_1 = int((labels == 1).sum())
    min_class_size = min(n_class_0, n_class_1)
    n_splits = min(N_SPLITS, min_class_size)

    if n_splits < 2:
        raise ValueError("At least two examples are required in each class for cross-validation.")

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=SEED,
    )

    variants = [
        "Base",
        "ELM_Only",
        "Enhanced_Text_ELM",
        "Extended_Text_ELM_Additional",
    ]

    fold_rows = []
    prediction_rows = []
    fold_assignment_rows = []

    for fold_number, (train_val_index, test_index) in enumerate(skf.split(texts, labels), start=1):
        print(f"\nFold {fold_number}/{n_splits}")

        train_index, validation_index = train_test_split(
            train_val_index,
            test_size=VALIDATION_SIZE_WITHIN_FOLD,
            random_state=SEED + fold_number,
            stratify=labels[train_val_index],
        )

        for split_name, indices in [
            ("train", train_index),
            ("validation", validation_index),
            ("test", test_index),
        ]:
            for index_value in indices:
                fold_assignment_rows.append(
                    {
                        "fold": fold_number,
                        "split": split_name,
                        "row_id": int(row_ids[index_value]),
                        "label": int(labels[index_value]),
                    }
                )

        tokenizer = Tokenizer(
            num_words=MAX_WORDS,
            oov_token="<OOV>",
        )

        tokenizer.fit_on_texts(texts[train_index])

        vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)

        X_train_seq = pad_sequences(
            tokenizer.texts_to_sequences(texts[train_index]),
            maxlen=MAX_SEQUENCE_LENGTH,
            padding="post",
            truncating="post",
        )

        X_val_seq = pad_sequences(
            tokenizer.texts_to_sequences(texts[validation_index]),
            maxlen=MAX_SEQUENCE_LENGTH,
            padding="post",
            truncating="post",
        )

        X_test_seq = pad_sequences(
            tokenizer.texts_to_sequences(texts[test_index]),
            maxlen=MAX_SEQUENCE_LENGTH,
            padding="post",
            truncating="post",
        )

        elm_scaler = StandardScaler()
        extended_scaler = StandardScaler()

        X_train_elm = elm_scaler.fit_transform(raw_elm_features[train_index]).astype(np.float32)
        X_val_elm = elm_scaler.transform(raw_elm_features[validation_index]).astype(np.float32)
        X_test_elm = elm_scaler.transform(raw_elm_features[test_index]).astype(np.float32)

        X_train_extended = extended_scaler.fit_transform(raw_extended_features[train_index]).astype(np.float32)
        X_val_extended = extended_scaler.transform(raw_extended_features[validation_index]).astype(np.float32)
        X_test_extended = extended_scaler.transform(raw_extended_features[test_index]).astype(np.float32)

        y_train = labels[train_index]
        y_val = labels[validation_index]
        y_test = labels[test_index]

        elm_dim = X_train_elm.shape[1]
        extended_dim = X_train_extended.shape[1]

        for variant_name in variants:
            print(f"  Training {variant_name}")

            tf.keras.backend.clear_session()
            set_seed(SEED + fold_number)

            model = build_model_for_variant(
                variant_name=variant_name,
                vocab_size=vocab_size,
                elm_dim=elm_dim,
                extended_dim=extended_dim,
            )

            train_input = prediction_input_for_variant(
                variant_name,
                X_train_seq,
                X_train_elm,
                X_train_extended,
            )

            validation_input = prediction_input_for_variant(
                variant_name,
                X_val_seq,
                X_val_elm,
                X_val_extended,
            )

            test_input = prediction_input_for_variant(
                variant_name,
                X_test_seq,
                X_test_elm,
                X_test_extended,
            )

            early_stopping = EarlyStopping(
                monitor="val_loss",
                patience=EARLY_STOPPING_PATIENCE,
                restore_best_weights=True,
                verbose=0,
            )

            history = model.fit(
                train_input,
                y_train,
                validation_data=(validation_input, y_val),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                callbacks=[early_stopping],
                verbose=0,
            )

            y_score = model.predict(
                test_input,
                batch_size=BATCH_SIZE,
                verbose=0,
            ).reshape(-1)

            metrics, y_pred = compute_classification_metrics(y_test, y_score)

            fold_row = {
                "fold": fold_number,
                "variant": variant_name,
                "epochs_trained": len(history.history.get("loss", [])),
                "vocab_size": int(vocab_size),
                **metrics,
            }

            fold_rows.append(fold_row)

            for local_position, dataset_index in enumerate(test_index):
                prediction_rows.append(
                    {
                        "fold": fold_number,
                        "variant": variant_name,
                        "row_id": int(row_ids[dataset_index]),
                        "y_true": int(y_test[local_position]),
                        "y_score": float(y_score[local_position]),
                        "y_pred": int(y_pred[local_position]),
                    }
                )

            print(
                f"    Acc={metrics['accuracy']:.4f}, "
                f"Prec={metrics['precision']:.4f}, "
                f"Rec={metrics['recall']:.4f}, "
                f"F1={metrics['f1']:.4f}, "
                f"ROC-AUC={metrics['roc_auc']:.4f}"
            )

    fold_results = pd.DataFrame(fold_rows)
    predictions = pd.DataFrame(prediction_rows)
    fold_assignments = pd.DataFrame(fold_assignment_rows)

    fold_results.to_csv(out_dir / "fold_level_results.csv", index=False)
    predictions.to_csv(out_dir / "out_of_fold_predictions.csv", index=False)
    fold_assignments.to_csv(out_dir / "fold_assignments.csv", index=False)

    return fold_results, predictions, fold_assignments


# ============================================================
# 9. Summaries, statistical tests and plots
# ============================================================

def create_aggregate_results(fold_results, out_dir):
    metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc"]

    mean_df = (
        fold_results
        .groupby("variant")[metric_columns]
        .mean()
        .reset_index()
    )

    std_df = (
        fold_results
        .groupby("variant")[metric_columns]
        .std()
        .reset_index()
    )

    mean_df = mean_df.rename(
        columns={column: f"{column}_mean" for column in metric_columns}
    )

    std_df = std_df.rename(
        columns={column: f"{column}_std" for column in metric_columns}
    )

    aggregate = mean_df.merge(std_df, on="variant", how="left")

    aggregate_percent = aggregate.copy()

    for column in aggregate_percent.columns:
        if column != "variant":
            aggregate_percent[column] = (aggregate_percent[column] * 100).round(2)

    aggregate.to_csv(out_dir / "aggregate_metrics_raw.csv", index=False)
    aggregate_percent.to_csv(out_dir / "aggregate_metrics_percent.csv", index=False)

    return aggregate, aggregate_percent


def create_improvement_summary(aggregate, out_dir):
    metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc"]

    base_row = aggregate[aggregate["variant"] == "Base"]

    if len(base_row) == 0:
        return pd.DataFrame()

    base_row = base_row.iloc[0]

    rows = []

    for _, row in aggregate.iterrows():
        if row["variant"] == "Base":
            continue

        improvement_row = {
            "comparison": f"{row['variant']} minus Base",
        }

        for metric in metric_columns:
            mean_column = f"{metric}_mean"
            improvement_row[f"{metric}_difference"] = float(row[mean_column] - base_row[mean_column])
            improvement_row[f"{metric}_difference_percentage_points"] = float(
                100 * (row[mean_column] - base_row[mean_column])
            )

        rows.append(improvement_row)

    improvement_df = pd.DataFrame(rows)
    improvement_df.to_csv(out_dir / "performance_improvement_vs_base.csv", index=False)

    return improvement_df


def create_statistical_tests(fold_results, out_dir):
    metric_columns = ["accuracy", "precision", "recall", "f1", "roc_auc"]

    comparisons = [
        ("Enhanced_Text_ELM", "Base"),
        ("Extended_Text_ELM_Additional", "Base"),
        ("Enhanced_Text_ELM", "ELM_Only"),
    ]

    rows = []

    for left_variant, right_variant in comparisons:
        left = fold_results[fold_results["variant"] == left_variant].sort_values("fold")
        right = fold_results[fold_results["variant"] == right_variant].sort_values("fold")

        if len(left) == 0 or len(right) == 0:
            continue

        for metric in metric_columns:
            left_scores = left[metric].values
            right_scores = right[metric].values
            differences = left_scores - right_scores

            try:
                wilcoxon_stat, wilcoxon_p = wilcoxon(
                    left_scores,
                    right_scores,
                    alternative="greater",
                    zero_method="wilcox",
                )
            except Exception:
                wilcoxon_stat, wilcoxon_p = np.nan, np.nan

            try:
                t_stat, t_p = ttest_rel(
                    left_scores,
                    right_scores,
                    alternative="greater",
                )
            except TypeError:
                t_stat_two_sided, t_p_two_sided = ttest_rel(left_scores, right_scores)
                t_stat = t_stat_two_sided

                if np.nanmean(differences) > 0:
                    t_p = t_p_two_sided / 2
                else:
                    t_p = 1 - (t_p_two_sided / 2)

            rows.append(
                {
                    "comparison": f"{left_variant} > {right_variant}",
                    "metric": metric,
                    "mean_left": float(np.nanmean(left_scores)),
                    "mean_right": float(np.nanmean(right_scores)),
                    "mean_difference": float(np.nanmean(differences)),
                    "wilcoxon_statistic": float(wilcoxon_stat) if not np.isnan(wilcoxon_stat) else np.nan,
                    "wilcoxon_p_value_one_sided": float(wilcoxon_p) if not np.isnan(wilcoxon_p) else np.nan,
                    "paired_t_statistic": float(t_stat) if not np.isnan(t_stat) else np.nan,
                    "paired_t_p_value_one_sided": float(t_p) if not np.isnan(t_p) else np.nan,
                    "n_folds": int(len(left_scores)),
                }
            )

    tests_df = pd.DataFrame(rows)
    tests_df.to_csv(out_dir / "statistical_tests.csv", index=False)

    return tests_df


def save_confusion_matrices(predictions, out_dir):
    variants = sorted(predictions["variant"].unique())

    for variant in variants:
        subset = predictions[predictions["variant"] == variant]
        cm = confusion_matrix(
            subset["y_true"],
            subset["y_pred"],
            labels=[0, 1],
        )

        cm_df = pd.DataFrame(
            cm,
            index=["actual_true_0", "actual_fake_1"],
            columns=["predicted_true_0", "predicted_fake_1"],
        )

        safe_name = variant.lower().replace("+", "_").replace(" ", "_")
        cm_df.to_csv(out_dir / f"confusion_matrix_{safe_name}.csv")

        plt.figure(figsize=(5, 4))
        plt.imshow(cm, interpolation="nearest")
        plt.title(f"Confusion Matrix: {variant}")
        plt.xticks([0, 1], ["Pred 0", "Pred 1"])
        plt.yticks([0, 1], ["Actual 0", "Actual 1"])
        plt.xlabel("Predicted label")
        plt.ylabel("True label")

        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                plt.text(j, i, str(cm[i, j]), ha="center", va="center")

        plt.tight_layout()
        plt.savefig(out_dir / f"confusion_matrix_{safe_name}.png", dpi=300)
        plt.close()


def save_roc_curve_plot(predictions, out_dir):
    plt.figure(figsize=(8, 6))

    auc_rows = []

    for variant in sorted(predictions["variant"].unique()):
        subset = predictions[predictions["variant"] == variant]

        fpr, tpr, _ = roc_curve(
            subset["y_true"].values,
            subset["y_score"].values,
            pos_label=POSITIVE_LABEL,
        )

        auc_value = roc_auc_score(
            subset["y_true"].values,
            subset["y_score"].values,
        )

        auc_rows.append(
            {
                "variant": variant,
                "roc_auc_out_of_fold": auc_value,
            }
        )

        plt.plot(fpr, tpr, label=f"{variant} AUC={auc_value:.4f}")

    plt.plot([0, 1], [0, 1], linestyle="--", label="Chance")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("Out-of-fold ROC Curves")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(out_dir / "roc_curves_all_variants.png", dpi=300)
    plt.close()

    auc_df = pd.DataFrame(auc_rows)
    auc_df.to_csv(out_dir / "out_of_fold_auc_summary.csv", index=False)

    return auc_df


def save_performance_plot(aggregate_percent, out_dir):
    metric_columns = [
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "roc_auc_mean",
    ]

    plot_df = aggregate_percent[["variant"] + metric_columns].copy()
    plot_df = plot_df.set_index("variant")

    plt.figure(figsize=(10, 6))

    x = np.arange(len(metric_columns))
    width = 0.18

    variants = list(plot_df.index)

    for idx, variant in enumerate(variants):
        values = plot_df.loc[variant, metric_columns].values.astype(float)
        plt.bar(x + idx * width, values, width=width, label=variant)

    plt.xticks(
        x + width * (len(variants) - 1) / 2,
        ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"],
        rotation=30,
        ha="right",
    )

    plt.ylabel("Mean score (%)")
    plt.title("Mean Cross-Validation Performance by Model Variant")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / "mean_performance_by_variant.png", dpi=300)
    plt.close()


# ============================================================
# 10. Main pipeline
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--fake", type=str, default=DEFAULT_FAKE_FILE)
    parser.add_argument("--true", type=str, default=DEFAULT_TRUE_FILE)
    parser.add_argument("--out", type=str, default=DEFAULT_OUTPUT_DIR)

    args, _ = parser.parse_known_args()

    set_seed(SEED)

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    fake_path, true_path = upload_data_if_needed(args.fake, args.true)

    print(f"\nUsing fake file: {fake_path}")
    print(f"Using true file: {true_path}")

    df = load_covid_fnir_data(fake_path, true_path)

    dataset_summary = save_dataset_summary(df, fake_path, true_path, out_dir)
    runtime_versions = save_runtime_versions(out_dir)
    feature_definitions = save_feature_definitions(out_dir)

    print("\nDataset summary")
    print(dataset_summary.to_string(index=False))

    print("\nRuntime versions")
    print(json.dumps(runtime_versions, indent=2)[:1000])

    print("\nRunning stratified cross-validation")
    fold_results, predictions, fold_assignments = run_cross_validation(df, out_dir)

    aggregate, aggregate_percent = create_aggregate_results(fold_results, out_dir)
    improvement_df = create_improvement_summary(aggregate, out_dir)
    tests_df = create_statistical_tests(fold_results, out_dir)

    save_confusion_matrices(predictions, out_dir)
    auc_df = save_roc_curve_plot(predictions, out_dir)
    save_performance_plot(aggregate_percent, out_dir)

    print("\nAggregate metrics, raw")
    print(aggregate.to_string(index=False))

    print("\nAggregate metrics, percent")
    print(aggregate_percent.to_string(index=False))

    print("\nPerformance improvement versus base")
    print(improvement_df.to_string(index=False))

    print("\nStatistical tests")
    print(tests_df.to_string(index=False))

    print("\nOutput files")
    for file in sorted(out_dir.glob("*")):
        print(" ", file)

    try:
        from IPython.display import display

        selected_outputs = [
            out_dir / "dataset_summary.csv",
            out_dir / "aggregate_metrics_percent.csv",
            out_dir / "performance_improvement_vs_base.csv",
            out_dir / "statistical_tests.csv",
            out_dir / "out_of_fold_auc_summary.csv",
            out_dir / "feature_definitions.csv",
        ]

        for output_file in selected_outputs:
            if output_file.exists():
                print("\n" + "=" * 90)
                print(output_file)
                print("=" * 90)
                display(pd.read_csv(output_file))

    except Exception:
        pass


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# Health misinformation propagation centrality pipeline
# Single uninterrupted Colab-ready script
# ============================================================

import argparse
import html
import json
import os
import random
import re
import shutil
import subprocess
import sys
import warnings
from collections import Counter
from pathlib import Path

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas",
        "numpy",
        "networkx",
        "scipy",
        "nltk",
        "matplotlib",
    ],
    check=True,
)

import networkx as nx
import numpy as np
import pandas as pd

from scipy.stats import spearmanr

warnings.filterwarnings("ignore")


# ============================================================
# 1. General configuration
# ============================================================

SEED = 42
TOP_K = 10
DAMPING_FACTOR = 0.85
MVC_ITERATIONS = 10
DIC_ITERATIONS = 10
BETWEENNESS_SAMPLE_SIZE = 500
MAX_CLOSENESS_CANDIDATES = 5000

DEFAULT_DISCUSSION_FILE = "discussion_post.csv"
DEFAULT_OUTPUT_DIR = "results_centrality"


# ============================================================
# 2. Data upload support
# ============================================================

def upload_data_if_needed(default_file=DEFAULT_DISCUSSION_FILE):
    default_path = Path(default_file)

    if default_path.exists():
        return str(default_path)

    data_dir = Path("data")
    data_dir.mkdir(parents=True, exist_ok=True)

    data_path = data_dir / default_path.name

    if data_path.exists():
        return str(data_path)

    try:
        from google.colab import files

        print("Upload the Monant discussion post CSV file.")
        uploaded = files.upload()

        for filename in uploaded.keys():
            source = Path(filename)
            destination = data_dir / filename

            if destination.exists():
                destination.unlink()

            shutil.move(str(source), str(destination))

        selected = find_discussion_file(data_dir)

        if selected is not None:
            return str(selected)

        selected = find_discussion_file(Path("."))

        if selected is not None:
            return str(selected)

        raise FileNotFoundError("No suitable discussion post CSV file was detected.")

    except Exception as exc:
        raise FileNotFoundError(
            "Discussion post data not found. Place the Monant discussion post CSV file "
            "in the current folder or in a data/ folder, then rerun the script."
        ) from exc


def find_discussion_file(folder):
    folder = Path(folder)

    if not folder.exists():
        return None

    csv_files = sorted(folder.glob("*.csv"))

    if len(csv_files) == 0:
        return None

    scored_files = []

    for file in csv_files:
        try:
            sample = pd.read_csv(file, nrows=5, low_memory=False)
            columns = {str(col).lower().strip() for col in sample.columns}

            score = 0

            if "author_id" in columns:
                score += 5

            if "parent_post_id" in columns:
                score += 5

            if "body" in columns or "body_raw" in columns:
                score += 5

            if "published_at" in columns:
                score += 2

            if "id" in columns:
                score += 2

            if "discussion" in file.name.lower() or "monant" in file.name.lower():
                score += 3

            scored_files.append((score, file))

        except Exception:
            continue

    if len(scored_files) == 0:
        return None

    scored_files = sorted(scored_files, key=lambda x: x[0], reverse=True)

    if scored_files[0][0] <= 0:
        return None

    return scored_files[0][1]


# ============================================================
# 3. Reproducibility and text utilities
# ============================================================

def set_seed(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)


def normalise_identifier(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if value == "" or value.lower() in {"nan", "none", "null"}:
        return None

    if re.fullmatch(r"\d+\.0", value):
        value = value[:-2]

    return value


def fix_text_encoding(text):
    if pd.isna(text):
        return ""

    text = str(text)

    try:
        text = text.encode("latin1").decode("utf-8")
    except Exception:
        pass

    text = html.unescape(text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = text.replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()

    return text


def clean_text(text):
    text = fix_text_encoding(text)
    text = re.sub(r"http\S+|www\.\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"#", "", text)
    text = re.sub(r"[^A-Za-z0-9!?.,;:'%\- ]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize(text):
    return re.findall(r"[a-zA-Z][a-zA-Z']+|\d+", str(text).lower())


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").fillna(0)


def minmax_scale_dict(score_dict):
    if len(score_dict) == 0:
        return {}

    values = np.array(list(score_dict.values()), dtype=float)

    min_value = np.nanmin(values)
    max_value = np.nanmax(values)

    if np.isnan(min_value) or np.isnan(max_value) or max_value == min_value:
        return {key: 0.0 for key in score_dict.keys()}

    return {
        key: float((value - min_value) / (max_value - min_value))
        for key, value in score_dict.items()
    }


def top_nodes(score_dict, metric_name, top_k=TOP_K):
    rows = []

    for rank, (node, score) in enumerate(
        sorted(score_dict.items(), key=lambda item: item[1], reverse=True)[:top_k],
        start=1,
    ):
        rows.append(
            {
                "metric": metric_name,
                "rank": rank,
                "node": node,
                "score": score,
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# 4. Data loading
# ============================================================

def load_discussion_posts(path):
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Discussion post file not found: {path}")

    df = pd.read_csv(path, low_memory=False)

    lower_to_original = {
        str(column).lower().strip(): column
        for column in df.columns
    }

    if "author_id" not in lower_to_original:
        raise ValueError("The discussion post file must contain an author_id column.")

    if "id" not in lower_to_original:
        raise ValueError("The discussion post file must contain an id column.")

    body_column = None

    for candidate in ["body", "body_raw", "text", "content", "title"]:
        if candidate in lower_to_original:
            body_column = lower_to_original[candidate]
            break

    if body_column is None:
        raise ValueError(
            "The discussion post file must contain a body, body_raw, text, content, or title column."
        )

    parent_column = lower_to_original.get("parent_post_id", None)
    published_column = lower_to_original.get("published_at", None)
    up_votes_column = lower_to_original.get("up_votes", None)
    down_votes_column = lower_to_original.get("down_votes", None)

    df = df.copy()

    df["post_id_norm"] = df[lower_to_original["id"]].map(normalise_identifier)
    df["author_id_norm"] = df[lower_to_original["author_id"]].map(normalise_identifier)

    if parent_column is not None:
        df["parent_post_id_norm"] = df[parent_column].map(normalise_identifier)
    else:
        df["parent_post_id_norm"] = None

    df["text"] = df[body_column].map(fix_text_encoding)
    df["text_clean"] = df["text"].map(clean_text)

    if published_column is not None:
        df["published_at_parsed"] = pd.to_datetime(
            df[published_column],
            errors="coerce",
            utc=True,
        )
    else:
        df["published_at_parsed"] = pd.NaT

    if up_votes_column is not None:
        df["up_votes_numeric"] = safe_numeric(df[up_votes_column])
    else:
        df["up_votes_numeric"] = 0

    if down_votes_column is not None:
        df["down_votes_numeric"] = safe_numeric(df[down_votes_column])
    else:
        df["down_votes_numeric"] = 0

    df["engagement_proxy"] = (
        df["up_votes_numeric"] - df["down_votes_numeric"]
    ).clip(lower=0)

    df = df[
        df["post_id_norm"].notna()
        & df["author_id_norm"].notna()
        & (df["text_clean"].str.len() > 0)
    ].reset_index(drop=True)

    return df


def save_dataset_summary(df, out_dir):
    out_dir = Path(out_dir)

    summary = {
        "n_posts": int(len(df)),
        "n_unique_authors": int(df["author_id_norm"].nunique()),
        "n_posts_with_parent_id": int(df["parent_post_id_norm"].notna().sum()),
        "n_posts_without_parent_id": int(df["parent_post_id_norm"].isna().sum()),
        "n_exact_duplicate_clean_text": int(df["text_clean"].duplicated().sum()),
        "date_min": str(df["published_at_parsed"].min()),
        "date_max": str(df["published_at_parsed"].max()),
        "engagement_proxy_sum": float(df["engagement_proxy"].sum()),
        "engagement_proxy_mean": float(df["engagement_proxy"].mean()),
    }

    summary_df = pd.DataFrame([summary])

    summary_df.to_csv(out_dir / "dataset_summary.csv", index=False)

    with open(out_dir / "dataset_summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)

    return summary_df


# ============================================================
# 5. Graph construction
# ============================================================

def build_reply_graph(df):
    graph = nx.DiGraph()

    all_authors = sorted(df["author_id_norm"].dropna().unique())

    graph.add_nodes_from(all_authors)

    post_to_author = dict(
        zip(df["post_id_norm"], df["author_id_norm"])
    )

    edge_counter = Counter()

    for _, row in df.iterrows():
        source_author = row["author_id_norm"]
        parent_post_id = row["parent_post_id_norm"]

        if source_author is None or parent_post_id is None:
            continue

        parent_author = post_to_author.get(parent_post_id)

        if parent_author is None:
            continue

        if source_author == parent_author:
            continue

        edge = (source_author, parent_author)
        edge_counter[edge] += 1

    for (source, target), weight in edge_counter.items():
        graph.add_edge(source, target, weight=float(weight))

    return graph


def save_graph_summary(graph, out_dir):
    out_dir = Path(out_dir)

    n_nodes = graph.number_of_nodes()
    n_edges = graph.number_of_edges()

    weak_components = list(nx.weakly_connected_components(graph))

    if len(weak_components) > 0:
        largest_component_size = max(len(component) for component in weak_components)
    else:
        largest_component_size = 0

    total_edge_weight = sum(
        data.get("weight", 1.0)
        for _, _, data in graph.edges(data=True)
    )

    summary = {
        "n_nodes": int(n_nodes),
        "n_edges": int(n_edges),
        "total_edge_weight": float(total_edge_weight),
        "density": float(nx.density(graph)) if n_nodes > 1 else 0.0,
        "n_weak_components": int(len(weak_components)),
        "largest_weak_component_size": int(largest_component_size),
        "edge_direction": "reply_author_to_parent_author",
    }

    summary_df = pd.DataFrame([summary])
    summary_df.to_csv(out_dir / "graph_summary.csv", index=False)

    with open(out_dir / "graph_summary.json", "w", encoding="utf-8") as file:
        json.dump(summary, file, indent=2)

    return summary_df


# ============================================================
# 6. Traditional centrality metrics
# ============================================================

def compute_degree_scores(graph):
    in_degree = dict(graph.in_degree(weight="weight"))
    out_degree = dict(graph.out_degree(weight="weight"))

    total_degree = {
        node: float(in_degree.get(node, 0.0) + out_degree.get(node, 0.0))
        for node in graph.nodes()
    }

    return in_degree, out_degree, total_degree


def compute_pagerank_score(graph):
    if graph.number_of_edges() == 0:
        return {node: 0.0 for node in graph.nodes()}

    try:
        return nx.pagerank(
            graph,
            alpha=DAMPING_FACTOR,
            weight="weight",
            max_iter=200,
            tol=1e-08,
        )
    except Exception:
        return {node: 0.0 for node in graph.nodes()}


def compute_eigenvector_score(graph, fallback_score):
    if graph.number_of_edges() == 0:
        return {node: 0.0 for node in graph.nodes()}

    if graph.number_of_nodes() > 50000:
        return fallback_score.copy()

    try:
        return nx.eigenvector_centrality(
            graph,
            max_iter=500,
            tol=1e-06,
            weight="weight",
        )
    except Exception:
        return fallback_score.copy()


def compute_betweenness_score(graph):
    if graph.number_of_edges() == 0:
        return {node: 0.0 for node in graph.nodes()}

    n_nodes = graph.number_of_nodes()

    if n_nodes <= 2000:
        try:
            return nx.betweenness_centrality(
                graph,
                normalized=True,
                weight=None,
            )
        except Exception:
            return {node: 0.0 for node in graph.nodes()}

    sample_size = min(BETWEENNESS_SAMPLE_SIZE, n_nodes)

    try:
        return nx.betweenness_centrality(
            graph,
            k=sample_size,
            normalized=True,
            weight=None,
            seed=SEED,
        )
    except Exception:
        return {node: 0.0 for node in graph.nodes()}


def incoming_harmonic_for_node(graph, node):
    reverse_graph = graph.reverse(copy=False)
    lengths = nx.single_source_shortest_path_length(reverse_graph, node)

    score = 0.0

    for target, distance in lengths.items():
        if target == node:
            continue

        if distance > 0:
            score += 1.0 / distance

    return score


def compute_closeness_score(graph, candidate_base_score):
    if graph.number_of_edges() == 0:
        return {node: 0.0 for node in graph.nodes()}

    n_nodes = graph.number_of_nodes()

    if n_nodes <= MAX_CLOSENESS_CANDIDATES:
        candidates = list(graph.nodes())
    else:
        candidates = [
            node for node, _ in sorted(
                candidate_base_score.items(),
                key=lambda item: item[1],
                reverse=True,
            )[:MAX_CLOSENESS_CANDIDATES]
        ]

    scores = {node: 0.0 for node in graph.nodes()}

    for index, node in enumerate(candidates, start=1):
        if index % 500 == 0:
            print(f"  Closeness approximation progress: {index}/{len(candidates)}")

        try:
            scores[node] = incoming_harmonic_for_node(graph, node)
        except Exception:
            scores[node] = 0.0

    return scores


# ============================================================
# 7. Advanced centrality metrics
# ============================================================

def compute_propagation_centrality(graph):
    return compute_pagerank_score(graph)


def create_emotion_lexicon():
    fear_words = {
        "afraid", "alarm", "alarming", "anxiety", "anxious", "danger", "dangerous",
        "dead", "death", "die", "dying", "fear", "fearful", "frightened", "harm",
        "harmful", "panic", "risk", "scared", "terrified", "threat", "unsafe",
        "worried", "worry",
    }

    outrage_words = {
        "angry", "betrayal", "corrupt", "corruption", "criminal", "disgusting",
        "evil", "fraud", "greed", "greedy", "hate", "horrific", "idiots",
        "immoral", "injustice", "lie", "lies", "outrage", "shame", "sickens",
        "unethical",
    }

    conspiracy_words = {
        "agenda", "cartel", "coverup", "cover-up", "deepstate", "elite",
        "globalist", "hoax", "plandemic", "propaganda", "rigged", "scheme",
        "secret", "setup", "they", "truth", "wake", "weapon", "conspiracy",
    }

    return fear_words | outrage_words | conspiracy_words


def aggregate_author_features(df):
    emotion_lexicon = create_emotion_lexicon()

    rows = []

    for author_id, group in df.groupby("author_id_norm"):
        texts = group["text_clean"].fillna("").astype(str).tolist()
        tokens = []

        for text in texts:
            tokens.extend(tokenize(text))

        token_count = len(tokens)
        emotion_count = sum(1 for token in tokens if token in emotion_lexicon)

        if token_count > 0:
            emotion_proportion = emotion_count / token_count
        else:
            emotion_proportion = 0.0

        engagement_sum = float(group["engagement_proxy"].sum())
        engagement_mean = float(group["engagement_proxy"].mean())
        post_count = int(len(group))

        rows.append(
            {
                "node": author_id,
                "post_count": post_count,
                "token_count": int(token_count),
                "emotion_count": int(emotion_count),
                "emotion_proportion": float(emotion_proportion),
                "engagement_sum": engagement_sum,
                "engagement_mean": engagement_mean,
            }
        )

    features = pd.DataFrame(rows)

    return features


def compute_misinformation_vulnerability_centrality(graph, author_features):
    in_degree = dict(graph.in_degree(weight="weight"))

    vulnerability = dict(
        zip(
            author_features["node"],
            author_features["emotion_proportion"],
        )
    )

    vulnerability = {
        node: float(vulnerability.get(node, 0.0))
        for node in graph.nodes()
    }

    vulnerability = minmax_scale_dict(vulnerability)

    current = vulnerability.copy()

    for _ in range(MVC_ITERATIONS):
        updated = {}

        for node in graph.nodes():
            exposure = float(in_degree.get(node, 0.0))
            updated[node] = exposure * current.get(node, 0.0)

        current = minmax_scale_dict(updated)

    return current


def compute_dynamic_influence_centrality(graph):
    current = {node: 1.0 for node in graph.nodes()}

    for _ in range(DIC_ITERATIONS):
        updated = {}

        for node in graph.nodes():
            incoming_sum = 0.0

            for predecessor in graph.predecessors(node):
                edge_weight = graph[predecessor][node].get("weight", 1.0)
                incoming_sum += current.get(predecessor, 0.0) * edge_weight

            updated[node] = current.get(node, 0.0) + incoming_sum

        current = minmax_scale_dict(updated)

    return current


# ============================================================
# 8. Output tables
# ============================================================

def combine_scores(
    graph,
    author_features,
    in_degree,
    out_degree,
    degree,
    eigenvector,
    betweenness,
    closeness,
    propagation,
    vulnerability,
    dynamic_influence,
):
    feature_map = author_features.set_index("node").to_dict(orient="index")

    rows = []

    for node in graph.nodes():
        features = feature_map.get(node, {})

        rows.append(
            {
                "node": node,
                "in_degree_weighted": float(in_degree.get(node, 0.0)),
                "out_degree_weighted": float(out_degree.get(node, 0.0)),
                "degree_total_weighted": float(degree.get(node, 0.0)),
                "eigenvector_centrality": float(eigenvector.get(node, 0.0)),
                "betweenness_centrality": float(betweenness.get(node, 0.0)),
                "closeness_harmonic_in": float(closeness.get(node, 0.0)),
                "propagation_centrality": float(propagation.get(node, 0.0)),
                "misinformation_vulnerability_centrality": float(vulnerability.get(node, 0.0)),
                "dynamic_influence_centrality": float(dynamic_influence.get(node, 0.0)),
                "post_count": int(features.get("post_count", 0)),
                "token_count": int(features.get("token_count", 0)),
                "emotion_count": int(features.get("emotion_count", 0)),
                "emotion_proportion": float(features.get("emotion_proportion", 0.0)),
                "engagement_sum": float(features.get("engagement_sum", 0.0)),
                "engagement_mean": float(features.get("engagement_mean", 0.0)),
            }
        )

    return pd.DataFrame(rows)


def save_top_node_tables(scores_df, out_dir, top_k=TOP_K):
    metric_columns = {
        "degree_total_weighted": "degree",
        "eigenvector_centrality": "eigenvector",
        "betweenness_centrality": "betweenness",
        "closeness_harmonic_in": "closeness",
        "propagation_centrality": "propagation",
        "misinformation_vulnerability_centrality": "vulnerability",
        "dynamic_influence_centrality": "dynamic_influence",
    }

    rows = []

    for column, metric_name in metric_columns.items():
        top_df = scores_df.sort_values(column, ascending=False).head(top_k)

        for rank, (_, row) in enumerate(top_df.iterrows(), start=1):
            rows.append(
                {
                    "metric": metric_name,
                    "rank": rank,
                    "node": row["node"],
                    "score": row[column],
                    "post_count": row["post_count"],
                    "engagement_sum": row["engagement_sum"],
                    "emotion_proportion": row["emotion_proportion"],
                }
            )

    top_nodes_df = pd.DataFrame(rows)
    top_nodes_df.to_csv(out_dir / "top_nodes_by_metric.csv", index=False)

    traditional = top_nodes_df[
        top_nodes_df["metric"].isin(
            ["degree", "eigenvector", "betweenness", "closeness"]
        )
    ].copy()

    advanced = top_nodes_df[
        top_nodes_df["metric"].isin(
            ["propagation", "vulnerability", "dynamic_influence"]
        )
    ].copy()

    traditional.to_csv(out_dir / "traditional_centrality_top_nodes.csv", index=False)
    advanced.to_csv(out_dir / "advanced_centrality_top_nodes.csv", index=False)

    return top_nodes_df


def create_overlap_tables(top_nodes_df, out_dir):
    metric_sets = {
        metric: set(group["node"].astype(str))
        for metric, group in top_nodes_df.groupby("metric")
    }

    rows = []

    metrics = sorted(metric_sets.keys())

    for metric_a in metrics:
        for metric_b in metrics:
            set_a = metric_sets[metric_a]
            set_b = metric_sets[metric_b]

            intersection = set_a & set_b
            union = set_a | set_b

            if len(union) > 0:
                jaccard = len(intersection) / len(union)
            else:
                jaccard = 0.0

            rows.append(
                {
                    "metric_a": metric_a,
                    "metric_b": metric_b,
                    "overlap_count": len(intersection),
                    "jaccard": round(jaccard, 4),
                    "shared_nodes": ", ".join(sorted(intersection)),
                }
            )

    overlap_df = pd.DataFrame(rows)
    overlap_df.to_csv(out_dir / "metric_overlap_summary.csv", index=False)

    traditional_union = set()

    for metric in ["degree", "eigenvector", "betweenness", "closeness"]:
        traditional_union |= metric_sets.get(metric, set())

    advanced_union = set()

    for metric in ["propagation", "vulnerability", "dynamic_influence"]:
        advanced_union |= metric_sets.get(metric, set())

    union_summary = pd.DataFrame(
        [
            {
                "traditional_union_count": len(traditional_union),
                "advanced_union_count": len(advanced_union),
                "combined_union_count": len(traditional_union | advanced_union),
                "advanced_exclusive_count": len(advanced_union - traditional_union),
                "advanced_exclusive_nodes": ", ".join(sorted(advanced_union - traditional_union)),
            }
        ]
    )

    union_summary.to_csv(out_dir / "traditional_advanced_union_summary.csv", index=False)

    return overlap_df, union_summary


# ============================================================
# 9. Proxy alignment and intervention analysis
# ============================================================

def overlap_at_k(scores_df, metric_col, proxy_col, k):
    metric_nodes = set(
        scores_df.sort_values(metric_col, ascending=False)
        .head(k)["node"]
        .astype(str)
    )

    proxy_nodes = set(
        scores_df.sort_values(proxy_col, ascending=False)
        .head(k)["node"]
        .astype(str)
    )

    if k == 0:
        return 0.0

    return len(metric_nodes & proxy_nodes) / k


def safe_spearman(scores_df, metric_col, proxy_col):
    temp = scores_df[[metric_col, proxy_col]].replace([np.inf, -np.inf], np.nan).dropna()

    if len(temp) < 3:
        return np.nan

    if temp[metric_col].nunique() <= 1 or temp[proxy_col].nunique() <= 1:
        return np.nan

    return float(spearmanr(temp[metric_col], temp[proxy_col]).correlation)


def create_proxy_alignment_table(scores_df, out_dir):
    active_df = scores_df[scores_df["in_degree_weighted"] > 0].copy()

    if len(active_df) == 0:
        active_df = scores_df.copy()

    comparisons = [
        (
            "propagation_vs_engagement_sum",
            "propagation_centrality",
            "engagement_sum",
        ),
        (
            "dynamic_influence_vs_engagement_sum",
            "dynamic_influence_centrality",
            "engagement_sum",
        ),
        (
            "propagation_vs_engagement_mean",
            "propagation_centrality",
            "engagement_mean",
        ),
        (
            "dynamic_influence_vs_engagement_mean",
            "dynamic_influence_centrality",
            "engagement_mean",
        ),
        (
            "vulnerability_vs_emotion_proportion",
            "misinformation_vulnerability_centrality",
            "emotion_proportion",
        ),
        (
            "vulnerability_vs_emotion_count",
            "misinformation_vulnerability_centrality",
            "emotion_count",
        ),
    ]

    rows = []

    for comparison_name, metric_col, proxy_col in comparisons:
        row = {
            "comparison": comparison_name,
            "metric": metric_col,
            "proxy": proxy_col,
            "spearman_rho": safe_spearman(active_df, metric_col, proxy_col),
            "n_nodes_evaluated": int(len(active_df)),
        }

        for k in [25, 50, 100, 200]:
            usable_k = min(k, len(active_df))

            if usable_k > 0:
                row[f"overlap_at_{k}"] = overlap_at_k(
                    active_df,
                    metric_col,
                    proxy_col,
                    usable_k,
                )
            else:
                row[f"overlap_at_{k}"] = np.nan

        rows.append(row)

    alignment_df = pd.DataFrame(rows)
    alignment_df.to_csv(out_dir / "proxy_alignment_summary.csv", index=False)

    return alignment_df


def remove_nodes_and_measure(graph, nodes_to_remove):
    original_weight = sum(
        data.get("weight", 1.0)
        for _, _, data in graph.edges(data=True)
    )

    reduced_graph = graph.copy()
    reduced_graph.remove_nodes_from(nodes_to_remove)

    remaining_weight = sum(
        data.get("weight", 1.0)
        for _, _, data in reduced_graph.edges(data=True)
    )

    if original_weight == 0:
        reduction = 0.0
    else:
        reduction = 1.0 - (remaining_weight / original_weight)

    return {
        "removed_node_count": int(len(set(nodes_to_remove))),
        "original_edge_weight": float(original_weight),
        "remaining_edge_weight": float(remaining_weight),
        "proportional_reduction": float(reduction),
    }


def create_intervention_summary(graph, top_nodes_df, out_dir):
    traditional_metrics = ["degree", "eigenvector", "betweenness", "closeness"]
    advanced_metrics = ["propagation", "vulnerability", "dynamic_influence"]

    traditional_nodes = set(
        top_nodes_df[top_nodes_df["metric"].isin(traditional_metrics)]["node"].astype(str)
    )

    advanced_nodes = set(
        top_nodes_df[top_nodes_df["metric"].isin(advanced_metrics)]["node"].astype(str)
    )

    combined_nodes = traditional_nodes | advanced_nodes

    rows = []

    traditional_result = remove_nodes_and_measure(graph, traditional_nodes)
    traditional_result["intervention_set"] = "traditional_top_nodes"
    rows.append(traditional_result)

    advanced_result = remove_nodes_and_measure(graph, advanced_nodes)
    advanced_result["intervention_set"] = "advanced_top_nodes"
    rows.append(advanced_result)

    combined_result = remove_nodes_and_measure(graph, combined_nodes)
    combined_result["intervention_set"] = "traditional_plus_advanced_top_nodes"
    rows.append(combined_result)

    intervention_df = pd.DataFrame(rows)
    intervention_df = intervention_df[
        [
            "intervention_set",
            "removed_node_count",
            "original_edge_weight",
            "remaining_edge_weight",
            "proportional_reduction",
        ]
    ]

    intervention_df.to_csv(out_dir / "node_removal_intervention_summary.csv", index=False)

    return intervention_df


# ============================================================
# 10. Main pipeline
# ============================================================

def main():
    parser = argparse.ArgumentParser()

    parser.add_argument("--discussion", type=str, default=DEFAULT_DISCUSSION_FILE)
    parser.add_argument("--out", type=str, default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--top_k", type=int, default=TOP_K)

    args, _ = parser.parse_known_args()

    set_seed(SEED)

    out_dir = Path(args.out)
    out_dir.mkdir(parents=True, exist_ok=True)

    discussion_path = upload_data_if_needed(args.discussion)

    print(f"\nUsing discussion post file: {discussion_path}")

    df = load_discussion_posts(discussion_path)

    dataset_summary = save_dataset_summary(df, out_dir)

    print("\nDataset summary")
    print(dataset_summary.to_string(index=False))

    graph = build_reply_graph(df)

    graph_summary = save_graph_summary(graph, out_dir)

    print("\nGraph summary")
    print(graph_summary.to_string(index=False))

    author_features = aggregate_author_features(df)
    author_features.to_csv(out_dir / "author_proxy_features.csv", index=False)

    in_degree, out_degree, degree = compute_degree_scores(graph)

    print("\nComputing propagation centrality")
    propagation = compute_propagation_centrality(graph)

    print("Computing eigenvector centrality")
    eigenvector = compute_eigenvector_score(graph, propagation)

    print("Computing betweenness centrality")
    betweenness = compute_betweenness_score(graph)

    print("Computing closeness centrality")
    candidate_base_score = {
        node: degree.get(node, 0.0) + propagation.get(node, 0.0)
        for node in graph.nodes()
    }
    closeness = compute_closeness_score(graph, candidate_base_score)

    print("Computing misinformation vulnerability centrality")
    vulnerability = compute_misinformation_vulnerability_centrality(
        graph,
        author_features,
    )

    print("Computing dynamic influence centrality")
    dynamic_influence = compute_dynamic_influence_centrality(graph)

    scores_df = combine_scores(
        graph=graph,
        author_features=author_features,
        in_degree=in_degree,
        out_degree=out_degree,
        degree=degree,
        eigenvector=eigenvector,
        betweenness=betweenness,
        closeness=closeness,
        propagation=propagation,
        vulnerability=vulnerability,
        dynamic_influence=dynamic_influence,
    )

    scores_df.to_csv(out_dir / "all_centrality_scores.csv", index=False)

    top_nodes_df = save_top_node_tables(scores_df, out_dir, top_k=args.top_k)

    overlap_df, union_summary = create_overlap_tables(top_nodes_df, out_dir)

    alignment_df = create_proxy_alignment_table(scores_df, out_dir)

    intervention_df = create_intervention_summary(graph, top_nodes_df, out_dir)

    print("\nTop nodes by metric")
    print(top_nodes_df.head(args.top_k * 7).to_string(index=False))

    print("\nTraditional and advanced union summary")
    print(union_summary.to_string(index=False))

    print("\nProxy alignment summary")
    print(alignment_df.to_string(index=False))

    print("\nNode removal intervention summary")
    print(intervention_df.to_string(index=False))

    print("\nFinished. Output files:")

    for file in sorted(out_dir.glob("*")):
        print("  ", file)

    try:
        from IPython.display import display

        selected_outputs = [
            out_dir / "dataset_summary.csv",
            out_dir / "graph_summary.csv",
            out_dir / "traditional_centrality_top_nodes.csv",
            out_dir / "advanced_centrality_top_nodes.csv",
            out_dir / "traditional_advanced_union_summary.csv",
            out_dir / "proxy_alignment_summary.csv",
            out_dir / "node_removal_intervention_summary.csv",
        ]

        for output_file in selected_outputs:
            if output_file.exists():
                print("\n" + "=" * 90)
                print(output_file)
                print("=" * 90)
                display(pd.read_csv(output_file))

    except Exception:
        pass


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# ELM-SIRMMM propagation modelling pipeline
# Datasets: FibVID, MC-Fake, and Monant
#
# Outputs:
# - dataset-level daily signals
# - Plain SIRMMM fit
# - ELM-SIRMMM fit
# - fit metrics
# - trajectory summaries
# - ELM coefficients
# - robustness checks
# - plots for each dataset
# - combined report-ready tables
# ============================================================

import os
import re
import sys
import json
import math
import random
import warnings
import subprocess
import importlib.util
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Any

# ============================================================
# 0. Install missing packages
# ============================================================

PACKAGE_IMPORTS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "scikit-learn": "sklearn",
    "matplotlib": "matplotlib",
    "textblob": "textblob",
    "ftfy": "ftfy",
}

for package_name, module_name in PACKAGE_IMPORTS.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            check=True
        )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from textblob import TextBlob
from ftfy import fix_text

warnings.filterwarnings("ignore")

# ============================================================
# 1. Global configuration
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_ROOT = Path("tri_dataset_elm_sirmmm_results")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SMOOTH_WINDOW = 3
BETA_CAP = 5.0
EPS = 1e-8

MISINFO_NUMERIC_LABELS = {1}
MISINFO_STRING_LABELS = {
    "1", "fake", "false", "misinformation", "misinfo",
    "rumor", "rumour", "unverified", "misleading"
}

# ============================================================
# 2. Dataset configuration
# ============================================================

@dataclass
class DatasetConfig:
    name: str
    default_file: str

    date_candidates: List[str]
    text_candidates: List[str]
    label_candidates: List[str]

    engagement_candidates: List[str]
    incidence_candidates: List[str]

    user_candidates: List[str] = field(default_factory=list)

    misinfo_numeric_labels: set = field(default_factory=lambda: MISINFO_NUMERIC_LABELS)
    misinfo_string_labels: set = field(default_factory=lambda: MISINFO_STRING_LABELS)

    treat_all_rows_as_misinfo_if_no_label: bool = False

    engagement_mode_label: str = "auto"
    incidence_mode_label: str = "auto"

    notes: str = ""


DATASET_CONFIGS = [
    DatasetConfig(
        name="FibVID",
        default_file="fibvid.csv",
        date_candidates=[
            "created_at", "tweet_date", "date", "datetime",
            "timestamp", "publish_date", "created"
        ],
        text_candidates=[
            "text", "tweet", "tweet_text", "full_text", "content",
            "body", "message"
        ],
        label_candidates=[
            "label", "labels", "misinformation", "is_fake",
            "fake", "veracity", "class", "target"
        ],
        engagement_candidates=[
            "retweet_count", "retweets", "n_retweets",
            "like_count", "likes", "favorite_count", "favorites",
            "reply_count", "replies", "n_replies",
            "quote_count", "quotes"
        ],
        incidence_candidates=[
            "retweet_count", "retweets", "n_retweets",
            "reply_count", "replies", "n_replies",
            "quote_count", "quotes"
        ],
        user_candidates=[
            "user_id", "author_id", "screen_name", "username"
        ],
        treat_all_rows_as_misinfo_if_no_label=False,
        engagement_mode_label="log1p(retweets + replies + likes + quotes where available)",
        incidence_mode_label="1 + available retweet/reply/quote activity per misinformation tweet",
        notes="FibVID is used as the main COVID-19 misinformation training and within-dataset comparison corpus."
    ),

    DatasetConfig(
        name="MC-Fake",
        default_file="mc_fake.csv",
        date_candidates=[
            "publish_date", "date", "created_at", "timestamp"
        ],
        text_candidates=[
            "title", "text"
        ],
        label_candidates=[
            "labels", "label", "class", "target"
        ],
        engagement_candidates=[
            "n_retweets", "n_replies"
        ],
        incidence_candidates=[
            "n_tweets", "n_retweets", "n_replies"
        ],
        user_candidates=[
            "n_users", "user_ids"
        ],
        treat_all_rows_as_misinfo_if_no_label=False,
        engagement_mode_label="log1p(n_retweets + n_replies)",
        incidence_mode_label="n_tweets + n_retweets + n_replies",
        notes="MC-Fake is used as a high-volatility cascade setting with claim-level tweet, retweet and reply counts."
    ),

    DatasetConfig(
        name="Monant",
        default_file="monant.csv",
        date_candidates=[
            "date", "created_at", "timestamp", "post_date",
            "published_at", "time"
        ],
        text_candidates=[
            "title", "text", "body", "content", "post",
            "comment", "question", "answer"
        ],
        label_candidates=[
            "label", "labels", "misinformation", "is_fake",
            "veracity", "class", "target"
        ],
        engagement_candidates=[
            "up_votes", "upvotes", "upvote", "down_votes",
            "downvotes", "downvote", "score", "votes",
            "reply_count", "replies", "comments", "n_comments"
        ],
        incidence_candidates=[
            "reply_count", "replies", "comments", "n_comments",
            "up_votes", "upvotes", "down_votes", "downvotes",
            "score", "votes"
        ],
        user_candidates=[
            "user_id", "author_id", "username"
        ],
        treat_all_rows_as_misinfo_if_no_label=True,
        engagement_mode_label="log1p(up_votes + down_votes + replies/comments where available)",
        incidence_mode_label="1 + available forum interaction activity",
        notes="Monant is used as a low-variance health-discussion setting where behavioural signals may be sparse or flat."
    )
]

# ============================================================
# 3. File handling and robust loading
# ============================================================

def normalise_name(x: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")


def upload_data_if_needed(default_file: str, dataset_name: str) -> str:
    path = Path(default_file)

    if path.exists():
        return str(path)

    data_dir = Path("data")
    data_dir.mkdir(exist_ok=True)

    candidate = data_dir / default_file
    if candidate.exists():
        return str(candidate)

    csv_candidates = (
        list(data_dir.glob("*.csv")) +
        list(data_dir.glob("*.tsv")) +
        list(data_dir.glob("*.txt"))
    )

    if csv_candidates:
        print(f"[{dataset_name}] Candidate files found in ./data:")
        for idx, p in enumerate(csv_candidates):
            print(f"  {idx + 1}. {p}")
        print(f"[{dataset_name}] Using first candidate: {csv_candidates[0]}")
        return str(csv_candidates[0])

    try:
        from google.colab import files
        print(f"\nUpload the {dataset_name} file now. Expected file name: {default_file}")
        uploaded = files.upload()

        if len(uploaded) == 0:
            raise FileNotFoundError("No file uploaded.")

        filename = list(uploaded.keys())[0]
        uploaded_path = Path(filename)
        destination = data_dir / filename

        if destination.exists():
            destination.unlink()

        uploaded_path.rename(destination)
        return str(destination)

    except Exception as exc:
        raise FileNotFoundError(
            f"No file found for {dataset_name}. Place {default_file} in the current folder or ./data, "
            f"or upload it in Colab."
        ) from exc


def try_read_table(file_path: str) -> pd.DataFrame:
    file_path = Path(file_path)

    attempts = []
    separators = ["\t", ",", ";", "|", None]
    encodings = ["utf-8", "latin1", "cp1252"]

    for enc in encodings:
        for sep in separators:
            try:
                df = pd.read_csv(
                    file_path,
                    sep=sep,
                    dtype=str,
                    encoding=enc,
                    engine="python",
                    on_bad_lines="skip"
                )

                if df.shape[1] > 1:
                    attempts.append((sep, enc, df))

            except Exception:
                pass

    if not attempts:
        raise ValueError(f"Could not read file: {file_path}")

    best_sep, best_enc, best_df = max(attempts, key=lambda item: item[2].shape[1])
    best_df.columns = [str(c).strip() for c in best_df.columns]

    print(
        f"Loaded {file_path} using separator={repr(best_sep)}, "
        f"encoding={best_enc}, shape={best_df.shape}"
    )
    print("Detected columns:", list(best_df.columns)[:30])

    return best_df.copy()

# ============================================================
# 4. Column matching and preprocessing utilities
# ============================================================

def find_first_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    col_map = {normalise_name(c): c for c in df.columns}
    candidate_norms = [normalise_name(c) for c in candidates]

    for cand in candidate_norms:
        if cand in col_map:
            return col_map[cand]

    for cand in candidate_norms:
        for norm_col, original_col in col_map.items():
            if cand == norm_col or cand in norm_col or norm_col in cand:
                return original_col

    return None


def find_matching_columns(df: pd.DataFrame, candidates: List[str]) -> List[str]:
    col_map = {normalise_name(c): c for c in df.columns}
    candidate_norms = [normalise_name(c) for c in candidates]
    matches = []

    for cand in candidate_norms:
        for norm_col, original_col in col_map.items():
            if cand == norm_col or cand in norm_col or norm_col in cand:
                if original_col not in matches:
                    matches.append(original_col)

    return matches


def clean_text_value(x: Any) -> str:
    if pd.isna(x):
        return ""

    x = str(x)
    x = fix_text(x)
    x = re.sub(r"http\S+|www\.\S+", " URL ", x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"@\w+", " USER ", x)
    x = re.sub(r"#", "", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def safe_numeric(series: pd.Series, default: float = 0.0) -> pd.Series:
    return pd.to_numeric(series, errors="coerce").fillna(default).astype(float)


def parse_dates(series: pd.Series) -> pd.Series:
    parsed = pd.to_datetime(series, errors="coerce", dayfirst=False)

    if parsed.isna().mean() > 0.50:
        parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)

    return parsed


def sentiment_polarity(text: str) -> float:
    try:
        return float(TextBlob(str(text)).sentiment.polarity)
    except Exception:
        return 0.0


def word_count(text: str) -> int:
    return len(re.findall(r"\b\w+\b", str(text)))


def combine_text_columns(df: pd.DataFrame, text_cols: List[str]) -> pd.Series:
    combined = []

    for _, row in df.iterrows():
        parts = []

        for col in text_cols:
            value = clean_text_value(row.get(col, ""))

            if value:
                parts.append(value)

        combined.append(" ".join(parts).strip())

    return pd.Series(combined, index=df.index)


def build_misinfo_mask(
    df: pd.DataFrame,
    label_col: Optional[str],
    config: DatasetConfig
) -> pd.Series:
    if label_col is None:
        if config.treat_all_rows_as_misinfo_if_no_label:
            print(f"[{config.name}] No label column found. Treating all rows as misinformation cascade activity.")
            return pd.Series(True, index=df.index)

        raise ValueError(
            f"[{config.name}] No label column found. Available columns: {list(df.columns)}"
        )

    raw = df[label_col].astype(str).str.strip()
    raw_lower = raw.str.lower()

    numeric = pd.to_numeric(raw, errors="coerce")
    numeric_match = numeric.isin(list(config.misinfo_numeric_labels))
    string_match = raw_lower.isin(config.misinfo_string_labels)

    mask = numeric_match | string_match

    if mask.sum() == 0:
        print(f"[{config.name}] Warning: no rows matched misinformation labels using column '{label_col}'.")
        print("Unique label examples:", raw.dropna().unique()[:20])

    return mask.fillna(False)


def build_numeric_sum(df: pd.DataFrame, cols: List[str]) -> pd.Series:
    if not cols:
        return pd.Series(0.0, index=df.index)

    total = pd.Series(0.0, index=df.index)

    for col in cols:
        total = total + safe_numeric(df[col], default=0.0)

    return total

# ============================================================
# 5. Dataset preparation
# ============================================================

def prepare_dataset(raw_df: pd.DataFrame, config: DatasetConfig) -> Dict[str, Any]:
    df = raw_df.copy()
    df.columns = [str(c).strip() for c in df.columns]

    date_col = find_first_column(df, config.date_candidates)
    label_col = find_first_column(df, config.label_candidates)
    text_cols = find_matching_columns(df, config.text_candidates)
    engagement_cols = find_matching_columns(df, config.engagement_candidates)
    incidence_cols = find_matching_columns(df, config.incidence_candidates)
    user_cols = find_matching_columns(df, config.user_candidates)

    if date_col is None:
        raise ValueError(
            f"[{config.name}] No date column found. Candidate names were: {config.date_candidates}"
        )

    if not text_cols:
        raise ValueError(
            f"[{config.name}] No text column found. Candidate names were: {config.text_candidates}"
        )

    print(f"\n[{config.name}] Column mapping")
    print("  Date column:", date_col)
    print("  Label column:", label_col)
    print("  Text columns:", text_cols)
    print("  Engagement columns:", engagement_cols)
    print("  Incidence columns:", incidence_cols)
    print("  User columns:", user_cols)

    df["date"] = parse_dates(df[date_col])
    df["date_day"] = df["date"].dt.floor("D")

    df["combined_text"] = combine_text_columns(df, text_cols)
    df["sentiment"] = df["combined_text"].map(sentiment_polarity).astype(float)
    df["cognition"] = df["combined_text"].map(word_count).astype(float)

    df["engagement_raw"] = build_numeric_sum(df, engagement_cols)
    df["engagement"] = np.log1p(df["engagement_raw"].clip(lower=0))

    incidence_sum = build_numeric_sum(df, incidence_cols)

    if config.name == "MC-Fake":
        df["incidence_item"] = incidence_sum.clip(lower=0)
    else:
        df["incidence_item"] = 1.0 + incidence_sum.clip(lower=0)

    if user_cols:
        first_user_col = user_cols[0]
        user_numeric = pd.to_numeric(df[first_user_col], errors="coerce")

        if user_numeric.notna().mean() > 0.90:
            df["n_users_proxy"] = user_numeric.fillna(0.0).astype(float)
        else:
            df["n_users_proxy"] = (
                df[first_user_col]
                .astype(str)
                .replace("nan", np.nan)
                .notna()
                .astype(float)
            )
    else:
        df["n_users_proxy"] = 1.0

    df["is_misinfo"] = build_misinfo_mask(df, label_col, config)

    before = len(df)
    df = df[df["date_day"].notna()].copy()
    df = df[df["combined_text"].str.len() > 0].copy()
    after = len(df)

    print(f"[{config.name}] Rows before filtering: {before}, after date/text filtering: {after}")
    print(f"[{config.name}] Misinformation rows: {int(df['is_misinfo'].sum())}")

    mapping = {
        "date_col": date_col,
        "label_col": label_col,
        "text_cols": text_cols,
        "engagement_cols": engagement_cols,
        "incidence_cols": incidence_cols,
        "user_cols": user_cols,
        "engagement_mode_label": config.engagement_mode_label,
        "incidence_mode_label": config.incidence_mode_label,
        "notes": config.notes
    }

    return {
        "df": df.reset_index(drop=True),
        "mapping": mapping
    }

# ============================================================
# 6. Daily aggregation and normalisation
# ============================================================

def aggregate_daily(
    prepared_df: pd.DataFrame,
    config: DatasetConfig,
    smooth_window: int = SMOOTH_WINDOW
) -> pd.DataFrame:
    df_m = prepared_df[prepared_df["is_misinfo"]].copy()

    if len(df_m) == 0:
        raise ValueError(f"[{config.name}] No misinformation rows after filtering.")

    grouped = (
        df_m.groupby("date_day")
        .agg(
            incidence=("incidence_item", "sum"),
            n_items=("combined_text", "count"),
            sentiment=("sentiment", "mean"),
            engagement=("engagement", "mean"),
            cognition=("cognition", "mean"),
            engagement_raw_sum=("engagement_raw", "sum"),
            n_users_proxy=("n_users_proxy", "sum")
        )
        .reset_index()
        .sort_values("date_day")
    )

    full_dates = pd.date_range(grouped["date_day"].min(), grouped["date_day"].max(), freq="D")

    grouped = (
        grouped.set_index("date_day")
        .reindex(full_dates)
        .rename_axis("date_day")
        .reset_index()
    )

    for col in ["incidence", "n_items", "engagement_raw_sum", "n_users_proxy"]:
        grouped[col] = grouped[col].fillna(0.0)

    for col in ["sentiment", "engagement", "cognition"]:
        grouped[col] = grouped[col].interpolate(limit_direction="both").fillna(0.0)

    grouped["incidence_raw"] = grouped["incidence"].astype(float)

    if smooth_window and smooth_window > 1:
        grouped["incidence"] = (
            grouped["incidence_raw"]
            .rolling(window=smooth_window, min_periods=1, center=True)
            .mean()
        )

    grouped["day"] = np.arange(len(grouped), dtype=float)

    return grouped


def add_normalised_features(daily: pd.DataFrame, scaler_type: str = "zscore") -> pd.DataFrame:
    daily = daily.copy()

    feature_cols = ["sentiment", "engagement", "cognition"]

    if scaler_type == "zscore":
        scaler = StandardScaler()
    elif scaler_type == "minmax":
        scaler = MinMaxScaler()
    else:
        raise ValueError("scaler_type must be 'zscore' or 'minmax'.")

    values = daily[feature_cols].values.astype(float)

    if len(daily) < 2:
        z = np.zeros_like(values)
    else:
        z = scaler.fit_transform(values)

    daily["z_sentiment"] = z[:, 0]
    daily["z_engagement"] = z[:, 1]
    daily["z_cognition"] = z[:, 2]

    return daily

# ============================================================
# 7. ODE model definitions
# ============================================================

def interpolate_series(t, days, values):
    return np.interp(t, days, values)


def sirmmm_plain_rhs(t, y, beta_m, gamma_m, N):
    MS, MI, MR = y

    force = beta_m * MS * MI / max(N, EPS)

    dMS = -force
    dMI = force - gamma_m * MI
    dMR = gamma_m * MI

    return [dMS, dMI, dMR]


def sirmmm_elm_rhs(t, y, params, N, days, z_sent, z_eng, z_cog):
    beta0, beta_s, beta_e, beta_c, gamma_m = params

    MS, MI, MR = y

    s_t = interpolate_series(t, days, z_sent)
    e_t = interpolate_series(t, days, z_eng)
    c_t = interpolate_series(t, days, z_cog)

    beta_t = beta0 + beta_s * s_t + beta_e * e_t + beta_c * c_t
    beta_t = float(np.clip(beta_t, EPS, BETA_CAP))

    force = beta_t * MS * MI / max(N, EPS)

    dMS = -force
    dMI = force - gamma_m * MI
    dMR = gamma_m * MI

    return [dMS, dMI, dMR]


def solve_plain(days, beta_m, gamma_m, N, MI0):
    MS0 = max(N - MI0, EPS)
    MR0 = 0.0

    if len(days) < 2:
        return np.array([[MS0, MI0, MR0]])

    sol = solve_ivp(
        fun=lambda t, y: sirmmm_plain_rhs(t, y, beta_m, gamma_m, N),
        t_span=(days.min(), days.max()),
        y0=[MS0, MI0, MR0],
        t_eval=days,
        method="RK45",
        rtol=1e-6,
        atol=1e-6,
    )

    if not sol.success:
        raise RuntimeError("Plain SIRMMM solver failed.")

    return sol.y.T


def solve_elm(days, params, N, MI0, z_sent, z_eng, z_cog):
    MS0 = max(N - MI0, EPS)
    MR0 = 0.0

    if len(days) < 2:
        return np.array([[MS0, MI0, MR0]])

    sol = solve_ivp(
        fun=lambda t, y: sirmmm_elm_rhs(t, y, params, N, days, z_sent, z_eng, z_cog),
        t_span=(days.min(), days.max()),
        y0=[MS0, MI0, MR0],
        t_eval=days,
        method="RK45",
        rtol=1e-6,
        atol=1e-6,
    )

    if not sol.success:
        raise RuntimeError("ELM-SIRMMM solver failed.")

    return sol.y.T


def beta_m_time_series(days, params, z_sent, z_eng, z_cog):
    beta0, beta_s, beta_e, beta_c, gamma_m = params

    beta_values = (
        beta0
        + beta_s * z_sent
        + beta_e * z_eng
        + beta_c * z_cog
    )

    return np.clip(beta_values, EPS, BETA_CAP)

# ============================================================
# 8. Fitting functions
# ============================================================

def define_population_and_target(daily: pd.DataFrame):
    y_obs = daily["incidence"].values.astype(float)

    if np.nanmax(y_obs) <= 0:
        raise ValueError("Observed incidence is all zero. Check label mapping and incidence fields.")

    cumulative_activity = np.nansum(y_obs)
    peak_activity = np.nanmax(y_obs)

    N = max(cumulative_activity, peak_activity * 10.0, 1000.0)
    MI0 = max(y_obs[0], 1.0)

    return y_obs, float(N), float(MI0)


def fit_plain_model(daily: pd.DataFrame):
    days = daily["day"].values.astype(float)
    y_obs, N, MI0 = define_population_and_target(daily)

    def residuals(theta):
        beta_m, gamma_m = theta
        traj = solve_plain(days, beta_m, gamma_m, N, MI0)
        MI = traj[:, 1]
        return MI - y_obs

    result = least_squares(
        residuals,
        x0=np.array([0.20, 0.10]),
        bounds=([EPS, EPS], [BETA_CAP, 2.0]),
        max_nfev=3000,
    )

    params = result.x
    traj = solve_plain(days, params[0], params[1], N, MI0)

    return {
        "params": params,
        "trajectory": traj,
        "N": N,
        "MI0": MI0,
        "success": result.success,
        "cost": result.cost,
    }


def fit_elm_model(daily: pd.DataFrame):
    days = daily["day"].values.astype(float)
    y_obs, N, MI0 = define_population_and_target(daily)

    z_sent = daily["z_sentiment"].values.astype(float)
    z_eng = daily["z_engagement"].values.astype(float)
    z_cog = daily["z_cognition"].values.astype(float)

    def residuals(theta):
        traj = solve_elm(days, theta, N, MI0, z_sent, z_eng, z_cog)
        MI = traj[:, 1]
        return MI - y_obs

    result = least_squares(
        residuals,
        x0=np.array([0.20, -0.02, -0.02, -0.02, 0.10]),
        bounds=(
            np.array([EPS, -BETA_CAP, -BETA_CAP, -BETA_CAP, EPS]),
            np.array([BETA_CAP, BETA_CAP, BETA_CAP, BETA_CAP, 2.0])
        ),
        max_nfev=5000,
    )

    params = result.x
    traj = solve_elm(days, params, N, MI0, z_sent, z_eng, z_cog)
    beta_values = beta_m_time_series(days, params, z_sent, z_eng, z_cog)

    return {
        "params": params,
        "trajectory": traj,
        "beta_values": beta_values,
        "N": N,
        "MI0": MI0,
        "success": result.success,
        "cost": result.cost,
    }

# ============================================================
# 9. Evaluation summaries
# ============================================================

def evaluate_fit(y_obs, y_pred):
    rmse = math.sqrt(mean_squared_error(y_obs, y_pred))
    mae = mean_absolute_error(y_obs, y_pred)

    try:
        r2 = r2_score(y_obs, y_pred)
    except Exception:
        r2 = np.nan

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
    }


def trajectory_summary(daily, trajectory, N, dataset_name, model_name):
    days = daily["day"].values.astype(float)
    MS = trajectory[:, 0]
    MI = trajectory[:, 1]
    MR = trajectory[:, 2]

    peak_idx = int(np.argmax(MI))

    return {
        "dataset": dataset_name,
        "model": model_name,
        "peak_MI_day_index": float(days[peak_idx]),
        "peak_MI_calendar_date": str(daily.loc[peak_idx, "date_day"].date()),
        "peak_MI_count": float(MI[peak_idx]),
        "peak_MI_percent_population": float(100.0 * MI[peak_idx] / N),
        "MS_end_count": float(MS[-1]),
        "MI_end_count": float(MI[-1]),
        "MR_end_count": float(MR[-1]),
        "MS_end_percent": float(100.0 * MS[-1] / N),
        "MI_end_percent": float(100.0 * MI[-1] / N),
        "MR_end_percent": float(100.0 * MR[-1] / N),
        "correction_efficiency_MR_end_over_MI_peak": float(MR[-1] / max(MI[peak_idx], EPS)),
    }


def build_results_tables(dataset_name, daily, plain_fit, elm_fit, out_dir):
    y_obs = daily["incidence"].values.astype(float)

    plain_MI = plain_fit["trajectory"][:, 1]
    elm_MI = elm_fit["trajectory"][:, 1]

    plain_metrics = evaluate_fit(y_obs, plain_MI)
    elm_metrics = evaluate_fit(y_obs, elm_MI)

    perf = pd.DataFrame([
        {"dataset": dataset_name, "model": "Plain_SIRMMM", **plain_metrics},
        {"dataset": dataset_name, "model": "ELM_SIRMMM", **elm_metrics},
    ])

    rmse_plain = perf.loc[perf["model"] == "Plain_SIRMMM", "RMSE"].iloc[0]
    rmse_elm = perf.loc[perf["model"] == "ELM_SIRMMM", "RMSE"].iloc[0]

    perf["RMSE_improvement_vs_plain_percent"] = np.nan
    perf.loc[perf["model"] == "ELM_SIRMMM", "RMSE_improvement_vs_plain_percent"] = (
        100.0 * (rmse_plain - rmse_elm) / max(rmse_plain, EPS)
    )

    plain_summary = trajectory_summary(
        daily, plain_fit["trajectory"], plain_fit["N"], dataset_name, "Plain_SIRMMM"
    )
    elm_summary = trajectory_summary(
        daily, elm_fit["trajectory"], elm_fit["N"], dataset_name, "ELM_SIRMMM"
    )

    traj = pd.DataFrame([plain_summary, elm_summary])

    beta0, beta_s, beta_e, beta_c, gamma_m = elm_fit["params"]

    coefficients = pd.DataFrame([
        {
            "dataset": dataset_name,
            "parameter": "beta0",
            "value": beta0,
            "interpretation": "Baseline misinformation transmission"
        },
        {
            "dataset": dataset_name,
            "parameter": "beta_sentiment",
            "value": beta_s,
            "interpretation": "Conditional effect of sentiment on beta_m(t)"
        },
        {
            "dataset": dataset_name,
            "parameter": "beta_engagement",
            "value": beta_e,
            "interpretation": "Conditional effect of engagement on beta_m(t)"
        },
        {
            "dataset": dataset_name,
            "parameter": "beta_cognition",
            "value": beta_c,
            "interpretation": "Conditional effect of cognitive effort or word count on beta_m(t)"
        },
        {
            "dataset": dataset_name,
            "parameter": "gamma_m",
            "value": gamma_m,
            "interpretation": "Disengagement or misinformation recovery rate"
        },
    ])

    daily_out = daily.copy()
    daily_out["plain_MS"] = plain_fit["trajectory"][:, 0]
    daily_out["plain_MI"] = plain_fit["trajectory"][:, 1]
    daily_out["plain_MR"] = plain_fit["trajectory"][:, 2]
    daily_out["elm_MS"] = elm_fit["trajectory"][:, 0]
    daily_out["elm_MI"] = elm_fit["trajectory"][:, 1]
    daily_out["elm_MR"] = elm_fit["trajectory"][:, 2]
    daily_out["elm_beta_m_t"] = elm_fit["beta_values"]

    perf.to_csv(out_dir / "performance_summary.csv", index=False)
    traj.to_csv(out_dir / "trajectory_summary.csv", index=False)
    coefficients.to_csv(out_dir / "elm_coefficients.csv", index=False)
    daily_out.to_csv(out_dir / "daily_model_outputs.csv", index=False)

    return perf, traj, coefficients, daily_out

# ============================================================
# 10. Robustness checks
# ============================================================

def rerun_with_feature_variant(prepared_df, config, engagement_transform, scaler_type):
    df = prepared_df.copy()

    if engagement_transform == "log1p_default":
        df["engagement"] = np.log1p(df["engagement_raw"].clip(lower=0))
    elif engagement_transform == "raw_minmax":
        df["engagement"] = df["engagement_raw"].clip(lower=0)
    elif engagement_transform == "binary_activity":
        df["engagement"] = (df["engagement_raw"] > 0).astype(float)
    else:
        raise ValueError("Unknown engagement_transform.")

    daily = aggregate_daily(df, config)
    daily = add_normalised_features(daily, scaler_type=scaler_type)

    plain_fit = fit_plain_model(daily)
    elm_fit = fit_elm_model(daily)

    y_obs = daily["incidence"].values.astype(float)
    plain_MI = plain_fit["trajectory"][:, 1]
    elm_MI = elm_fit["trajectory"][:, 1]

    plain_metrics = evaluate_fit(y_obs, plain_MI)
    elm_metrics = evaluate_fit(y_obs, elm_MI)

    beta0, beta_s, beta_e, beta_c, gamma_m = elm_fit["params"]

    return {
        "plain_RMSE": plain_metrics["RMSE"],
        "elm_RMSE": elm_metrics["RMSE"],
        "RMSE_improvement_percent": 100.0 * (
            plain_metrics["RMSE"] - elm_metrics["RMSE"]
        ) / max(plain_metrics["RMSE"], EPS),
        "beta0": beta0,
        "beta_sentiment": beta_s,
        "beta_engagement": beta_e,
        "beta_cognition": beta_c,
        "gamma_m": gamma_m,
        "elm_success": elm_fit["success"],
    }


def run_robustness(prepared_df, config, out_dir):
    rows = []

    configs = [
        ("log1p_default", "zscore"),
        ("log1p_default", "minmax"),
        ("raw_minmax", "zscore"),
        ("binary_activity", "zscore"),
    ]

    for engagement_transform, scaler_type in configs:
        cfg_name = f"{engagement_transform}_{scaler_type}"

        try:
            result = rerun_with_feature_variant(
                prepared_df=prepared_df,
                config=config,
                engagement_transform=engagement_transform,
                scaler_type=scaler_type
            )

            rows.append({
                "dataset": config.name,
                "config": cfg_name,
                "engagement_transform": engagement_transform,
                "scaler_type": scaler_type,
                **result
            })

        except Exception as exc:
            rows.append({
                "dataset": config.name,
                "config": cfg_name,
                "engagement_transform": engagement_transform,
                "scaler_type": scaler_type,
                "error": str(exc)
            })

    robustness = pd.DataFrame(rows)
    robustness.to_csv(out_dir / "robustness_checks.csv", index=False)

    return robustness

# ============================================================
# 11. Plotting
# ============================================================

def plot_observed_vs_fitted(dataset_name, daily_out, out_dir):
    plt.figure(figsize=(10, 6))
    plt.scatter(
        daily_out["date_day"],
        daily_out["incidence"],
        label=f"Observed {dataset_name} incidence",
        s=20
    )
    plt.plot(daily_out["date_day"], daily_out["plain_MI"], label="Plain SIRMMM")
    plt.plot(daily_out["date_day"], daily_out["elm_MI"], label="ELM-SIRMMM")
    plt.xlabel("Date")
    plt.ylabel("Misinformation incidence / cascade intensity")
    plt.title(f"{dataset_name}: observed incidence vs fitted SIRMMM trajectories")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(out_dir / f"{normalise_name(dataset_name)}_observed_vs_fitted.png", dpi=300)
    plt.close()


def plot_compartments(dataset_name, daily_out, out_dir):
    plt.figure(figsize=(10, 6))
    plt.plot(daily_out["date_day"], daily_out["elm_MS"], label="MS, misinformation susceptible")
    plt.plot(daily_out["date_day"], daily_out["elm_MI"], label="MI, active misinformation sharers")
    plt.plot(daily_out["date_day"], daily_out["elm_MR"], label="MR, misinformation recovered")
    plt.xlabel("Date")
    plt.ylabel("Compartment count")
    plt.title(f"{dataset_name}: ELM-SIRMMM compartment trajectories")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(out_dir / f"{normalise_name(dataset_name)}_elm_sirmmm_compartments.png", dpi=300)
    plt.close()


def plot_beta_and_signals(dataset_name, daily_out, out_dir):
    plt.figure(figsize=(10, 6))
    plt.plot(daily_out["date_day"], daily_out["elm_beta_m_t"], label="beta_m(t)")
    plt.xlabel("Date")
    plt.ylabel("Transmission coefficient")
    plt.title(f"{dataset_name}: time-varying misinformation transmission rate beta_m(t)")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(out_dir / f"{normalise_name(dataset_name)}_beta_m_time_series.png", dpi=300)
    plt.close()

    plt.figure(figsize=(10, 6))
    plt.plot(daily_out["date_day"], daily_out["z_sentiment"], label="Sentiment, normalised")
    plt.plot(daily_out["date_day"], daily_out["z_engagement"], label="Engagement, normalised")
    plt.plot(daily_out["date_day"], daily_out["z_cognition"], label="Cognition, normalised")
    plt.xlabel("Date")
    plt.ylabel("Normalised value")
    plt.title(f"{dataset_name}: normalised ELM behavioural inputs")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(out_dir / f"{normalise_name(dataset_name)}_elm_behavioural_inputs.png", dpi=300)
    plt.close()


def make_all_plots(dataset_name, daily_out, out_dir):
    plot_observed_vs_fitted(dataset_name, daily_out, out_dir)
    plot_compartments(dataset_name, daily_out, out_dir)
    plot_beta_and_signals(dataset_name, daily_out, out_dir)

# ============================================================
# 12. Dataset-level interpretation output
# ============================================================

def write_dataset_interpretation(config, perf, traj, coefficients, robustness, out_dir):
    dataset_name = config.name

    plain_rmse = perf.loc[perf["model"] == "Plain_SIRMMM", "RMSE"].iloc[0]
    elm_rmse = perf.loc[perf["model"] == "ELM_SIRMMM", "RMSE"].iloc[0]
    improvement = 100.0 * (plain_rmse - elm_rmse) / max(plain_rmse, EPS)

    beta_s = coefficients.loc[coefficients["parameter"] == "beta_sentiment", "value"].iloc[0]
    beta_e = coefficients.loc[coefficients["parameter"] == "beta_engagement", "value"].iloc[0]
    beta_c = coefficients.loc[coefficients["parameter"] == "beta_cognition", "value"].iloc[0]

    elm_peak = traj.loc[traj["model"] == "ELM_SIRMMM", "peak_MI_percent_population"].iloc[0]
    elm_peak_date = traj.loc[traj["model"] == "ELM_SIRMMM", "peak_MI_calendar_date"].iloc[0]
    mr_end = traj.loc[traj["model"] == "ELM_SIRMMM", "MR_end_percent"].iloc[0]
    ms_end = traj.loc[traj["model"] == "ELM_SIRMMM", "MS_end_percent"].iloc[0]

    text = f"""
{dataset_name} branch interpretation within the tri-dataset ELM-SIRMMM experiment

Dataset role:
{config.notes}

Signal construction:
Engagement: {config.engagement_mode_label}
Incidence: {config.incidence_mode_label}

The plain SIRMMM model achieved RMSE = {plain_rmse:.4f}.
The ELM-SIRMMM model achieved RMSE = {elm_rmse:.4f}.
This corresponds to an RMSE change of {improvement:.2f} percent relative to the fixed-rate baseline.

The fitted ELM coefficients were:
beta_sentiment = {beta_s:.4f}
beta_engagement = {beta_e:.4f}
beta_cognition = {beta_c:.4f}

These coefficients should be interpreted as conditional associations within beta_m(t), not as causal effects. The ELM trajectory reached its peak MI value on {elm_peak_date}, with peak MI equal to {elm_peak:.4f} percent of the modelling population. At the final simulated timestep, MR accounted for {mr_end:.4f} percent and MS accounted for {ms_end:.4f} percent of the modelling population.

For reporting, the defensible interpretation is that this dataset provides one context-specific test of whether sentiment, engagement and cognitive effort provide enough temporal signal to modulate the misinformation transmission rate. If the robustness checks show stable coefficient signs and similar RMSE behaviour, the coefficient-level interpretation can be discussed cautiously. If signs vary across proxy definitions, interpretation should focus on trajectory-level behaviour, fit improvement and sensitivity.
""".strip()

    with open(out_dir / "interpretation_summary.txt", "w", encoding="utf-8") as f:
        f.write(text)

    return text

# ============================================================
# 13. Process one dataset
# ============================================================

def process_dataset(config: DatasetConfig):
    dataset_dir = OUTPUT_ROOT / normalise_name(config.name)
    dataset_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 80)
    print(f"Processing dataset: {config.name}")
    print("=" * 80)

    file_path = upload_data_if_needed(config.default_file, config.name)
    raw_df = try_read_table(file_path)

    prepared = prepare_dataset(raw_df, config)
    df = prepared["df"]
    mapping = prepared["mapping"]

    with open(dataset_dir / "column_mapping.json", "w", encoding="utf-8") as f:
        json.dump(mapping, f, indent=2, default=str)

    dataset_summary = {
        "dataset": config.name,
        "file_path": str(file_path),
        "n_rows_raw": int(len(raw_df)),
        "n_rows_prepared": int(len(df)),
        "n_misinfo_rows": int(df["is_misinfo"].sum()),
        "date_min": str(df["date_day"].min()),
        "date_max": str(df["date_day"].max()),
        "engagement_mode": config.engagement_mode_label,
        "incidence_mode": config.incidence_mode_label,
        "notes": config.notes,
    }

    with open(dataset_dir / "dataset_summary.json", "w", encoding="utf-8") as f:
        json.dump(dataset_summary, f, indent=2, default=str)

    pd.DataFrame([dataset_summary]).to_csv(dataset_dir / "dataset_summary.csv", index=False)
    df.to_csv(dataset_dir / "prepared_rows.csv", index=False)

    print(f"[{config.name}] Aggregating daily signals...")
    daily = aggregate_daily(df, config)
    daily = add_normalised_features(daily, scaler_type="zscore")
    daily.to_csv(dataset_dir / "daily_aggregated.csv", index=False)

    print(f"[{config.name}] Daily series length: {len(daily)}")
    print(daily.head())

    print(f"[{config.name}] Fitting plain SIRMMM...")
    plain_fit = fit_plain_model(daily)

    print(f"[{config.name}] Fitting ELM-SIRMMM...")
    elm_fit = fit_elm_model(daily)

    print(f"[{config.name}] Building results tables...")
    perf, traj, coefficients, daily_out = build_results_tables(
        config.name,
        daily,
        plain_fit,
        elm_fit,
        dataset_dir
    )

    print(f"[{config.name}] Running robustness checks...")
    robustness = run_robustness(df, config, dataset_dir)

    print(f"[{config.name}] Generating plots...")
    make_all_plots(config.name, daily_out, dataset_dir)

    print(f"[{config.name}] Writing interpretation...")
    interpretation = write_dataset_interpretation(
        config,
        perf,
        traj,
        coefficients,
        robustness,
        dataset_dir
    )

    print(f"\n[{config.name}] Performance summary")
    print(perf)

    print(f"\n[{config.name}] Trajectory summary")
    print(traj)

    print(f"\n[{config.name}] ELM coefficients")
    print(coefficients)

    return {
        "dataset": config.name,
        "dataset_dir": dataset_dir,
        "summary": dataset_summary,
        "performance": perf,
        "trajectory": traj,
        "coefficients": coefficients,
        "robustness": robustness,
        "daily_out": daily_out,
        "interpretation": interpretation,
    }

# ============================================================
# 14. Combined report-ready tables
# ============================================================

def write_combined_outputs(results):
    combined_dir = OUTPUT_ROOT / "combined"
    combined_dir.mkdir(parents=True, exist_ok=True)

    all_perf = pd.concat([r["performance"] for r in results], ignore_index=True)
    all_traj = pd.concat([r["trajectory"] for r in results], ignore_index=True)
    all_coef = pd.concat([r["coefficients"] for r in results], ignore_index=True)
    all_robust = pd.concat([r["robustness"] for r in results], ignore_index=True)

    all_perf.to_csv(combined_dir / "all_performance_summary.csv", index=False)
    all_traj.to_csv(combined_dir / "all_trajectory_summary.csv", index=False)
    all_coef.to_csv(combined_dir / "all_elm_coefficients.csv", index=False)
    all_robust.to_csv(combined_dir / "all_robustness_checks.csv", index=False)

    report_fit_table = (
        all_perf
        .pivot(index="dataset", columns="model", values="RMSE")
        .reset_index()
        .rename(columns={
            "Plain_SIRMMM": "Plain_RMSE",
            "ELM_SIRMMM": "ELM_RMSE"
        })
    )

    report_fit_table["RMSE_improvement_percent"] = (
        100.0 * (report_fit_table["Plain_RMSE"] - report_fit_table["ELM_RMSE"])
        / report_fit_table["Plain_RMSE"].clip(lower=EPS)
    )

    report_fit_table.to_csv(combined_dir / "report_fit_table.csv", index=False)

    report_traj_table = all_traj[all_traj["model"] == "ELM_SIRMMM"].copy()

    keep_cols = [
        "dataset",
        "peak_MI_calendar_date",
        "peak_MI_day_index",
        "peak_MI_percent_population",
        "MR_end_percent",
        "MS_end_percent",
        "MI_end_percent",
        "correction_efficiency_MR_end_over_MI_peak"
    ]

    report_traj_table = report_traj_table[keep_cols]
    report_traj_table.to_csv(combined_dir / "report_elm_trajectory_table.csv", index=False)

    coef_wide = all_coef.pivot(index="dataset", columns="parameter", values="value").reset_index()
    coef_wide.to_csv(combined_dir / "report_elm_coefficients_wide.csv", index=False)

    with open(combined_dir / "report_fit_table.tex", "w", encoding="utf-8") as f:
        f.write(report_fit_table.to_latex(index=False, float_format="%.4f"))

    with open(combined_dir / "report_elm_trajectory_table.tex", "w", encoding="utf-8") as f:
        f.write(report_traj_table.to_latex(index=False, float_format="%.4f"))

    with open(combined_dir / "report_elm_coefficients_wide.tex", "w", encoding="utf-8") as f:
        f.write(coef_wide.to_latex(index=False, float_format="%.4f"))

    narrative = """
Tri-dataset ELM-SIRMMM summary

This experiment evaluates the ELM-SIRMMM framework across FibVID, MC-Fake and Monant. FibVID anchors the main within-dataset comparison between fixed-rate SIRMMM and ELM-conditioned SIRMMM. MC-Fake tests the same behavioural modulation framework in a high-volatility cascade environment with claim-level retweet and reply activity. Monant tests whether the ELM layer remains functionally active when behavioural signals are sparse, low-variance or forum-like.

The combined outputs should be interpreted at two levels. First, RMSE and trajectory summaries indicate whether ELM-conditioned beta_m(t) improves fit relative to a fixed-rate misinformation transmission process. Second, fitted coefficients indicate how sentiment, engagement and cognition enter beta_m(t), but these coefficients are conditional associations rather than causal effects. Robustness checks should therefore be used to decide whether coefficient-level interpretation is stable enough for formal discussion.
""".strip()

    with open(combined_dir / "combined_report_summary.txt", "w", encoding="utf-8") as f:
        f.write(narrative)

    return {
        "combined_dir": combined_dir,
        "all_performance": all_perf,
        "all_trajectory": all_traj,
        "all_coefficients": all_coef,
        "all_robustness": all_robust,
        "report_fit_table": report_fit_table,
        "report_traj_table": report_traj_table,
        "coef_wide": coef_wide,
    }

# ============================================================
# 15. Main execution
# ============================================================

def main():
    results = []

    for config in DATASET_CONFIGS:
        try:
            result = process_dataset(config)
            results.append(result)

        except Exception as exc:
            print("\n" + "!" * 80)
            print(f"FAILED: {config.name}")
            print("Reason:", str(exc))
            print("!" * 80)

            error_dir = OUTPUT_ROOT / normalise_name(config.name)
            error_dir.mkdir(parents=True, exist_ok=True)

            with open(error_dir / "error_log.txt", "w", encoding="utf-8") as f:
                f.write(str(exc))

    if not results:
        raise RuntimeError("No datasets were processed successfully.")

    combined = write_combined_outputs(results)

    print("\n" + "=" * 80)
    print("Combined report-ready outputs saved to:")
    print(combined["combined_dir"])
    print("=" * 80)

    print("\nFit table")
    print(combined["report_fit_table"])

    print("\nELM trajectory table")
    print(combined["report_traj_table"])

    print("\nELM coefficients wide table")
    print(combined["coef_wide"])

    print("\nSaved files:")
    for p in sorted(OUTPUT_ROOT.rglob("*")):
        if p.is_file():
            print(" -", p)

    try:
        from IPython.display import display
        display(combined["report_fit_table"])
        display(combined["report_traj_table"])
        display(combined["coef_wide"])
    except Exception:
        pass


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# BERTopic-VP pipeline
# Datasets: COVID--19_FNIR, Constraint, and Monkeypox
#
# Unit of analysis: one social media post or tweet
#
# VP signal:
# - Monkeypox: observed engagement E_i = log(1 + likes + retweets + replies + quotes)
# - COVID--19_FNIR: proxy propensity score eta_i transferred from Monkeypox
# - Constraint: proxy propensity score eta_i transferred from Monkeypox
#
# ============================================================

import os
import re
import csv
import sys
import json
import math
import random
import zipfile
import warnings
import subprocess
import importlib.util
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any

# ============================================================
# 0. Install packages
# ============================================================

PACKAGE_IMPORTS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "scikit-learn": "sklearn",
    "sentence-transformers": "sentence_transformers",
    "bertopic": "bertopic",
    "umap-learn": "umap",
    "hdbscan": "hdbscan",
    "textblob": "textblob",
    "ftfy": "ftfy",
    "textstat": "textstat",
    "gensim": "gensim",
}

for package_name, module_name in PACKAGE_IMPORTS.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            check=True
        )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ftfy import fix_text
from textblob import TextBlob
import textstat

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import normalized_mutual_info_score, adjusted_rand_score
from sklearn.metrics.cluster import contingency_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

warnings.filterwarnings("ignore")

try:
    from gensim.corpora import Dictionary
    from gensim.models import CoherenceModel
    GENSIM_AVAILABLE = True
except Exception:
    GENSIM_AVAILABLE = False

# ============================================================
# 1. Global configuration
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("bertopic_vp_covid_constraint_monkeypox_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

VP_THRESHOLDS = [1, 5, 10]

FILTER_MISINFO_ONLY = False

HIGH_ENGAGEMENT_QUANTILE = 0.10

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_N_COMPONENTS = 5

HDBSCAN_MIN_CLUSTER_SIZE = 50
HDBSCAN_MIN_SAMPLES = 10

EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

MISINFO_NUMERIC_VALUES = {1}

MISINFO_STRING_VALUES = {
    "1", "fake", "false", "misinformation", "misinfo",
    "misleading", "rumour", "rumor", "unverified"
}

FACTUAL_STRING_VALUES = {
    "0", "real", "true", "factual", "fact", "not misinformation",
    "non-misinfo", "non_misinfo", "not fake"
}

# ============================================================
# 2. Dataset configuration
# ============================================================

@dataclass
class DatasetConfig:
    name: str
    file_name: str

    text_candidates: List[str]
    label_candidates: List[str]
    date_candidates: List[str] = field(default_factory=list)

    engagement_candidates: List[str] = field(default_factory=list)

    uses_observed_engagement: bool = False
    vp_signal_type: str = "proxy"

    notes: str = ""


DATASETS = [
    DatasetConfig(
        name="COVID--19_FNIR",
        file_name="covid19_fnir.csv",
        text_candidates=[
            "tweet", "text", "tweet_text", "clean_text", "content",
            "statement", "claim", "body", "title"
        ],
        label_candidates=[
            "label", "labels", "class", "target", "veracity",
            "is_fake", "misinformation"
        ],
        date_candidates=[
            "date", "created_at", "timestamp", "tweet_date",
            "publish_date", "created"
        ],
        engagement_candidates=[],
        uses_observed_engagement=False,
        vp_signal_type="proxy_eta",
        notes="COVID--19_FNIR has no native engagement metadata in the released schema, so VP uses transferred proxy eta_i."
    ),

    DatasetConfig(
        name="Constraint",
        file_name="constraint.csv",
        text_candidates=[
            "tweet", "text", "tweet_text", "clean_text", "content",
            "statement", "claim", "body"
        ],
        label_candidates=[
            "label", "labels", "class", "target", "veracity",
            "is_fake", "misinformation"
        ],
        date_candidates=[
            "date", "created_at", "timestamp", "tweet_date",
            "publish_date", "created"
        ],
        engagement_candidates=[],
        uses_observed_engagement=False,
        vp_signal_type="proxy_eta",
        notes="Constraint has no native engagement metadata in the released schema, so VP uses transferred proxy eta_i."
    ),

    DatasetConfig(
        name="Monkeypox",
        file_name="monkeypox.csv",
        text_candidates=[
            "tweet", "text", "tweet_text", "clean_text", "content",
            "statement", "claim", "body", "Tweet", "Tweet Text"
        ],
        label_candidates=[
            "Binary Label", "binary_label", "label", "labels",
            "class", "target", "veracity", "is_fake", "misinformation"
        ],
        date_candidates=[
            "date", "created_at", "timestamp", "tweet_date",
            "publish_date", "created"
        ],
        engagement_candidates=[
            "likes", "like_count", "favorite_count", "favourites", "favorites",
            "retweets", "retweet_count", "reposts",
            "replies", "reply_count", "comments",
            "quotes", "quote_count"
        ],
        uses_observed_engagement=True,
        vp_signal_type="observed_engagement",
        notes="Monkeypox contains observed engagement metadata and is used to train the proxy propensity model."
    ),
]

# ============================================================
# 3. Utility functions
# ============================================================

def normalise_colname(x):
    return re.sub(r"[^a-z0-9]+", "_", str(x).strip().lower()).strip("_")


def find_first_column(df, candidates):
    col_map = {normalise_colname(c): c for c in df.columns}
    candidate_norms = [normalise_colname(c) for c in candidates]

    for cand in candidate_norms:
        if cand in col_map:
            return col_map[cand]

    for cand in candidate_norms:
        for norm_col, original_col in col_map.items():
            if cand == norm_col or cand in norm_col or norm_col in cand:
                return original_col

    return None


def find_matching_columns(df, candidates):
    col_map = {normalise_colname(c): c for c in df.columns}
    candidate_norms = [normalise_colname(c) for c in candidates]
    matches = []

    for cand in candidate_norms:
        for norm_col, original_col in col_map.items():
            if cand == norm_col or cand in norm_col or norm_col in cand:
                if original_col not in matches:
                    matches.append(original_col)

    return matches


def get_data_file(file_name):
    local_path = Path(file_name)
    data_path = DATA_DIR / file_name

    if local_path.exists():
        return str(local_path)

    if data_path.exists():
        return str(data_path)

    try:
        from google.colab import files
        print(f"Upload dataset file: {file_name}")
        uploaded = files.upload()

        if not uploaded:
            raise FileNotFoundError("No file uploaded.")

        uploaded_name = list(uploaded.keys())[0]
        uploaded_path = Path(uploaded_name)
        destination = DATA_DIR / uploaded_name

        if destination.exists():
            destination.unlink()

        uploaded_path.rename(destination)
        return str(destination)

    except Exception as exc:
        raise FileNotFoundError(
            f"No dataset found for {file_name}. Place it in the working directory or in ./data/."
        ) from exc


def load_table(file_path):
    file_path = Path(file_path)
    attempts = []

    for sep in ["\t", ",", ";", "|", None]:
        for enc in ["utf-8", "latin1", "cp1252"]:
            try:
                df = pd.read_csv(
                    file_path,
                    sep=sep,
                    dtype=str,
                    encoding=enc,
                    engine="python",
                    on_bad_lines="skip",
                    quoting=csv.QUOTE_MINIMAL
                )

                if df.shape[1] > 1:
                    attempts.append((sep, enc, df))
            except Exception:
                pass

    if not attempts:
        raise ValueError(f"Could not read file: {file_path}")

    sep, enc, df = max(attempts, key=lambda item: item[2].shape[1])
    df.columns = [str(c).strip() for c in df.columns]

    print(f"Loaded {file_path} with separator={repr(sep)}, encoding={enc}, shape={df.shape}")
    print("Columns:", list(df.columns)[:40])

    return df.copy()


def clean_text(x):
    if pd.isna(x):
        return ""

    x = str(x)
    x = fix_text(x)

    x = re.sub(r"http\S+|www\.\S+", " URL ", x)
    x = re.sub(r"@\w+", " USER ", x)
    x = re.sub(r"#", "", x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"\s+", " ", x).strip()

    return x


def to_numeric_safe(series, default=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(default).astype(float)


def parse_dates_safe(series):
    parsed = pd.to_datetime(series, errors="coerce", dayfirst=False)

    if parsed.isna().mean() > 0.50:
        parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)

    return parsed


def combine_text_columns(df, text_cols):
    docs = []

    for _, row in df.iterrows():
        parts = []

        for col in text_cols:
            value = clean_text(row.get(col, ""))
            if value:
                parts.append(value)

        docs.append(" ".join(parts).strip())

    return pd.Series(docs, index=df.index)


def sentiment_score(text):
    try:
        return float(TextBlob(str(text)).sentiment.polarity)
    except Exception:
        return 0.0


def word_count(text):
    return len(re.findall(r"\b\w+\b", str(text)))


def type_token_ratio(text):
    tokens = re.findall(r"\b\w+\b", str(text).lower())
    if len(tokens) == 0:
        return 0.0
    return len(set(tokens)) / len(tokens)


def uppercase_ratio(text):
    chars = [c for c in str(text) if c.isalpha()]
    if len(chars) == 0:
        return 0.0
    return sum(1 for c in chars if c.isupper()) / len(chars)


def punctuation_count(text, punct):
    return str(text).count(punct)


def urgency_count(text):
    terms = {
        "urgent", "now", "immediately", "breaking", "warning",
        "alert", "must", "share", "wake up", "truth"
    }

    text_l = str(text).lower()
    return sum(text_l.count(term) for term in terms)


def flesch_reading_ease_safe(text):
    try:
        return float(textstat.flesch_reading_ease(text))
    except Exception:
        return np.nan


def flesch_kincaid_safe(text):
    try:
        return float(textstat.flesch_kincaid_grade(text))
    except Exception:
        return np.nan


def normalise_label_value(x):
    if pd.isna(x):
        return np.nan

    raw = str(x).strip()
    raw_l = raw.lower()

    numeric = pd.to_numeric(raw, errors="coerce")
    if not pd.isna(numeric):
        if int(numeric) in MISINFO_NUMERIC_VALUES:
            return 1
        if int(numeric) == 0:
            return 0

    if raw_l in MISINFO_STRING_VALUES:
        return 1

    if raw_l in FACTUAL_STRING_VALUES:
        return 0

    return np.nan


def purity_score(y_true, y_pred):
    cm = contingency_matrix(y_true, y_pred)
    return np.sum(np.amax(cm, axis=0)) / np.sum(cm)


def tokenise_for_coherence(text):
    tokens = re.findall(r"\b[a-zA-Z]{3,}\b", str(text).lower())
    stop = {
        "the", "and", "for", "with", "that", "this", "you", "are", "was",
        "were", "from", "have", "has", "had", "not", "but", "they", "their",
        "will", "can", "about", "into", "your", "our", "out", "all"
    }
    return [t for t in tokens if t not in stop]

# ============================================================
# 4. Dataset preparation
# ============================================================

def prepare_dataset(raw_df, config):
    df = raw_df.copy()

    text_cols = find_matching_columns(df, config.text_candidates)
    label_col = find_first_column(df, config.label_candidates)
    date_col = find_first_column(df, config.date_candidates)
    engagement_cols = find_matching_columns(df, config.engagement_candidates)

    if not text_cols:
        raise ValueError(f"{config.name}: no text columns found.")

    if label_col is None:
        raise ValueError(f"{config.name}: no label column found.")

    print(f"\n{config.name} column mapping")
    print("Text columns:", text_cols)
    print("Label column:", label_col)
    print("Date column:", date_col)
    print("Engagement columns:", engagement_cols)

    df["doc_text"] = combine_text_columns(df, text_cols)
    df["label_num"] = df[label_col].map(normalise_label_value)

    if date_col is not None:
        df["date"] = parse_dates_safe(df[date_col])
        df["date_day"] = df["date"].dt.floor("D")
    else:
        df["date"] = pd.NaT
        df["date_day"] = pd.NaT

    if config.uses_observed_engagement:
        if not engagement_cols:
            raise ValueError(
                f"{config.name}: observed engagement is required, but no engagement columns were detected."
            )

        engagement_total = pd.Series(0.0, index=df.index)
        for col in engagement_cols:
            engagement_total += to_numeric_safe(df[col], default=0.0)

        df["engagement_raw"] = engagement_total.clip(lower=0)
        df["observed_E"] = np.log1p(df["engagement_raw"])
    else:
        df["engagement_raw"] = np.nan
        df["observed_E"] = np.nan

    df["sentiment"] = df["doc_text"].map(sentiment_score)
    df["word_count"] = df["doc_text"].map(word_count)
    df["ttr"] = df["doc_text"].map(type_token_ratio)
    df["uppercase_ratio"] = df["doc_text"].map(uppercase_ratio)
    df["exclamation_count"] = df["doc_text"].map(lambda x: punctuation_count(x, "!"))
    df["question_count"] = df["doc_text"].map(lambda x: punctuation_count(x, "?"))
    df["urgency_count"] = df["doc_text"].map(urgency_count)
    df["flesch_reading_ease"] = df["doc_text"].map(flesch_reading_ease_safe)
    df["flesch_kincaid_grade"] = df["doc_text"].map(flesch_kincaid_safe)

    df["flesch_reading_ease"] = df["flesch_reading_ease"].fillna(df["flesch_reading_ease"].median())
    df["flesch_kincaid_grade"] = df["flesch_kincaid_grade"].fillna(df["flesch_kincaid_grade"].median())

    df = df[df["doc_text"].str.len() > 5].copy()
    df = df[df["label_num"].notna()].copy()

    if FILTER_MISINFO_ONLY:
        df = df[df["label_num"] == 1].copy()

    df = df.reset_index(drop=True)

    print(f"{config.name} prepared shape:", df.shape)
    print(f"{config.name} label counts:")
    print(df["label_num"].value_counts(dropna=False))

    mapping = {
        "dataset": config.name,
        "text_cols": text_cols,
        "label_col": label_col,
        "date_col": date_col,
        "engagement_cols": engagement_cols,
        "vp_signal_type": config.vp_signal_type,
        "notes": config.notes
    }

    return df, mapping

# ============================================================
# 5. Propensity model for proxy VP
# ============================================================

PROXY_FEATURES = [
    "sentiment",
    "word_count",
    "ttr",
    "uppercase_ratio",
    "exclamation_count",
    "question_count",
    "urgency_count",
    "flesch_reading_ease",
    "flesch_kincaid_grade",
]


def fit_propensity_model(monkeypox_df):
    if "observed_E" not in monkeypox_df.columns:
        raise ValueError("Monkeypox data must contain observed_E for proxy training.")

    if monkeypox_df["observed_E"].notna().sum() == 0:
        raise ValueError("Monkeypox observed_E is empty. Check engagement columns.")

    threshold = np.nanquantile(monkeypox_df["observed_E"], 1 - HIGH_ENGAGEMENT_QUANTILE)
    y = (monkeypox_df["observed_E"] >= threshold).astype(int)

    if y.nunique() < 2:
        raise ValueError(
            "High-engagement training target has one class only. Check Monkeypox engagement distribution."
        )

    X = monkeypox_df[PROXY_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED
        ))
    ])

    model.fit(X, y)

    auc_note = {
        "high_engagement_quantile": HIGH_ENGAGEMENT_QUANTILE,
        "threshold": float(threshold),
        "positive_rate": float(y.mean()),
        "n_training_rows": int(len(monkeypox_df))
    }

    return model, auc_note


def apply_vp_signal(df, config, propensity_model=None):
    df = df.copy()

    if config.uses_observed_engagement:
        df["vp_signal"] = df["observed_E"].astype(float)
        df["vp_signal_source"] = "observed_engagement_E_i"
        df["proxy_eta"] = np.nan
        df["proxy_probability"] = np.nan
    else:
        if propensity_model is None:
            raise ValueError(f"{config.name}: propensity model required for proxy VP signal.")

        X = df[PROXY_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

        eta = propensity_model.decision_function(X)
        prob = propensity_model.predict_proba(X)[:, 1]

        df["proxy_eta"] = eta
        df["proxy_probability"] = prob
        df["vp_signal"] = eta
        df["vp_signal_source"] = "proxy_eta_from_monkeypox_model"

    return df

# ============================================================
# 6. BERTopic fitting
# ============================================================

def fit_bertopic_vp(df, dataset_name):
    docs = df["doc_text"].tolist()
    n_docs = len(docs)

    if n_docs < 10:
        raise ValueError(f"{dataset_name}: too few documents for BERTopic.")

    n_neighbors = min(UMAP_N_NEIGHBORS, max(2, n_docs - 1))
    min_cluster_size = min(HDBSCAN_MIN_CLUSTER_SIZE, max(2, n_docs // 5))
    min_samples = min(HDBSCAN_MIN_SAMPLES, max(1, min_cluster_size // 2))
    min_df = 1 if n_docs < 100 else 2

    print(f"\nFitting BERTopic-VP for {dataset_name}")
    print("n_docs:", n_docs)
    print("UMAP n_neighbors:", n_neighbors)
    print("HDBSCAN min_cluster_size:", min_cluster_size)
    print("HDBSCAN min_samples:", min_samples)

    embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    umap_model = UMAP(
        n_neighbors=n_neighbors,
        n_components=UMAP_N_COMPONENTS,
        min_dist=UMAP_MIN_DIST,
        metric="cosine",
        random_state=SEED
    )

    hdbscan_model = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True
    )

    vectorizer_model = CountVectorizer(
        stop_words="english",
        ngram_range=(1, 2),
        min_df=min_df
    )

    topic_model = BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=True
    )

    topics, probs = topic_model.fit_transform(docs)

    df_out = df.copy()
    df_out["topic"] = topics

    return topic_model, df_out

# ============================================================
# 7. VP summaries
# ============================================================

def get_topic_terms(topic_model, topic_id, top_n=8):
    if topic_id == -1:
        return "outlier"

    topic = topic_model.get_topic(topic_id)

    if not topic:
        return ""

    return ", ".join([term for term, weight in topic[:top_n]])


def add_vp_flags(df):
    df = df.copy()

    for x in VP_THRESHOLDS:
        threshold = np.nanquantile(df["vp_signal"], 1 - x / 100)
        df[f"vp_top_{x}pct"] = df["vp_signal"] >= threshold

    return df


def build_cluster_summary(topic_model, df_out, dataset_name):
    df = add_vp_flags(df_out)
    non_outlier = df[df["topic"] != -1].copy()

    if len(non_outlier) == 0:
        return df, pd.DataFrame()

    cluster = (
        non_outlier.groupby("topic")
        .agg(
            n_items=("topic", "size"),
            mean_vp=("vp_signal", "mean"),
            median_vp=("vp_signal", "median"),
            max_vp=("vp_signal", "max"),
            misinfo_rate=("label_num", lambda s: np.mean(s == 1)),
            mean_sentiment=("sentiment", "mean"),
            mean_word_count=("word_count", "mean"),
            mean_ttr=("ttr", "mean"),
            mean_flesch_reading_ease=("flesch_reading_ease", "mean"),
            mean_flesch_kincaid_grade=("flesch_kincaid_grade", "mean"),
            date_min=("date_day", "min"),
            date_max=("date_day", "max")
        )
        .reset_index()
    )

    for x in VP_THRESHOLDS:
        v_col = f"vp_top_{x}pct"

        density = non_outlier.groupby("topic")[v_col].mean().reset_index()
        density = density.rename(columns={v_col: f"viral_density_top_{x}pct"})
        cluster = cluster.merge(density, on="topic", how="left")

        has_flag = non_outlier.groupby("topic")[v_col].max().reset_index()
        has_flag = has_flag.rename(columns={v_col: f"has_vp_item_top_{x}pct"})
        cluster = cluster.merge(has_flag, on="topic", how="left")

        cluster_threshold = np.nanquantile(cluster["mean_vp"], 1 - x / 100)
        cluster[f"cluster_priority_top_{x}pct"] = cluster["mean_vp"] >= cluster_threshold

    cluster["top_terms"] = cluster["topic"].map(lambda t: get_topic_terms(topic_model, t))
    cluster["dataset"] = dataset_name

    cluster = cluster.sort_values("mean_vp", ascending=False).reset_index(drop=True)

    return df, cluster


def build_vp_count_table(cluster, dataset_name, signal_source):
    rows = []

    total_clusters = len(cluster)

    for x in VP_THRESHOLDS:
        rows.append({
            "dataset": dataset_name,
            "vp_signal_source": signal_source,
            "threshold": f"Top {x}%",
            "clusters_with_at_least_one_vp_item": int(cluster[f"has_vp_item_top_{x}pct"].sum()),
            "total_non_outlier_clusters": int(total_clusters),
            "share": float(cluster[f"has_vp_item_top_{x}pct"].mean()) if total_clusters > 0 else np.nan
        })

    return pd.DataFrame(rows)

# ============================================================
# 8. Clustering metrics and coherence
# ============================================================

def compute_clustering_metrics(df_out, dataset_name):
    non_outlier = df_out[df_out["topic"] != -1].copy()

    if len(non_outlier) == 0:
        return pd.DataFrame([{
            "dataset": dataset_name,
            "purity": np.nan,
            "nmi": np.nan,
            "ari": np.nan,
            "n_clusters": 0,
            "noise_rate": 1.0
        }])

    y_true = non_outlier["label_num"].astype(int).values
    y_pred = non_outlier["topic"].astype(int).values

    if len(np.unique(y_true)) < 2 or len(np.unique(y_pred)) < 2:
        purity = np.nan
        nmi = np.nan
        ari = np.nan
    else:
        purity = purity_score(y_true, y_pred)
        nmi = normalized_mutual_info_score(y_true, y_pred)
        ari = adjusted_rand_score(y_true, y_pred)

    metrics = {
        "dataset": dataset_name,
        "purity": purity,
        "nmi": nmi,
        "ari": ari,
        "n_clusters": int(non_outlier["topic"].nunique()),
        "noise_rate": float((df_out["topic"] == -1).mean())
    }

    return pd.DataFrame([metrics])


def compute_topic_coherence(topic_model, df_out, dataset_name, top_n=10):
    if not GENSIM_AVAILABLE:
        return pd.DataFrame([{
            "dataset": dataset_name,
            "mean_cv_coherence": np.nan,
            "n_topics_used": 0,
            "note": "gensim unavailable"
        }])

    non_outlier = df_out[df_out["topic"] != -1].copy()

    if len(non_outlier) == 0:
        return pd.DataFrame([{
            "dataset": dataset_name,
            "mean_cv_coherence": np.nan,
            "n_topics_used": 0,
            "note": "no non-outlier topics"
        }])

    tokenised_docs = [tokenise_for_coherence(t) for t in non_outlier["doc_text"].tolist()]
    tokenised_docs = [toks for toks in tokenised_docs if len(toks) > 0]

    if len(tokenised_docs) == 0:
        return pd.DataFrame([{
            "dataset": dataset_name,
            "mean_cv_coherence": np.nan,
            "n_topics_used": 0,
            "note": "empty tokenised docs"
        }])

    topic_words = []

    for topic_id in sorted(non_outlier["topic"].unique()):
        topic = topic_model.get_topic(topic_id)
        if topic:
            words = [term for term, weight in topic[:top_n]]
            words = [w.replace(" ", "_") for w in words if isinstance(w, str)]
            if len(words) >= 2:
                topic_words.append(words)

    if len(topic_words) == 0:
        return pd.DataFrame([{
            "dataset": dataset_name,
            "mean_cv_coherence": np.nan,
            "n_topics_used": 0,
            "note": "no valid topic words"
        }])

    dictionary = Dictionary(tokenised_docs)

    try:
        coherence_model = CoherenceModel(
            topics=topic_words,
            texts=tokenised_docs,
            dictionary=dictionary,
            coherence="c_v"
        )

        per_topic = coherence_model.get_coherence_per_topic()
        mean_coherence = float(np.mean(per_topic))

        return pd.DataFrame([{
            "dataset": dataset_name,
            "mean_cv_coherence": mean_coherence,
            "n_topics_used": len(topic_words),
            "note": "computed"
        }])

    except Exception as exc:
        return pd.DataFrame([{
            "dataset": dataset_name,
            "mean_cv_coherence": np.nan,
            "n_topics_used": len(topic_words),
            "note": str(exc)
        }])

# ============================================================
# 9. Representative documents
# ============================================================

def build_representative_docs(df_flagged, cluster, dataset_name, top_n_topics=20, top_n_per_topic=3):
    rows = []

    if cluster.empty:
        return pd.DataFrame()

    top_topics = cluster.head(top_n_topics)["topic"].tolist()

    for topic in top_topics:
        sub = (
            df_flagged[df_flagged["topic"] == topic]
            .sort_values("vp_signal", ascending=False)
            .head(top_n_per_topic)
        )

        terms = cluster.loc[cluster["topic"] == topic, "top_terms"].iloc[0]

        for _, row in sub.iterrows():
            rows.append({
                "dataset": dataset_name,
                "topic": topic,
                "top_terms": terms,
                "label_num": row.get("label_num", ""),
                "vp_signal": row.get("vp_signal", ""),
                "vp_signal_source": row.get("vp_signal_source", ""),
                "observed_E": row.get("observed_E", ""),
                "proxy_eta": row.get("proxy_eta", ""),
                "proxy_probability": row.get("proxy_probability", ""),
                "date": row.get("date_day", ""),
                "text_excerpt": str(row.get("doc_text", ""))[:400]
            })

    return pd.DataFrame(rows)

# ============================================================
# 10. Plots
# ============================================================

def plot_top_topics(cluster, dataset_name, out_dir):
    if cluster.empty:
        return

    top = cluster.head(15).copy()

    plt.figure(figsize=(10, 6))
    plt.barh(top["topic"].astype(str), top["mean_vp"])
    plt.xlabel("Mean VP signal")
    plt.ylabel("Topic")
    plt.title(f"{dataset_name}: top topics by BERTopic-VP signal")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(out_dir / f"{normalise_colname(dataset_name)}_top_topics_by_vp.png", dpi=300)
    plt.close()


def plot_topic_volume_vs_vp(cluster, dataset_name, out_dir):
    if cluster.empty:
        return

    plt.figure(figsize=(8, 6))
    plt.scatter(cluster["n_items"], cluster["mean_vp"])
    plt.xlabel("Cluster size")
    plt.ylabel("Mean VP signal")
    plt.title(f"{dataset_name}: topic size versus VP signal")
    plt.tight_layout()
    plt.savefig(out_dir / f"{normalise_colname(dataset_name)}_topic_size_vs_vp.png", dpi=300)
    plt.close()


def plot_daily_vp(df_flagged, dataset_name, out_dir):
    if "date_day" not in df_flagged.columns:
        return

    dated = df_flagged.dropna(subset=["date_day"]).copy()

    if len(dated) == 0:
        return

    daily = (
        dated.groupby("date_day")
        .agg(mean_vp=("vp_signal", "mean"), n_posts=("doc_text", "count"))
        .reset_index()
        .sort_values("date_day")
    )

    plt.figure(figsize=(10, 6))
    plt.plot(daily["date_day"], daily["mean_vp"])
    plt.xlabel("Date")
    plt.ylabel("Mean VP signal")
    plt.title(f"{dataset_name}: daily mean VP signal")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(out_dir / f"{normalise_colname(dataset_name)}_daily_vp_signal.png", dpi=300)
    plt.close()


def make_plots(cluster, df_flagged, dataset_name, out_dir):
    plot_top_topics(cluster, dataset_name, out_dir)
    plot_topic_volume_vs_vp(cluster, dataset_name, out_dir)
    plot_daily_vp(df_flagged, dataset_name, out_dir)

# ============================================================
# 11. Dataset processing
# ============================================================

def process_single_dataset(config, prepared_df, propensity_model):
    dataset_dir = OUTPUT_DIR / normalise_colname(config.name)
    dataset_dir.mkdir(parents=True, exist_ok=True)

    df_scored = apply_vp_signal(
        prepared_df,
        config=config,
        propensity_model=propensity_model
    )

    df_scored.to_csv(dataset_dir / f"{normalise_colname(config.name)}_prepared_scored.csv", index=False)

    topic_model, df_topics = fit_bertopic_vp(df_scored, config.name)

    df_flagged, cluster = build_cluster_summary(
        topic_model=topic_model,
        df_out=df_topics,
        dataset_name=config.name
    )

    signal_source = df_flagged["vp_signal_source"].iloc[0] if len(df_flagged) else config.vp_signal_type

    vp_counts = build_vp_count_table(cluster, config.name, signal_source)
    metrics = compute_clustering_metrics(df_topics, config.name)
    coherence = compute_topic_coherence(topic_model, df_topics, config.name)
    reps = build_representative_docs(df_flagged, cluster, config.name)

    df_flagged.to_csv(dataset_dir / f"{normalise_colname(config.name)}_documents_with_topics_and_vp.csv", index=False)
    cluster.to_csv(dataset_dir / f"{normalise_colname(config.name)}_cluster_vp_summary.csv", index=False)
    vp_counts.to_csv(dataset_dir / f"{normalise_colname(config.name)}_vp_flagged_cluster_counts.csv", index=False)
    metrics.to_csv(dataset_dir / f"{normalise_colname(config.name)}_clustering_metrics.csv", index=False)
    coherence.to_csv(dataset_dir / f"{normalise_colname(config.name)}_coherence.csv", index=False)
    reps.to_csv(dataset_dir / f"{normalise_colname(config.name)}_representative_vp_topics.csv", index=False)

    try:
        topic_model.save(
            str(dataset_dir / f"bertopic_model_{normalise_colname(config.name)}"),
            serialization="safetensors"
        )
    except Exception as exc:
        print(f"{config.name}: model save skipped:", exc)

    make_plots(cluster, df_flagged, config.name, dataset_dir)

    return {
        "dataset": config.name,
        "dataset_dir": dataset_dir,
        "df_flagged": df_flagged,
        "cluster": cluster,
        "vp_counts": vp_counts,
        "metrics": metrics,
        "coherence": coherence,
        "representatives": reps
    }

# ============================================================
# 12. Combined output
# ============================================================

def write_combined_outputs(results, propensity_note, mappings):
    combined_dir = OUTPUT_DIR / "combined"
    combined_dir.mkdir(parents=True, exist_ok=True)

    all_clusters = pd.concat([r["cluster"] for r in results if not r["cluster"].empty], ignore_index=True)
    all_vp_counts = pd.concat([r["vp_counts"] for r in results], ignore_index=True)
    all_metrics = pd.concat([r["metrics"] for r in results], ignore_index=True)
    all_coherence = pd.concat([r["coherence"] for r in results], ignore_index=True)
    all_reps = pd.concat([r["representatives"] for r in results if not r["representatives"].empty], ignore_index=True)

    all_clusters.to_csv(combined_dir / "all_cluster_vp_summaries.csv", index=False)
    all_vp_counts.to_csv(combined_dir / "all_vp_flagged_cluster_counts.csv", index=False)
    all_metrics.to_csv(combined_dir / "all_clustering_metrics.csv", index=False)
    all_coherence.to_csv(combined_dir / "all_coherence_results.csv", index=False)
    all_reps.to_csv(combined_dir / "all_representative_vp_topics.csv", index=False)

    with open(combined_dir / "propensity_model_training_note.json", "w", encoding="utf-8") as f:
        json.dump(propensity_note, f, indent=2, default=str)

    with open(combined_dir / "column_mappings.json", "w", encoding="utf-8") as f:
        json.dump(mappings, f, indent=2, default=str)

    summary = f"""
BERTopic-VP corrected tri-dataset summary

This pipeline uses COVID--19_FNIR, Constraint and Monkeypox only.

Monkeypox uses observed engagement:
E_i = log(1 + available likes + retweets + replies + quotes).

COVID--19_FNIR and Constraint do not use observed platform engagement in this pipeline. Their VP signal is a transferred proxy score eta_i from a logistic propensity model trained on Monkeypox, where the positive class is defined as the top {HIGH_ENGAGEMENT_QUANTILE:.2f} share of observed engagement.

The VP outputs for Monkeypox therefore represent realised interaction counts. The VP outputs for COVID--19_FNIR and Constraint represent estimated diffusion propensity and should be interpreted as within-dataset rankings, not as observed virality magnitudes.

Primary combined files:
- all_cluster_vp_summaries.csv
- all_vp_flagged_cluster_counts.csv
- all_clustering_metrics.csv
- all_coherence_results.csv
- all_representative_vp_topics.csv
""".strip()

    with open(combined_dir / "corrected_bertopic_vp_summary.txt", "w", encoding="utf-8") as f:
        f.write(summary)

    zip_path = OUTPUT_DIR.with_suffix(".zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for file in OUTPUT_DIR.rglob("*"):
            if file.is_file():
                z.write(file, arcname=str(file.relative_to(OUTPUT_DIR)))

    print("\nCombined outputs saved to:", combined_dir)
    print("Zipped outputs:", zip_path)

    return {
        "combined_dir": combined_dir,
        "all_clusters": all_clusters,
        "all_vp_counts": all_vp_counts,
        "all_metrics": all_metrics,
        "all_coherence": all_coherence,
        "all_representatives": all_reps,
        "zip_path": zip_path
    }

# ============================================================
# 13. Main
# ============================================================

def main():
    prepared_data = {}
    mappings = {}

    for config in DATASETS:
        print("\n" + "=" * 80)
        print(f"Loading and preparing {config.name}")
        print("=" * 80)

        file_path = get_data_file(config.file_name)
        raw = load_table(file_path)

        prepared, mapping = prepare_dataset(raw, config)

        prepared_data[config.name] = prepared
        mappings[config.name] = mapping

    monkeypox_df = prepared_data["Monkeypox"]

    print("\n" + "=" * 80)
    print("Training Monkeypox-based propensity model for proxy VP")
    print("=" * 80)

    propensity_model, propensity_note = fit_propensity_model(monkeypox_df)

    results = []

    for config in DATASETS:
        print("\n" + "=" * 80)
        print(f"Running BERTopic-VP for {config.name}")
        print("=" * 80)

        result = process_single_dataset(
            config=config,
            prepared_df=prepared_data[config.name],
            propensity_model=propensity_model
        )

        results.append(result)

        print(f"\n{config.name} clustering metrics")
        print(result["metrics"])

        print(f"\n{config.name} coherence")
        print(result["coherence"])

        print(f"\n{config.name} VP counts")
        print(result["vp_counts"])

        if not result["cluster"].empty:
            print(f"\n{config.name} top VP-ranked clusters")
            print(result["cluster"][[
                "dataset", "topic", "n_items", "mean_vp",
                "viral_density_top_1pct", "viral_density_top_5pct",
                "top_terms", "misinfo_rate"
            ]].head(10))

    combined = write_combined_outputs(results, propensity_note, mappings)

    print("\n" + "=" * 80)
    print("Final combined metrics")
    print("=" * 80)

    print("\nClustering metrics")
    print(combined["all_metrics"])

    print("\nCoherence")
    print(combined["all_coherence"])

    print("\nVP flagged cluster counts")
    print(combined["all_vp_counts"])

    try:
        from IPython.display import display
        display(combined["all_metrics"])
        display(combined["all_coherence"])
        display(combined["all_vp_counts"])
        display(combined["all_clusters"].head(20))
        display(combined["all_representatives"].head(20))
    except Exception:
        pass


if __name__ == "__main__":
    main()

In [ ]:
# ============================================================
# Linguistic Patterns Pipeline
# Datasets:
#   1. COVID--19_FNIR
#   2. Constraint
#   3. Monkeypox
#
# Purpose:
#   Comparative linguistic analysis of readability, rhetorical
#   punctuation, persuasive/fear lexicon, sentiment, and engagement
#   or engagement-propensity ranking.
#
# Notes:
#   - Observed engagement is used only where native engagement fields exist.
#   - Monkeypox usually contains engagement metadata.
#   - COVID--19_FNIR and Constraint usually do not contain engagement metadata.
#   - For datasets without engagement, a proxy engagement-propensity score
#     is learned from Monkeypox and transferred as a within-dataset ranking signal.
# ============================================================

import os
import re
import csv
import sys
import json
import math
import random
import zipfile
import warnings
import subprocess
import importlib.util
from pathlib import Path

# ============================================================
# 0. Install missing packages
# ============================================================

PACKAGE_IMPORTS = {
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
    "scikit-learn": "sklearn",
    "matplotlib": "matplotlib",
    "textblob": "textblob",
    "ftfy": "ftfy",
    "textstat": "textstat",
    "scikit-posthocs": "scikit_posthocs",
}

for package_name, module_name in PACKAGE_IMPORTS.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", package_name],
            check=True
        )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import textstat
import scikit_posthocs as sp

from ftfy import fix_text
from textblob import TextBlob
from scipy import stats
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings("ignore")

# ============================================================
# 1. Configuration
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

OUTPUT_DIR = Path("linguistic_patterns_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Change these paths to match your actual file names
DATASET_CONFIGS = {
    "COVID--19_FNIR": {
        "path": "covid19_fnir.csv",
        "text_candidates": [
            "text", "tweet", "content", "claim", "title", "full_text",
            "Text", "Tweet", "Content", "Claim"
        ],
        "label_candidates": [
            "label", "labels", "class", "target", "Binary Label",
            "Label", "Class", "Target"
        ],
        "date_candidates": [
            "date", "created_at", "publish_date", "timestamp",
            "Date", "Created At", "Timestamp"
        ],
    },
    "Constraint": {
        "path": "constraint.csv",
        "text_candidates": [
            "tweet", "text", "content", "title", "full_text",
            "Tweet", "Text", "Content"
        ],
        "label_candidates": [
            "label", "labels", "class", "target", "Binary Label",
            "Label", "Class", "Target"
        ],
        "date_candidates": [
            "date", "created_at", "publish_date", "timestamp",
            "Date", "Created At", "Timestamp"
        ],
    },
    "Monkeypox": {
        "path": "monkeypox.csv",
        "text_candidates": [
            "tweet", "text", "content", "title", "full_text",
            "Tweet", "Text", "Content"
        ],
        "label_candidates": [
            "Binary Label", "label", "labels", "class", "target",
            "Label", "Class", "Target"
        ],
        "date_candidates": [
            "date", "created_at", "publish_date", "timestamp",
            "Date", "Created At", "Timestamp"
        ],
    },
}

ENGAGEMENT_CANDIDATES = {
    "likes": [
        "like_count", "likes", "favorite_count", "favourites_count",
        "favorites", "Like Count", "Likes"
    ],
    "retweets": [
        "retweet_count", "retweets", "n_retweets",
        "Retweet Count", "Retweets"
    ],
    "replies": [
        "reply_count", "replies", "n_replies",
        "Reply Count", "Replies"
    ],
    "quotes": [
        "quote_count", "quotes", "n_quotes",
        "Quote Count", "Quotes"
    ],
    "shares": [
        "share_count", "shares", "n_shares",
        "Share Count", "Shares"
    ],
}

PERSUASIVE_TERMS = [
    "urgent",
    "emergency",
    "fear",
    "panic",
    "alarming",
    "crisis",
    "warning",
    "disaster",
]

MIN_WORDS = 3
HIGH_ENGAGEMENT_QUANTILE = 0.90

FEATURES_FOR_PROPENSITY = [
    "token_count",
    "char_count",
    "ttr",
    "flesch_reading_ease",
    "flesch_kincaid_grade",
    "exclamation_rate",
    "question_rate",
    "persuasive_rate",
    "sentiment_polarity",
    "uppercase_ratio",
]

PRIORITY_THRESHOLDS = [1, 5, 10]

# ============================================================
# 2. General helpers
# ============================================================

def find_existing_column(df, candidates):
    cols = list(df.columns)
    lower_map = {str(c).strip().lower(): c for c in cols}

    for candidate in candidates:
        c = str(candidate).strip()

        if c in cols:
            return c

        c_lower = c.lower()
        if c_lower in lower_map:
            return lower_map[c_lower]

    return None


def robust_read_csv(file_path):
    file_path = Path(file_path)

    if not file_path.exists():
        try:
            from google.colab import files
            print(f"File not found locally: {file_path}")
            print("Upload the required CSV file now.")
            uploaded = files.upload()

            if not uploaded:
                raise FileNotFoundError("No file uploaded.")

            uploaded_name = list(uploaded.keys())[0]
            file_path = Path(uploaded_name)

        except Exception as exc:
            raise FileNotFoundError(
                f"File not found: {file_path}. Place it in the working directory or upload it in Colab."
            ) from exc

    attempts = []

    for sep in [",", "\t", ";", None]:
        for enc in ["utf-8", "utf-8-sig", "latin1"]:
            try:
                df = pd.read_csv(
                    file_path,
                    sep=sep,
                    encoding=enc,
                    dtype=str,
                    engine="python",
                    on_bad_lines="skip",
                    quoting=csv.QUOTE_MINIMAL
                )
                df.columns = [str(c).strip() for c in df.columns]
                attempts.append((sep, enc, df))
            except Exception:
                pass

    if not attempts:
        raise ValueError(f"Could not read {file_path}. Check encoding and delimiter.")

    best_sep, best_enc, best_df = max(attempts, key=lambda x: x[2].shape[1])

    print(f"Loaded {file_path} with separator={repr(best_sep)}, encoding={best_enc}")
    print(f"Shape: {best_df.shape}")
    print(f"Columns: {list(best_df.columns)[:30]}")

    return best_df.copy()


def safe_numeric(series, default=np.nan):
    return pd.to_numeric(series, errors="coerce").fillna(default).astype(float)


def parse_dates_safe(series):
    parsed = pd.to_datetime(series, errors="coerce", dayfirst=False)

    if parsed.isna().mean() > 0.50:
        parsed = pd.to_datetime(series, errors="coerce", dayfirst=True)

    return parsed


def clean_text_standard(x):
    if pd.isna(x):
        return ""

    x = str(x)
    x = fix_text(x)

    x = re.sub(r"http\S+|www\.\S+", " URL ", x)
    x = re.sub(r"@\w+", " USER ", x)
    x = re.sub(r"#", "", x)
    x = re.sub(r"<[^>]+>", " ", x)

    # Remove non-ASCII symbols, including most emojis, for readability stability
    x = x.encode("ascii", "ignore").decode("ascii")

    x = re.sub(r"\s+", " ", x).strip()
    return x


def clean_text_minimal(x):
    if pd.isna(x):
        return ""

    x = str(x)
    x = fix_text(x)
    x = re.sub(r"http\S+|www\.\S+", " URL ", x)
    x = re.sub(r"<[^>]+>", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def word_tokens(text):
    return re.findall(r"\b\w+\b", str(text).lower())


def word_count(text):
    return len(word_tokens(text))


def type_token_ratio(text):
    tokens = word_tokens(text)

    if len(tokens) == 0:
        return 0.0

    return len(set(tokens)) / len(tokens)


def sentiment_polarity(text):
    try:
        return float(TextBlob(str(text)).sentiment.polarity)
    except Exception:
        return 0.0


def flesch_reading_ease_safe(text):
    try:
        if word_count(text) < MIN_WORDS:
            return np.nan
        return float(textstat.flesch_reading_ease(text))
    except Exception:
        return np.nan


def flesch_kincaid_grade_safe(text):
    try:
        if word_count(text) < MIN_WORDS:
            return np.nan
        return float(textstat.flesch_kincaid_grade(text))
    except Exception:
        return np.nan


def uppercase_ratio(text):
    text = str(text)
    letters = re.findall(r"[A-Za-z]", text)

    if len(letters) == 0:
        return 0.0

    upper = sum(1 for ch in letters if ch.isupper())
    return upper / len(letters)


def persuasive_count(text):
    tokens = word_tokens(text)

    if len(tokens) == 0:
        return 0

    term_set = set(PERSUASIVE_TERMS)
    return sum(1 for tok in tokens if tok in term_set)


def safe_excerpt(text, max_chars=240):
    text = str(text)
    text = re.sub(r"http\S+|www\.\S+", "[URL]", text)
    text = re.sub(r"@\w+", "[USER]", text)
    text = re.sub(r"\s+", " ", text).strip()

    if len(text) > max_chars:
        return text[:max_chars].rstrip() + "..."

    return text


def safe_filename(name):
    name = str(name)
    name = name.replace("/", "_")
    name = name.replace("\\", "_")
    name = name.replace(" ", "_")
    name = name.replace("-", "_")
    return name

# ============================================================
# 3. Load and harmonise datasets
# ============================================================

def load_and_prepare_dataset(dataset_name, config):
    raw = robust_read_csv(config["path"])

    text_col = find_existing_column(raw, config["text_candidates"])
    label_col = find_existing_column(raw, config["label_candidates"])
    date_col = find_existing_column(raw, config["date_candidates"])

    if text_col is None:
        raise ValueError(
            f"No text column found for {dataset_name}. "
            f"Available columns: {list(raw.columns)}"
        )

    df = raw.copy()

    df["dataset"] = dataset_name
    df["raw_text"] = df[text_col].fillna("").astype(str)
    df["text_standard"] = df["raw_text"].map(clean_text_standard)
    df["text_minimal"] = df["raw_text"].map(clean_text_minimal)

    if label_col is not None:
        df["label_raw"] = df[label_col].astype(str)
    else:
        df["label_raw"] = np.nan

    if date_col is not None:
        df["date"] = parse_dates_safe(df[date_col])
        df["date_day"] = df["date"].dt.floor("D")
    else:
        df["date"] = pd.NaT
        df["date_day"] = pd.NaT

    df["source_text_column"] = text_col
    df["source_label_column"] = label_col if label_col is not None else ""
    df["source_date_column"] = date_col if date_col is not None else ""

    df = df[df["text_standard"].str.len() > 0].copy()
    df["token_count_initial"] = df["text_standard"].map(word_count)
    df = df[df["token_count_initial"] >= MIN_WORDS].copy()

    df = df.reset_index(drop=True)

    return df


def load_all_datasets():
    frames = []

    for dataset_name, config in DATASET_CONFIGS.items():
        print("\n" + "=" * 70)
        print(f"Loading {dataset_name}")
        print("=" * 70)

        df = load_and_prepare_dataset(dataset_name, config)
        frames.append(df)

    combined = pd.concat(frames, ignore_index=True)
    return combined

# ============================================================
# 4. Feature extraction
# ============================================================

def compute_linguistic_features(df, text_col="text_standard", suffix=""):
    df = df.copy()
    text = df[text_col].fillna("").astype(str)

    df[f"char_count{suffix}"] = text.str.len()
    df[f"token_count{suffix}"] = text.map(word_count)
    df[f"ttr{suffix}"] = text.map(type_token_ratio)
    df[f"sentiment_polarity{suffix}"] = text.map(sentiment_polarity)
    df[f"uppercase_ratio{suffix}"] = text.map(uppercase_ratio)

    df[f"flesch_reading_ease{suffix}"] = text.map(flesch_reading_ease_safe)
    df[f"flesch_kincaid_grade{suffix}"] = text.map(flesch_kincaid_grade_safe)

    df[f"exclamation_count{suffix}"] = text.str.count("!")
    df[f"question_count{suffix}"] = text.str.count(r"\?")

    df[f"exclamation_rate{suffix}"] = (
        df[f"exclamation_count{suffix}"] /
        df[f"token_count{suffix}"].replace(0, np.nan)
    ).fillna(0.0)

    df[f"question_rate{suffix}"] = (
        df[f"question_count{suffix}"] /
        df[f"token_count{suffix}"].replace(0, np.nan)
    ).fillna(0.0)

    df[f"eqr{suffix}"] = (
        df[f"exclamation_count{suffix}"] /
        df[f"question_count{suffix}"].replace(0, np.nan)
    )

    df[f"eqr{suffix}"] = df[f"eqr{suffix}"].replace([np.inf, -np.inf], np.nan)

    df[f"persuasive_count{suffix}"] = text.map(persuasive_count)

    df[f"persuasive_rate{suffix}"] = (
        df[f"persuasive_count{suffix}"] /
        df[f"token_count{suffix}"].replace(0, np.nan)
    ).fillna(0.0)

    return df


def add_all_linguistic_features(df):
    df = compute_linguistic_features(df, text_col="text_standard", suffix="")
    df = compute_linguistic_features(df, text_col="text_minimal", suffix="_minimal")
    return df

# ============================================================
# 5. Engagement handling
# ============================================================

def detect_engagement_columns(df):
    found = {}

    for channel, candidates in ENGAGEMENT_CANDIDATES.items():
        col = find_existing_column(df, candidates)

        if col is not None:
            found[channel] = col

    return found


def add_observed_engagement(df):
    df = df.copy()

    df["observed_engagement_available"] = False
    df["observed_engagement"] = np.nan

    engagement_report = []

    for dataset_name, sub_idx in df.groupby("dataset").groups.items():
        sub = df.loc[sub_idx].copy()
        found = detect_engagement_columns(sub)

        if len(found) == 0:
            engagement_report.append({
                "dataset": dataset_name,
                "observed_engagement_available": False,
                "engagement_columns_used": "",
            })
            continue

        engagement_components = []

        for channel, col in found.items():
            numeric_col = f"{channel}_engagement_numeric"
            df.loc[sub_idx, numeric_col] = safe_numeric(df.loc[sub_idx, col], default=0.0)
            engagement_components.append(numeric_col)

        df.loc[sub_idx, "observed_engagement"] = df.loc[sub_idx, engagement_components].sum(axis=1)
        df.loc[sub_idx, "observed_engagement_available"] = True

        engagement_report.append({
            "dataset": dataset_name,
            "observed_engagement_available": True,
            "engagement_columns_used": ", ".join([f"{k}:{v}" for k, v in found.items()]),
        })

    report = pd.DataFrame(engagement_report)

    return df, report


def train_and_apply_engagement_propensity(df):
    df = df.copy()

    df["engagement_propensity_probability"] = np.nan
    df["engagement_propensity_logit"] = np.nan
    df["priority_signal"] = np.nan
    df["priority_signal_type"] = ""

    monkey = df[
        (df["dataset"] == "Monkeypox") &
        (df["observed_engagement_available"] == True)
    ].copy()

    if len(monkey) < 50 or monkey["observed_engagement"].notna().sum() < 50:
        print("Insufficient Monkeypox observed engagement data. Proxy model skipped.")

        observed_mask = df["observed_engagement_available"] == True

        df.loc[observed_mask, "priority_signal"] = (
            df.loc[observed_mask, "observed_engagement"]
        )
        df.loc[observed_mask, "priority_signal_type"] = "observed_engagement"

        return df, pd.DataFrame([{
            "proxy_model_status": "skipped",
            "reason": "Insufficient Monkeypox observed engagement data",
            "training_dataset": "",
            "high_engagement_quantile": HIGH_ENGAGEMENT_QUANTILE,
            "n_train": 0,
            "roc_auc_mean": np.nan,
            "average_precision_mean": np.nan,
        }])

    threshold = monkey["observed_engagement"].quantile(HIGH_ENGAGEMENT_QUANTILE)
    monkey["high_engagement"] = (monkey["observed_engagement"] >= threshold).astype(int)

    if monkey["high_engagement"].nunique() < 2:
        print("Monkeypox high engagement target has one class only. Proxy model skipped.")

        observed_mask = df["observed_engagement_available"] == True

        df.loc[observed_mask, "priority_signal"] = (
            df.loc[observed_mask, "observed_engagement"]
        )
        df.loc[observed_mask, "priority_signal_type"] = "observed_engagement"

        return df, pd.DataFrame([{
            "proxy_model_status": "skipped",
            "reason": "One-class high engagement target",
            "training_dataset": "",
            "high_engagement_quantile": HIGH_ENGAGEMENT_QUANTILE,
            "n_train": 0,
            "roc_auc_mean": np.nan,
            "average_precision_mean": np.nan,
        }])

    for col in FEATURES_FOR_PROPENSITY:
        if col not in df.columns:
            df[col] = 0.0

    train = monkey.dropna(subset=FEATURES_FOR_PROPENSITY + ["high_engagement"]).copy()

    X = train[FEATURES_FOR_PROPENSITY].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y = train["high_engagement"].astype(int)

    model = Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            random_state=SEED
        ))
    ])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    roc_scores = cross_val_score(model, X, y, cv=cv, scoring="roc_auc")
    ap_scores = cross_val_score(model, X, y, cv=cv, scoring="average_precision")

    model.fit(X, y)

    X_all = df[FEATURES_FOR_PROPENSITY].replace([np.inf, -np.inf], np.nan).fillna(0.0)

    prob = model.predict_proba(X_all)[:, 1]
    prob_clip = np.clip(prob, 1e-8, 1 - 1e-8)
    logit = np.log(prob_clip / (1.0 - prob_clip))

    df["engagement_propensity_probability"] = prob
    df["engagement_propensity_logit"] = logit

    observed_mask = df["observed_engagement_available"] == True
    proxy_mask = ~observed_mask

    df.loc[observed_mask, "priority_signal"] = df.loc[observed_mask, "observed_engagement"]
    df.loc[observed_mask, "priority_signal_type"] = "observed_engagement"

    df.loc[proxy_mask, "priority_signal"] = df.loc[proxy_mask, "engagement_propensity_logit"]
    df.loc[proxy_mask, "priority_signal_type"] = "proxy_propensity_logit"

    model_report = pd.DataFrame([{
        "proxy_model_status": "fitted",
        "training_dataset": "Monkeypox",
        "high_engagement_quantile": HIGH_ENGAGEMENT_QUANTILE,
        "high_engagement_threshold": float(threshold),
        "n_train": int(len(train)),
        "positive_class_count": int(y.sum()),
        "negative_class_count": int((1 - y).sum()),
        "roc_auc_mean": float(np.mean(roc_scores)),
        "roc_auc_sd": float(np.std(roc_scores)),
        "average_precision_mean": float(np.mean(ap_scores)),
        "average_precision_sd": float(np.std(ap_scores)),
        "features_used": ", ".join(FEATURES_FOR_PROPENSITY),
    }])

    return df, model_report


def add_priority_flags(df, thresholds=PRIORITY_THRESHOLDS):
    df = df.copy()

    for dataset_name, sub_idx in df.groupby("dataset").groups.items():
        sub = df.loc[sub_idx]
        valid = sub["priority_signal"].replace([np.inf, -np.inf], np.nan).dropna()

        for x in thresholds:
            flag_col = f"priority_top_{x}pct"

            if len(valid) == 0:
                df.loc[sub_idx, flag_col] = False
                continue

            q = 1 - x / 100
            threshold = valid.quantile(q)

            df.loc[sub_idx, flag_col] = (
                sub["priority_signal"] >= threshold
            ).fillna(False)

    return df

# ============================================================
# 6. Descriptive summaries
# ============================================================

def dataset_summary(df):
    rows = []

    for dataset_name, sub in df.groupby("dataset"):
        row = {
            "dataset": dataset_name,
            "entries": int(len(sub)),
            "mean_length_chars": float(sub["char_count"].mean()),
            "sd_length_chars": float(sub["char_count"].std()),
            "mean_tokens": float(sub["token_count"].mean()),
            "sd_tokens": float(sub["token_count"].std()),
            "date_min": sub["date_day"].min(),
            "date_max": sub["date_day"].max(),
            "observed_engagement_available": bool(sub["observed_engagement_available"].any()),
            "mean_observed_engagement": float(sub["observed_engagement"].mean()) if sub["observed_engagement"].notna().any() else np.nan,
            "sd_observed_engagement": float(sub["observed_engagement"].std()) if sub["observed_engagement"].notna().any() else np.nan,
            "source_text_column": sub["source_text_column"].iloc[0],
            "source_label_column": sub["source_label_column"].iloc[0],
            "source_date_column": sub["source_date_column"].iloc[0],
        }
        rows.append(row)

    return pd.DataFrame(rows)


def grouped_feature_summary(df, features):
    rows = []

    for dataset_name, sub in df.groupby("dataset"):
        for feature in features:
            values = sub[feature].replace([np.inf, -np.inf], np.nan).dropna()

            if len(values) == 0:
                rows.append({
                    "dataset": dataset_name,
                    "feature": feature,
                    "n": 0,
                    "mean": np.nan,
                    "sd": np.nan,
                    "median": np.nan,
                    "iqr": np.nan,
                    "min": np.nan,
                    "max": np.nan,
                    "skew": np.nan,
                    "kurtosis": np.nan,
                })
                continue

            rows.append({
                "dataset": dataset_name,
                "feature": feature,
                "n": int(len(values)),
                "mean": float(values.mean()),
                "sd": float(values.std()),
                "median": float(values.median()),
                "iqr": float(values.quantile(0.75) - values.quantile(0.25)),
                "min": float(values.min()),
                "max": float(values.max()),
                "skew": float(stats.skew(values)) if len(values) > 2 else np.nan,
                "kurtosis": float(stats.kurtosis(values)) if len(values) > 3 else np.nan,
            })

    return pd.DataFrame(rows)

# ============================================================
# 7. Inferential tests
# ============================================================

def epsilon_squared(H, k, n):
    if n <= k:
        return np.nan

    return (H - k + 1) / (n - k)


def kruskal_tests(df, features):
    rows = []

    for feature in features:
        groups = []
        dataset_names = []

        for dataset_name, sub in df.groupby("dataset"):
            values = sub[feature].replace([np.inf, -np.inf], np.nan).dropna()

            if len(values) > 0:
                groups.append(values.values)
                dataset_names.append(dataset_name)

        if len(groups) < 2:
            rows.append({
                "feature": feature,
                "H": np.nan,
                "p_value": np.nan,
                "epsilon_squared": np.nan,
                "datasets": ", ".join(dataset_names),
            })
            continue

        H, p = stats.kruskal(*groups)
        n = sum(len(g) for g in groups)
        k = len(groups)

        rows.append({
            "feature": feature,
            "H": float(H),
            "p_value": float(p),
            "epsilon_squared": float(epsilon_squared(H, k, n)),
            "datasets": ", ".join(dataset_names),
        })

    return pd.DataFrame(rows)


def dunn_posthoc_tests(df, features):
    all_rows = []

    for feature in features:
        tmp = df[["dataset", feature]].copy()
        tmp[feature] = tmp[feature].replace([np.inf, -np.inf], np.nan)
        tmp = tmp.dropna()

        if tmp["dataset"].nunique() < 2:
            continue

        p_matrix = sp.posthoc_dunn(
            tmp,
            val_col=feature,
            group_col="dataset",
            p_adjust="bonferroni"
        )

        groups = list(p_matrix.index)

        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                g1 = groups[i]
                g2 = groups[j]

                all_rows.append({
                    "feature": feature,
                    "group_1": g1,
                    "group_2": g2,
                    "p_bonferroni": float(p_matrix.loc[g1, g2]),
                })

    return pd.DataFrame(all_rows)


def normality_checks(df, features, max_sample=5000):
    rows = []

    for dataset_name, sub in df.groupby("dataset"):
        for feature in features:
            values = sub[feature].replace([np.inf, -np.inf], np.nan).dropna()

            if len(values) < 3:
                continue

            sample = values.sample(
                n=min(max_sample, len(values)),
                random_state=SEED
            )

            try:
                shapiro_stat, shapiro_p = stats.shapiro(sample)
            except Exception:
                shapiro_stat, shapiro_p = np.nan, np.nan

            try:
                standardized = (sample - sample.mean()) / sample.std(ddof=0)
                standardized = standardized.replace([np.inf, -np.inf], np.nan).dropna()

                if len(standardized) > 3:
                    ks_stat, ks_p = stats.kstest(standardized, "norm")
                else:
                    ks_stat, ks_p = np.nan, np.nan
            except Exception:
                ks_stat, ks_p = np.nan, np.nan

            rows.append({
                "dataset": dataset_name,
                "feature": feature,
                "n_tested": int(len(sample)),
                "shapiro_stat": float(shapiro_stat) if not pd.isna(shapiro_stat) else np.nan,
                "shapiro_p": float(shapiro_p) if not pd.isna(shapiro_p) else np.nan,
                "ks_stat": float(ks_stat) if not pd.isna(ks_stat) else np.nan,
                "ks_p": float(ks_p) if not pd.isna(ks_p) else np.nan,
            })

    return pd.DataFrame(rows)


def spearman_readability_consistency(df):
    values = df[["flesch_reading_ease", "flesch_kincaid_grade"]].replace(
        [np.inf, -np.inf], np.nan
    ).dropna()

    if len(values) < 3:
        return pd.DataFrame([{
            "rho": np.nan,
            "p_value": np.nan,
            "n": int(len(values))
        }])

    rho, p = stats.spearmanr(
        values["flesch_reading_ease"],
        values["flesch_kincaid_grade"]
    )

    return pd.DataFrame([{
        "rho": float(rho),
        "p_value": float(p),
        "n": int(len(values))
    }])

# ============================================================
# 8. Robustness checks
# ============================================================

def length_restricted_subset(df):
    q10_by_dataset = df.groupby("dataset")["token_count"].quantile(0.10)
    q90_by_dataset = df.groupby("dataset")["token_count"].quantile(0.90)

    lower = int(math.ceil(q10_by_dataset.max()))
    upper = int(math.floor(q90_by_dataset.min()))

    restricted = df[
        (df["token_count"] >= lower) &
        (df["token_count"] <= upper)
    ].copy()

    info = pd.DataFrame([{
        "shared_lower_token_bound": lower,
        "shared_upper_token_bound": upper,
        "n_original": int(len(df)),
        "n_restricted": int(len(restricted)),
    }])

    return restricted, info


def minimal_cleaning_rhetorical_summary(df):
    features = [
        "exclamation_count_minimal",
        "question_count_minimal",
        "exclamation_rate_minimal",
        "question_rate_minimal",
        "eqr_minimal",
    ]

    return grouped_feature_summary(df, features)


def run_robustness_checks(df):
    restricted, restriction_info = length_restricted_subset(df)

    robust_features = [
        "flesch_reading_ease",
        "flesch_kincaid_grade",
        "exclamation_count",
        "question_count",
        "persuasive_rate",
    ]

    restricted_summary = grouped_feature_summary(restricted, robust_features)
    restricted_kw = kruskal_tests(restricted, robust_features)
    restricted_dunn = dunn_posthoc_tests(restricted, robust_features)
    minimal_rhet = minimal_cleaning_rhetorical_summary(df)

    return {
        "restriction_info": restriction_info,
        "restricted_summary": restricted_summary,
        "restricted_kw": restricted_kw,
        "restricted_dunn": restricted_dunn,
        "minimal_rhetorical_summary": minimal_rhet,
    }

# ============================================================
# 9. High-priority qualitative examples
# ============================================================

def build_high_priority_examples(df, top_fraction=0.10, n_examples_per_dataset=10):
    rows = []

    for dataset_name, sub in df.groupby("dataset"):
        valid = sub["priority_signal"].replace([np.inf, -np.inf], np.nan).dropna()

        if len(valid) == 0:
            continue

        threshold = valid.quantile(1 - top_fraction)

        high = sub[sub["priority_signal"] >= threshold].copy()
        high = high.sort_values("priority_signal", ascending=False)

        for _, row in high.head(n_examples_per_dataset).iterrows():
            rows.append({
                "dataset": dataset_name,
                "priority_signal_type": row.get("priority_signal_type", ""),
                "priority_signal": row.get("priority_signal", np.nan),
                "observed_engagement": row.get("observed_engagement", np.nan),
                "engagement_propensity_probability": row.get("engagement_propensity_probability", np.nan),
                "label_raw": row.get("label_raw", ""),
                "date_day": row.get("date_day", ""),
                "flesch_reading_ease": row.get("flesch_reading_ease", np.nan),
                "flesch_kincaid_grade": row.get("flesch_kincaid_grade", np.nan),
                "exclamation_count": row.get("exclamation_count", np.nan),
                "question_count": row.get("question_count", np.nan),
                "persuasive_rate": row.get("persuasive_rate", np.nan),
                "excerpt": safe_excerpt(row.get("raw_text", "")),
            })

    return pd.DataFrame(rows)

# ============================================================
# 10. Plotting
# ============================================================

def save_readability_boxplot(df):
    datasets = list(DATASET_CONFIGS.keys())

    fre_data = [
        df.loc[df["dataset"] == d, "flesch_reading_ease"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for d in datasets
    ]

    fkgl_data = [
        df.loc[df["dataset"] == d, "flesch_kincaid_grade"]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        for d in datasets
    ]

    plt.figure(figsize=(12, 6))

    plt.subplot(1, 2, 1)
    plt.boxplot(fre_data, labels=datasets, showfliers=True)
    plt.ylabel("Flesch Reading Ease")
    plt.title("FRE across datasets")
    plt.xticks(rotation=20)

    plt.subplot(1, 2, 2)
    plt.boxplot(fkgl_data, labels=datasets, showfliers=True)
    plt.ylabel("Flesch-Kincaid Grade Level")
    plt.title("FKGL across datasets")
    plt.xticks(rotation=20)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "readability_boxplot.png", dpi=300)
    plt.close()


def save_individual_readability_boxplots(df):
    datasets = list(DATASET_CONFIGS.keys())

    for feature, ylabel, filename in [
        ("flesch_reading_ease", "Flesch Reading Ease", "readability_fre_boxplot.png"),
        ("flesch_kincaid_grade", "Flesch-Kincaid Grade Level", "readability_fkgl_boxplot.png"),
    ]:
        data = [
            df.loc[df["dataset"] == d, feature]
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
            for d in datasets
        ]

        plt.figure(figsize=(9, 6))
        plt.boxplot(data, labels=datasets, showfliers=True)
        plt.ylabel(ylabel)
        plt.title(f"{ylabel} across datasets")
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / filename, dpi=300)
        plt.close()


def save_punctuation_bars(summary):
    punct = summary[
        summary["feature"].isin(["exclamation_count", "question_count"])
    ].copy()

    datasets = list(DATASET_CONFIGS.keys())
    features = ["exclamation_count", "question_count"]

    x = np.arange(len(datasets))
    width = 0.35

    means = {}
    sds = {}

    for feature in features:
        feature_means = []
        feature_sds = []

        for d in datasets:
            row = punct[
                (punct["dataset"] == d) &
                (punct["feature"] == feature)
            ]

            feature_means.append(row["mean"].values[0] if len(row) else np.nan)
            feature_sds.append(row["sd"].values[0] if len(row) else np.nan)

        means[feature] = feature_means
        sds[feature] = feature_sds

    plt.figure(figsize=(9, 6))
    plt.bar(
        x - width / 2,
        means["exclamation_count"],
        width,
        yerr=sds["exclamation_count"],
        label="Exclamation"
    )
    plt.bar(
        x + width / 2,
        means["question_count"],
        width,
        yerr=sds["question_count"],
        label="Question"
    )
    plt.ylabel("Mean count per post")
    plt.title("Punctuation markers across datasets")
    plt.xticks(x, datasets, rotation=20)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "punctuation_bars.png", dpi=300)
    plt.close()


def save_persuasive_terms_bar(summary):
    pers = summary[summary["feature"] == "persuasive_rate"].copy()
    datasets = list(DATASET_CONFIGS.keys())

    means = []
    sds = []

    for d in datasets:
        row = pers[pers["dataset"] == d]

        means.append(row["mean"].values[0] if len(row) else np.nan)
        sds.append(row["sd"].values[0] if len(row) else np.nan)

    plt.figure(figsize=(9, 6))
    plt.bar(datasets, means, yerr=sds)
    plt.ylabel("Mean persuasive term rate")
    plt.title("Persuasive or fear-related lexicon across datasets")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "persuasive_terms.png", dpi=300)
    plt.close()


def save_priority_signal_histograms(df):
    for dataset_name, sub in df.groupby("dataset"):
        values = sub["priority_signal"].replace([np.inf, -np.inf], np.nan).dropna()

        if len(values) == 0:
            continue

        name = safe_filename(dataset_name)

        plt.figure(figsize=(8, 5))
        plt.hist(values, bins=40)
        plt.xlabel("Priority signal")
        plt.ylabel("Frequency")
        plt.title(f"Priority signal distribution: {dataset_name}")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"priority_signal_histogram_{name}.png", dpi=300)
        plt.close()


def make_all_plots(df, feature_summary):
    save_readability_boxplot(df)
    save_individual_readability_boxplots(df)
    save_punctuation_bars(feature_summary)
    save_persuasive_terms_bar(feature_summary)
    save_priority_signal_histograms(df)

# ============================================================
# 11. Analysis summary writer
# ============================================================

def write_analysis_summary(
    ds_summary,
    feature_summary,
    kw_results,
    spearman_results,
    engagement_report,
    proxy_report
):
    def get_mean(dataset, feature):
        row = feature_summary[
            (feature_summary["dataset"] == dataset) &
            (feature_summary["feature"] == feature)
        ]

        if len(row) == 0:
            return np.nan

        return row["mean"].iloc[0]

    covid_fre = get_mean("COVID--19_FNIR", "flesch_reading_ease")
    constraint_fre = get_mean("Constraint", "flesch_reading_ease")
    monkey_fre = get_mean("Monkeypox", "flesch_reading_ease")

    covid_fkgl = get_mean("COVID--19_FNIR", "flesch_kincaid_grade")
    constraint_fkgl = get_mean("Constraint", "flesch_kincaid_grade")
    monkey_fkgl = get_mean("Monkeypox", "flesch_kincaid_grade")

    covid_pers = get_mean("COVID--19_FNIR", "persuasive_rate")
    constraint_pers = get_mean("Constraint", "persuasive_rate")
    monkey_pers = get_mean("Monkeypox", "persuasive_rate")

    rho = spearman_results.loc[0, "rho"]
    rho_p = spearman_results.loc[0, "p_value"]

    text = f"""
Linguistic-patterns analysis summary

This analysis uses three datasets: COVID--19_FNIR, Constraint, and Monkeypox.

The final retained dataset sizes are reported in dataset_summary.csv. The analysis extracts readability, rhetorical punctuation, persuasive or fear-related lexicon, sentiment, lexical diversity, and priority-ranking signals from each corpus under a shared preprocessing pipeline.

Mean Flesch Reading Ease values:
COVID--19_FNIR: {covid_fre:.3f}
Constraint: {constraint_fre:.3f}
Monkeypox: {monkey_fre:.3f}

Mean Flesch-Kincaid Grade Level values:
COVID--19_FNIR: {covid_fkgl:.3f}
Constraint: {constraint_fkgl:.3f}
Monkeypox: {monkey_fkgl:.3f}

Mean persuasive lexicon rates:
COVID--19_FNIR: {covid_pers:.6f}
Constraint: {constraint_pers:.6f}
Monkeypox: {monkey_pers:.6f}

The Spearman correlation between Flesch Reading Ease and Flesch-Kincaid Grade Level was rho = {rho:.4f}, p = {rho_p:.4g}. This supports internal consistency between the two readability measures because they are expected to move in opposite directions.

Observed engagement availability:
{engagement_report.to_string(index=False)}

Engagement-propensity model:
{proxy_report.to_string(index=False)}

Interpretation note:
Observed engagement should be interpreted only where native engagement fields exist. Where observed engagement is unavailable, the priority signal is a proxy engagement-propensity logit learned from Monkeypox and used only for within-dataset ranking. It is not an observed measure of likes, retweets, replies, quotes, or shares.
""".strip()

    with open(OUTPUT_DIR / "analysis_summary.txt", "w", encoding="utf-8") as f:
        f.write(text)

    return text

# ============================================================
# 12. Main pipeline
# ============================================================

def main():
    print("\nStarting linguistic-patterns pipeline")
    print("Datasets: COVID--19_FNIR, Constraint, Monkeypox")

    df = load_all_datasets()

    print("\nExtracting linguistic features...")
    df = add_all_linguistic_features(df)

    print("\nAdding observed engagement where available...")
    df, engagement_report = add_observed_engagement(df)

    print("\nTraining and applying engagement-propensity model...")
    df, proxy_report = train_and_apply_engagement_propensity(df)

    print("\nAdding top 1%, 5%, and 10% priority flags...")
    df = add_priority_flags(df, thresholds=PRIORITY_THRESHOLDS)

    core_features = [
        "flesch_reading_ease",
        "flesch_kincaid_grade",
        "exclamation_count",
        "question_count",
        "exclamation_rate",
        "question_rate",
        "eqr",
        "persuasive_count",
        "persuasive_rate",
        "sentiment_polarity",
        "token_count",
        "char_count",
        "ttr",
        "uppercase_ratio",
    ]

    test_features = [
        "flesch_reading_ease",
        "flesch_kincaid_grade",
        "exclamation_count",
        "question_count",
        "persuasive_rate",
    ]

    print("\nBuilding descriptive summaries...")
    ds_summary = dataset_summary(df)
    feature_summary = grouped_feature_summary(df, core_features)

    print("\nRunning normality checks...")
    normality = normality_checks(df, test_features)

    print("\nRunning Kruskal-Wallis tests...")
    kw_results = kruskal_tests(df, test_features)

    print("\nRunning Dunn post hoc tests...")
    dunn_results = dunn_posthoc_tests(df, test_features)

    print("\nComputing readability consistency...")
    spearman_results = spearman_readability_consistency(df)

    print("\nRunning robustness checks...")
    robustness = run_robustness_checks(df)

    print("\nBuilding high-priority examples...")
    examples = build_high_priority_examples(
        df,
        top_fraction=0.10,
        n_examples_per_dataset=10
    )

    print("\nGenerating plots...")
    make_all_plots(df, feature_summary)

    print("\nWriting outputs...")

    df.to_csv(OUTPUT_DIR / "posts_with_linguistic_features.csv", index=False)
    ds_summary.to_csv(OUTPUT_DIR / "dataset_summary.csv", index=False)
    feature_summary.to_csv(OUTPUT_DIR / "feature_summary_by_dataset.csv", index=False)
    normality.to_csv(OUTPUT_DIR / "normality_checks.csv", index=False)
    kw_results.to_csv(OUTPUT_DIR / "kruskal_wallis_results.csv", index=False)
    dunn_results.to_csv(OUTPUT_DIR / "dunn_posthoc_bonferroni_results.csv", index=False)
    spearman_results.to_csv(OUTPUT_DIR / "spearman_readability_consistency.csv", index=False)
    engagement_report.to_csv(OUTPUT_DIR / "engagement_availability_report.csv", index=False)
    proxy_report.to_csv(OUTPUT_DIR / "engagement_propensity_model_report.csv", index=False)
    examples.to_csv(OUTPUT_DIR / "high_priority_qualitative_examples.csv", index=False)

    robustness["restriction_info"].to_csv(
        OUTPUT_DIR / "robustness_length_restriction_info.csv", index=False
    )
    robustness["restricted_summary"].to_csv(
        OUTPUT_DIR / "robustness_length_restricted_summary.csv", index=False
    )
    robustness["restricted_kw"].to_csv(
        OUTPUT_DIR / "robustness_length_restricted_kruskal.csv", index=False
    )
    robustness["restricted_dunn"].to_csv(
        OUTPUT_DIR / "robustness_length_restricted_dunn.csv", index=False
    )
    robustness["minimal_rhetorical_summary"].to_csv(
        OUTPUT_DIR / "robustness_minimal_cleaning_rhetorical_summary.csv", index=False
    )

    analysis_summary = write_analysis_summary(
        ds_summary=ds_summary,
        feature_summary=feature_summary,
        kw_results=kw_results,
        spearman_results=spearman_results,
        engagement_report=engagement_report,
        proxy_report=proxy_report,
    )

    with open(OUTPUT_DIR / "pipeline_config.json", "w", encoding="utf-8") as f:
        json.dump(
            {
                "seed": SEED,
                "datasets": DATASET_CONFIGS,
                "persuasive_terms": PERSUASIVE_TERMS,
                "min_words": MIN_WORDS,
                "high_engagement_quantile": HIGH_ENGAGEMENT_QUANTILE,
                "features_for_propensity": FEATURES_FOR_PROPENSITY,
                "priority_thresholds": PRIORITY_THRESHOLDS,
            },
            f,
            indent=2,
            default=str
        )

    zip_path = OUTPUT_DIR.with_suffix(".zip")

    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for file in OUTPUT_DIR.glob("*"):
            if file.is_file():
                z.write(file, arcname=file.name)

    print("\nDataset summary")
    print(ds_summary)

    print("\nFeature summary")
    print(feature_summary.head(30))

    print("\nKruskal-Wallis results")
    print(kw_results)

    print("\nDunn post hoc results")
    print(dunn_results)

    print("\nSpearman readability consistency")
    print(spearman_results)

    print("\nEngagement availability")
    print(engagement_report)

    print("\nProxy model report")
    print(proxy_report)

    print("\nHigh-priority examples")
    print(examples.head(15))

    print("\nAnalysis summary")
    print(analysis_summary)

    print(f"\nSaved outputs to: {OUTPUT_DIR}")
    print(f"Zipped outputs to: {zip_path}")

    try:
        from IPython.display import display
        display(ds_summary)
        display(feature_summary)
        display(kw_results)
        display(dunn_results)
        display(spearman_results)
        display(engagement_report)
        display(proxy_report)
        display(examples.head(20))
    except Exception:
        pass


if __name__ == "__main__":
    main()